# A dispersion-aware, expression-gated pipeline reveals a conserved cassette-exon program across three solid tumors

**Md. Zulkarnain Sajid** -- Independent Researcher, Dhaka, Bangladesh
ORCID: [0009-0007-9421-3016](https://orcid.org/0009-0007-9421-3016)
Contact: zulkarnainsajid4768@gmail.com
Repository: <https://github.com/nain0017/aberrant-splicing-LUAD-BLCA-UCEC>
Archive: [10.5281/zenodo.21306629](https://doi.org/10.5281/zenodo.21306629)

---

This notebook is the reproducible analysis pipeline for the manuscript submitted to *NAR Genomics and Bioinformatics*. It processes three TCGA solid-tumor cohorts (LUAD, BLCA, UCEC) against GTEx tissue-matched normal cohorts and TCGA solid-tissue normal (SCN) samples, all uniformly re-aligned by recount3/Monorail (GENCODE v26).

**Pipeline stages, top to bottom:**

| Section | Stage | Output |
|---|---|---|
| 0 | Environment setup | Drive mount, imports, path constants |
| 1 | Stage A: expression gating | TPM matrices, expressed-in-both gene gates |
| 2 | Stage B: SUPPA2 event cataloging | Cassette-exon (SE) event catalog + junction map |
| 3 | Stage C: PSI computation | Per-sample PSI matrices with coverage gates |
| 4 | Stage D: beta-binomial GLM | Corrected primary aberrant sets (464 / 411 / 938) |
| 5 | M3' overlap + conserved core | 123-event three-way conserved core (115 genes) |
| 6 | Pathway enrichment | EMT + Myogenesis + Membrane Trafficking programs |
| 7 | M4 host-gene enrichment + audit | OR=14.25, p=2.83e-20 + six-test circularity audit |
| 8 | Figures 1-5 | Main manuscript figures |
| 9 | Supplementary figures + Tables T1-T10 | Supplementary materials |

**Reproducibility notes.** Every headline number in the manuscript traces to a printed output in the cells below. The Stage D beta-binomial GLM (Section 4) takes ~24 minutes total across the three cancers on Colab Pro (A100 not required; CPU is sufficient). Section 4 saves corrected result parquets to Drive; sections 5-9 read those parquets and can be re-run independently in ~5 minutes total.

**Methodological corrections.** The pre-registered deviation register `STAGE_C_DEVIATIONS.md` (in the repository root) documents seven strengthening deviations (D1-D7) applied during the analysis, each dated and justified in writing before the resulting numbers were used in the manuscript. The most impactful is D7 (dPSI_scn >= 0.05 floor in the 3C concordance check), which reduced pre-D7 primary counts from 1,266 / 1,047 / 1,545 to the final locked 464 / 411 / 938.

---
## Section 0 -- Environment setup

Mounts Google Drive, installs pinned dependencies, and defines project path constants used throughout the notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q \
    numpy==2.0.2 pandas==2.2.2 scipy==1.16.3 statsmodels==0.14.6 \
    matplotlib==3.10.0 matplotlib-venn==1.1.2 openpyxl==3.1.5 \
    gseapy==1.2.1 adjustText


In [ ]:
import os
import sys
import json
import time
import subprocess
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.optimize import minimize
from scipy.special import betaln, gammaln
from statsmodels.stats.multitest import multipletests

import matplotlib.pyplot as plt
import matplotlib as mpl

PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
DATA_DIR     = os.path.join(PROJECT_ROOT, "data")
RAW_DIR      = os.path.join(DATA_DIR, "raw")
PROC_DIR     = os.path.join(DATA_DIR, "processed")
STAGE_C_DIR  = os.path.join(PROC_DIR, "psi", "Stage2C_corrected")
FIG_DIR      = os.path.join(STAGE_C_DIR, "figures")
TABLE_DIR    = os.path.join(STAGE_C_DIR, "tables")

for d in (RAW_DIR, PROC_DIR, STAGE_C_DIR, FIG_DIR, TABLE_DIR):
    os.makedirs(d, exist_ok=True)

CANCERS = ["LUAD", "BLCA", "UCEC"]
NORMAL_MAP = {"LUAD": "LUNG", "BLCA": "BLADDER", "UCEC": "UTERUS"}

MIN_READS_PER_SAMPLE = 10
MIN_WELL_COVERED_SAMPLES = 20
DPSI_THRESHOLD = 0.15
FDR_THRESHOLD = 0.05
DPSI_SCN_FLOOR = 0.05
TPM_EXPRESSION_FLOOR = 1.0

print(f"Project root: {PROJECT_ROOT}")
print(f"Cancers: {CANCERS}")
print(f"Cancer -> normal: {NORMAL_MAP}")


---
## Section 1 -- Stage A: TPM-based expression gating

Downloads gene-level counts from recount3 for six cohorts (3 TCGA cancers + 3 GTEx normals), computes TPM per sample using the GENCODE v26 exon-sum length lookup, and builds per-cancer "expressed in both tumor and matched normal" gene gates at median TPM >= 1.

Genes failing this gate are excluded from all downstream splicing analysis. MUC16 (a tumor antigen expressed only in cancer, not normal tissue) is correctly excluded, validating the gate.

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import r as R
import time

print("PATH B WEEK 1 STEP 1: recount3 gene-level downloads")

# Install recount3 in R (fresh session needs this)
t0 = time.time()
print("\nInstalling R packages (recount3, SummarizedExperiment, Matrix)...")
print("Estimated time: 5-10 min on first call")

R('''
if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager", repos = "https://cran.rstudio.com/")
}
suppressMessages({
  if (!requireNamespace("recount3", quietly = TRUE)) {
    BiocManager::install(c("recount3", "SummarizedExperiment", "Matrix"),
                         update = FALSE, ask = FALSE)
  }
  library(recount3)
  library(SummarizedExperiment)
  library(Matrix)
})
cat("R packages ready\\n")
''')
print(f"[{time.time()-t0:.1f}s] R setup complete")

# Set save path
R(f'GENE_DIR <- "{PROC_GENE}"')

# Define download function for gene-level data
R('''
download_gene_counts <- function(source, project, save_dir) {
  cat("--- ", source, "-", project, " (gene-level) ---\\n", sep="")
  t0 <- Sys.time()

  human_projects <- available_projects(organism = "human")
  proj_row <- subset(human_projects, file_source == source & project == project_name)
  # Note: variable shadow issue, fix below

  if (nrow(proj_row) == 0) {
    cat("ERROR: project not found\\n")
    return(FALSE)
  }

  rse_gene <- create_rse(proj_row, type = "gene")
  cat("  Created RSE: ", nrow(rse_gene), " genes x ", ncol(rse_gene), " samples\\n", sep="")

  # recount3 gene counts need scaling (raw_counts have not been adjusted for read length)
  # transform_counts() applies the standard scaling
  scaled <- transform_counts(rse_gene)

  # Extract counts (dense for genes since only ~60K genes)
  gene_counts <- as.matrix(scaled)
  gene_meta <- as.data.frame(rowRanges(rse_gene))
  sample_meta <- as.data.frame(colData(rse_gene))

  prefix <- paste0(toupper(source), "_", project, "_gene")
  saveRDS(gene_counts, file.path(save_dir, paste0(prefix, "_counts.rds")), compress = "gzip")
  write.csv(gene_meta, file.path(save_dir, paste0(prefix, "_meta.csv")), row.names = FALSE)
  write.csv(sample_meta, file.path(save_dir, paste0(prefix, "_samples.csv")), row.names = FALSE)

  elapsed <- round(as.numeric(difftime(Sys.time(), t0, units = "secs")), 1)
  cat("  [OK] Saved in ", elapsed, "s\\n", sep="")

  rm(rse_gene, scaled, gene_counts, gene_meta, sample_meta)
  gc()
  return(TRUE)
}
''')

# Run downloads for all 6 cohorts
COHORTS = [
    ('tcga', 'BLCA'),
    ('tcga', 'UCEC'),
    ('tcga', 'LUAD'),
    ('gtex', 'BLADDER'),
    ('gtex', 'UTERUS'),
    ('gtex', 'LUNG'),
]

for source, proj in COHORTS:
    save_path = PROC_GENE
    R(f'''
    source <- "{source}"
    project_name <- "{proj}"
    save_dir <- "{save_path}"

    cat("--- ", source, "-", project_name, " (gene-level) ---\\n", sep="")
    t0 <- Sys.time()

    human_projects <- available_projects(organism = "human")
    proj_row <- subset(human_projects, file_source == source & project == project_name)

    if (nrow(proj_row) == 0) {{
      cat("ERROR: project not found\\n")
    }} else {{
      rse_gene <- create_rse(proj_row, type = "gene")
      cat("  Created RSE: ", nrow(rse_gene), " genes x ", ncol(rse_gene), " samples\\n", sep="")

      # Apply recount3's standard scaling
      scaled <- transform_counts(rse_gene)

      gene_counts <- as.matrix(scaled)
      gene_meta <- as.data.frame(rowRanges(rse_gene))
      sample_meta <- as.data.frame(colData(rse_gene))

      label <- toupper(source)
      prefix <- paste0(label, "_", project_name, "_gene")
      saveRDS(gene_counts, file.path(save_dir, paste0(prefix, "_counts.rds")), compress = "gzip")
      write.csv(gene_meta, file.path(save_dir, paste0(prefix, "_meta.csv")), row.names = FALSE)
      write.csv(sample_meta, file.path(save_dir, paste0(prefix, "_samples.csv")), row.names = FALSE)

      elapsed <- round(as.numeric(difftime(Sys.time(), t0, units = "secs")), 1)
      cat("  [OK] Saved in ", elapsed, "s\\n", sep="")

      rm(rse_gene, scaled, gene_counts, gene_meta, sample_meta)
      gc()
    }}
    ''')

# Verify
print("\n--- Verification ---")
for f in sorted(os.listdir(PROC_GENE)):
    size_mb = os.path.getsize(os.path.join(PROC_GENE, f)) / 1024**2
    print(f"  {f} ({size_mb:.1f} MB)")


In [ ]:
import os, time, pickle, gzip
import pandas as pd
from collections import defaultdict

print("PATH B WEEK 1 STEP 2A: gene length lookup from GENCODE v26")

# First, inspect one gene meta file to see what we got
sample_meta_path = os.path.join(PROC_GENE, 'TCGA_BLCA_gene_meta.csv')
print(f"\n--- Inspecting {os.path.basename(sample_meta_path)} ---")
meta = pd.read_csv(sample_meta_path)
print(f"Shape: {meta.shape}")
print(f"Columns: {list(meta.columns)}")
print(f"\nFirst 3 rows:")
print(meta.head(3).to_string())

# Now compute exon-sum length per gene from GTF
GTF_PATH = os.path.join(REF_DIR, 'gencode.v26.annotation.gtf.gz')
LENGTH_CACHE = os.path.join(REF_DIR, 'gencode_v26_gene_lengths.pkl')

if os.path.exists(LENGTH_CACHE):
    print(f"\nLoading cached gene lengths from {LENGTH_CACHE}...")
    with open(LENGTH_CACHE, 'rb') as f:
        gene_lengths = pickle.load(f)
    print(f"  Loaded {len(gene_lengths):,} gene lengths")
else:
    print(f"\nComputing exon-sum lengths from GTF (one-time, ~2 min)...")
    t0 = time.time()

    # For each gene, collect unique exon intervals (collapsed across transcripts)
    # Method: for each gene, union of all exon coordinates, then sum disjoint widths
    gene_exons = defaultdict(set)  # gene_id -> set of (start, end) tuples

    line_count = 0
    with gzip.open(GTF_PATH, 'rt') as f:
        for line in f:
            if line.startswith('#'): continue
            fields = line.rstrip().split('\t')
            if len(fields) < 9 or fields[2] != 'exon': continue

            start, end = int(fields[3]), int(fields[4])
            attrs = fields[8]
            gene_id = None
            for part in attrs.split(';'):
                part = part.strip()
                if part.startswith('gene_id '):
                    gene_id = part.split('"')[1]
                    break
            if gene_id:
                gene_exons[gene_id].add((start, end))
            line_count += 1
            if line_count % 500_000 == 0:
                print(f"  Parsed {line_count:,} exon records...")

    print(f"  Parsed {line_count:,} total exons across {len(gene_exons):,} genes")
    print(f"  Computing union of exon intervals per gene...")

    # For each gene, merge overlapping exons then sum widths
    def merge_and_sum(intervals):
        """Merge overlapping intervals, return total length."""
        sorted_iv = sorted(intervals)
        total = 0
        cur_start, cur_end = sorted_iv[0]
        for s, e in sorted_iv[1:]:
            if s <= cur_end:
                cur_end = max(cur_end, e)
            else:
                total += cur_end - cur_start + 1
                cur_start, cur_end = s, e
        total += cur_end - cur_start + 1
        return total

    gene_lengths = {gid: merge_and_sum(list(ivs)) for gid, ivs in gene_exons.items()}

    with open(LENGTH_CACHE, 'wb') as f:
        pickle.dump(gene_lengths, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"[{time.time()-t0:.1f}s] Computed and cached {len(gene_lengths):,} gene lengths")

# Sanity check
import numpy as np
lens = np.array(list(gene_lengths.values()))
print(f"\n--- Gene length stats ---")
print(f"  Mean: {lens.mean():.0f} bp")
print(f"  Median: {np.median(lens):.0f} bp")
print(f"  Min/Max: {lens.min()} / {lens.max():,}")
print(f"  5 longest genes:")
top5 = sorted(gene_lengths.items(), key=lambda x: x[1], reverse=True)[:5]
for gid, l in top5:
    print(f"    {gid}: {l:,} bp")


In [ ]:
import os, time, json
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import r as R

print("PATH B WEEK 1 STEP 2B: TPM + expressed-gene tables per cohort")

# Pre-committed criteria for "expressed gene":
#   median TPM across cohort samples >= 1
# This is the standard field threshold (Eisenberg 2013, ENCODE consortium)

EXPRESSED_TPM_THRESHOLD = 1.0
print(f"\nPre-committed expression threshold: median TPM >= {EXPRESSED_TPM_THRESHOLD}")
print("This is the field-standard 'expressed gene' cutoff (Eisenberg 2013)")

# Define cohorts
COHORTS = [
    ('TCGA', 'BLCA'),
    ('TCGA', 'UCEC'),
    ('TCGA', 'LUAD'),
    ('GTEX', 'BLADDER'),
    ('GTEX', 'UTERUS'),
    ('GTEX', 'LUNG'),
]

results = {}

for source, project in COHORTS:
    label = f"{source}_{project}"
    print(f"\n{'='*60}")
    print(f"Processing {label}")
    print(f"{'='*60}")
    t0 = time.time()

    # Load gene meta (gives us gene_id and bp_length)
    meta = pd.read_csv(os.path.join(PROC_GENE, f'{label}_gene_meta.csv'))
    bp_lengths = meta['bp_length'].values.astype(np.float64)  # in bp
    n_genes = len(bp_lengths)

    # Load counts via rpy2 (it's an R matrix)
    counts_path = os.path.join(PROC_GENE, f'{label}_gene_counts.rds')
    R(f'''
    suppressMessages(library(Matrix))
    gene_mat <- readRDS("{counts_path}")
    cat("Loaded class:", class(gene_mat)[1], "\\n")
    cat("Dimensions:", dim(gene_mat), "\\n")
    cat("First few values:", gene_mat[1:3, 1:3], "\\n")
    ''')

    # Extract as dense numpy (genes are only ~63K, samples ~21-655, fits in memory)
    counts_r = ro.r('gene_mat')
    counts = np.asarray(counts_r, dtype=np.float64).reshape(
        int(ro.r('nrow(gene_mat)')[0]), int(ro.r('ncol(gene_mat)')[0]),
        order='F'  # R uses column-major
    )
    R('rm(gene_mat); gc()')

    n_genes_data, n_samples = counts.shape
    print(f"  Counts matrix: {n_genes_data} genes x {n_samples} samples")
    print(f"  Counts stats: min={counts.min():.1f}, max={counts.max():.1f}, "
          f"median={np.median(counts):.1f}")

    # Sanity check: count matrix and meta should agree on gene count
    if n_genes_data != n_genes:
        print(f"  WARNING: counts has {n_genes_data} genes, meta has {n_genes}")

    # ---- Compute TPM ----
    # TPM_ij = (count_ij / length_i) / sum_k(count_kj / length_k) * 1e6
    # Step 1: counts per kilobase (CPK)
    print(f"  Computing TPM...")

    # Avoid divide-by-zero for genes with bp_length == 0 or missing
    valid_length = bp_lengths > 0
    n_valid = valid_length.sum()
    print(f"    Genes with valid bp_length: {n_valid:,} of {n_genes:,}")

    # Per-kilobase: divide each gene's count by its length in kb
    length_kb = bp_lengths / 1000.0
    length_kb_safe = np.where(valid_length, length_kb, 1.0)  # avoid div-by-zero

    # CPK per sample (genes x samples)
    cpk = counts / length_kb_safe[:, np.newaxis]
    cpk[~valid_length, :] = 0  # zero out genes without valid length

    # Per-sample CPK sum (for normalization), only across genes with valid length
    sample_cpk_sum = cpk.sum(axis=0)  # shape: (n_samples,)

    # TPM: divide each CPK by sample sum, multiply by 1M
    tpm = cpk / sample_cpk_sum[np.newaxis, :] * 1e6

    # Verify: each sample's TPM should sum to ~1e6
    sample_tpm_sums = tpm.sum(axis=0)
    print(f"    Per-sample TPM sum range: [{sample_tpm_sums.min():.0f}, {sample_tpm_sums.max():.0f}] (should be ~1e6)")

    # ---- Median TPM per gene ----
    median_tpm = np.median(tpm, axis=1)

    # ---- Identify expressed genes ----
    expressed_mask = median_tpm >= EXPRESSED_TPM_THRESHOLD
    n_expressed = expressed_mask.sum()
    print(f"  Expressed genes (median TPM >= {EXPRESSED_TPM_THRESHOLD}): "
          f"{n_expressed:,} of {n_genes:,} ({100*n_expressed/n_genes:.1f}%)")

    # Per-gene-type expressed counts (useful sanity check)
    if 'gene_type' in meta.columns:
        gene_types = meta['gene_type'].values
        pc_total = (gene_types == 'protein_coding').sum()
        pc_expressed = ((gene_types == 'protein_coding') & expressed_mask).sum()
        print(f"  Protein-coding: {pc_expressed:,} of {pc_total:,} expressed ({100*pc_expressed/pc_total:.1f}%)")

    # ---- Save outputs ----
    expression_df = pd.DataFrame({
        'gene_id': meta['gene_id'].values,
        'gene_name': meta['gene_name'].values,
        'gene_type': meta['gene_type'].values,
        'bp_length': bp_lengths,
        'median_tpm': median_tpm,
        'is_expressed': expressed_mask,
    })

    output_path = os.path.join(PROC_GENE, f'{label}_expression.parquet')
    expression_df.to_parquet(output_path, index=False, compression='snappy')
    size_mb = os.path.getsize(output_path) / 1024**2
    print(f"  Saved: {output_path} ({size_mb:.1f} MB)")

    results[label] = {
        'source': source,
        'project': project,
        'n_samples': n_samples,
        'n_expressed': int(n_expressed),
        'n_pc_expressed': int(pc_expressed) if 'gene_type' in meta.columns else None,
        'elapsed_sec': time.time() - t0,
    }

    # Cleanup
    del counts, cpk, tpm, expression_df

# ---- Final summary ----
print("\n" + "=" * 60)
print("PATH B WEEK 1 EXPRESSED-GENE SUMMARY")
print(f"{'Cohort':<18} {'Samples':>8} {'Expressed':>12} {'PC Expressed':>14}")
for label, r in results.items():
    pc_str = f"{r['n_pc_expressed']:,}" if r['n_pc_expressed'] is not None else 'N/A'
    print(f"{label:<18} {r['n_samples']:>8} {r['n_expressed']:>12,} {pc_str:>14}")

# Save summary
summary_path = os.path.join(PROC_GENE, 'expression_summary.json')
with open(summary_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f"\nSummary: {summary_path}")


In [ ]:
import os, time, json
import pandas as pd
import numpy as np

print("PATH B WEEK 1 STEP 3: 'expressed in both' gene gates (FIXED)")

PAIRINGS = [
    ('BLCA', 'BLADDER'),
    ('UCEC', 'UTERUS'),
    ('LUAD', 'LUNG'),
]

CANCER_SPLICING_GENES = {
    'CD19', 'MS4A1', 'CD44', 'CD274', 'PDCD1', 'CTLA4',
    'MET', 'EGFR', 'ERBB2', 'KIT', 'FGFR1', 'FGFR2', 'FGFR3',
    'FAS', 'BCL2L1', 'MCL1', 'BIRC5',
    'AR', 'ESR1',
    'SF3B1', 'U2AF1', 'SRSF2', 'ZRSR2', 'RBM10', 'RBM39',
    'TP53', 'CDKN2A', 'PTEN', 'BRCA1', 'BRCA2',
    'CTNNB1', 'APC',
    'HGF',
    'MDM2', 'MDM4', 'BCL2', 'VEGFA', 'KLF6',
}

PATH_A_FALSE_POSITIVES = {
    'MUC16', 'KIF1A', 'ABCA12', 'XDH', 'COL11A1', 'COL22A1',
    'MYO3B', 'FBN2', 'KIF14', 'FRAS1', 'CENPE', 'EPS8L3',
    'SMC1B', 'VIL1', 'ABCA4',
}

gate_summary = {}

for cancer, normal in PAIRINGS:
    print(f"\n{'='*60}")
    print(f"{cancer} vs {normal}")
    print(f"{'='*60}")

    tumor_df = pd.read_parquet(os.path.join(PROC_GENE, f'TCGA_{cancer}_expression.parquet'))
    normal_df = pd.read_parquet(os.path.join(PROC_GENE, f'GTEX_{normal}_expression.parquet'))

    # Rename normal_df columns BEFORE merge to avoid suffix confusion
    normal_subset = normal_df[['gene_id', 'median_tpm', 'is_expressed']].copy()
    normal_subset.columns = ['gene_id', 'median_tpm_normal', 'is_expressed_normal']

    # Rename tumor columns clearly too
    tumor_renamed = tumor_df.copy()
    tumor_renamed = tumor_renamed.rename(columns={
        'median_tpm': 'median_tpm_tumor',
        'is_expressed': 'is_expressed_tumor',
    })

    # Merge
    merged = tumor_renamed.merge(normal_subset, on='gene_id', how='inner')

    # Verify columns
    expected_cols = ['gene_id', 'gene_name', 'gene_type', 'bp_length',
                     'median_tpm_tumor', 'is_expressed_tumor',
                     'median_tpm_normal', 'is_expressed_normal']
    missing = [c for c in expected_cols if c not in merged.columns]
    if missing:
        print(f"  ERROR: missing columns {missing}")
        print(f"  Available: {list(merged.columns)}")
        break

    n_tumor_expr = merged['is_expressed_tumor'].sum()
    n_normal_expr = merged['is_expressed_normal'].sum()
    n_both = (merged['is_expressed_tumor'] & merged['is_expressed_normal']).sum()
    n_tumor_only = (merged['is_expressed_tumor'] & ~merged['is_expressed_normal']).sum()
    n_normal_only = (~merged['is_expressed_tumor'] & merged['is_expressed_normal']).sum()
    n_neither = (~merged['is_expressed_tumor'] & ~merged['is_expressed_normal']).sum()

    print(f"\n  Expression cross-tabulation:")
    print(f"    Tumor only:        {n_tumor_only:>6,}  (would have been Path A false positives)")
    print(f"    Normal only:       {n_normal_only:>6,}")
    print(f"    Both (eligible):   {n_both:>6,}  (Path B eligible for delta-PSI)")
    print(f"    Neither:           {n_neither:>6,}")

    # Protein-coding subset
    pc_mask = merged['gene_type'] == 'protein_coding'
    n_pc_both = (pc_mask & merged['is_expressed_tumor'] & merged['is_expressed_normal']).sum()
    n_pc_total = pc_mask.sum()
    print(f"    Protein-coding eligible: {n_pc_both:,} of {n_pc_total:,} PC genes total")

    # ---- SANITY CHECK 1: Path A false positives ----
    print(f"\n  Path A false-positive check (should be 'tumor only' or 'neither'):")
    fp_results = []
    fp_filtered_out = 0
    fp_still_eligible = 0
    for gname in sorted(PATH_A_FALSE_POSITIVES):
        gene_rows = merged[merged['gene_name'] == gname]
        if len(gene_rows) == 0:
            print(f"    {gname:12s} NOT IN MERGED (gene not in annotation)")
            continue
        r = gene_rows.iloc[0]
        tumor_e = r['is_expressed_tumor']
        normal_e = r['is_expressed_normal']
        eligible = tumor_e and normal_e

        if eligible:
            fp_still_eligible += 1
            marker = " <-- STILL ELIGIBLE"
        else:
            fp_filtered_out += 1
            marker = " <-- FILTERED OUT"

        tumor_str = f"expr" if tumor_e else "NOT"
        normal_str = f"expr" if normal_e else "NOT"
        print(f"    {gname:12s} tumor={r['median_tpm_tumor']:>9.2f} ({tumor_str:>4}) "
              f"normal={r['median_tpm_normal']:>9.2f} ({normal_str:>4}){marker}")
        fp_results.append({
            'gene': gname,
            'tumor_tpm': float(r['median_tpm_tumor']),
            'normal_tpm': float(r['median_tpm_normal']),
            'tumor_expressed': bool(tumor_e),
            'normal_expressed': bool(normal_e),
            'eligible': bool(eligible),
        })

    print(f"\n    Summary: {fp_filtered_out} filtered out, {fp_still_eligible} still eligible")

    # ---- SANITY CHECK 2: Hallmark gene preservation ----
    print(f"\n  Hallmark cancer-splicing gene check (should be eligible):")
    hallmark_eligible = 0
    hallmark_total_present = 0
    hallmark_details = []
    for gname in sorted(CANCER_SPLICING_GENES):
        gene_rows = merged[merged['gene_name'] == gname]
        if len(gene_rows) == 0:
            continue
        hallmark_total_present += 1
        r = gene_rows.iloc[0]
        eligible = r['is_expressed_tumor'] and r['is_expressed_normal']
        hallmark_details.append({
            'gene': gname,
            'tumor_tpm': float(r['median_tpm_tumor']),
            'normal_tpm': float(r['median_tpm_normal']),
            'eligible': bool(eligible),
        })
        if eligible:
            hallmark_eligible += 1

    print(f"    {hallmark_eligible} of {hallmark_total_present} hallmark genes are eligible")
    not_eligible = [h for h in hallmark_details if not h['eligible']]
    if not_eligible:
        print(f"    Hallmarks NOT eligible (lost from delta-PSI):")
        for h in not_eligible:
            print(f"      {h['gene']:8s} tumor={h['tumor_tpm']:>8.2f}  normal={h['normal_tpm']:>8.2f}")

    # ---- Save eligible-gene set ----
    eligible_df = merged[merged['is_expressed_tumor'] & merged['is_expressed_normal']].copy()
    eligible_df = eligible_df[['gene_id', 'gene_name', 'gene_type', 'bp_length',
                                'median_tpm_tumor', 'median_tpm_normal']]

    output_path = os.path.join(PROC_GENE, f'eligible_genes_{cancer}_vs_{normal}.parquet')
    eligible_df.to_parquet(output_path, index=False, compression='snappy')
    print(f"\n  Saved: {os.path.basename(output_path)} ({len(eligible_df):,} eligible genes)")

    gate_summary[f'{cancer}_vs_{normal}'] = {
        'cancer': cancer,
        'normal': normal,
        'n_eligible_all': int(n_both),
        'n_eligible_protein_coding': int(n_pc_both),
        'n_tumor_only_filtered_out': int(n_tumor_only),
        'n_normal_only_filtered_out': int(n_normal_only),
        'hallmark_eligible': hallmark_eligible,
        'hallmark_total_in_pairing': hallmark_total_present,
        'path_a_false_positives_filtered': fp_filtered_out,
        'path_a_false_positives_still_eligible': fp_still_eligible,
        'path_a_fp_details': fp_results,
        'hallmark_details': hallmark_details,
    }

# ---- Final summary ----
summary_path = os.path.join(PROC_GENE, 'eligible_genes_summary.json')
with open(summary_path, 'w') as f:
    json.dump(gate_summary, f, indent=2, default=str)

print("\n" + "=" * 60)
print("PATH B WEEK 1 GATE SUMMARY")
print(f"{'Pairing':<22} {'Eligible (PC)':>14} {'Path A FP killed':>20} {'Hallmarks':>14}")
for key, r in gate_summary.items():
    pairing = f"{r['cancer']} vs {r['normal']}"
    hallmark_str = f"{r['hallmark_eligible']}/{r['hallmark_total_in_pairing']}"
    fp_str = f"{r['path_a_false_positives_filtered']}/{r['path_a_false_positives_filtered']+r['path_a_false_positives_still_eligible']}"
    print(f"{pairing:<22} {r['n_eligible_protein_coding']:>14,} {fp_str:>20} {hallmark_str:>14}")

print(f"\nSaved: {summary_path}")
print("\n=== PATH B WEEK 1 COMPLETE ===")


---
## Section 2 -- Stage B: SUPPA2 cassette-exon event catalog

Installs SUPPA2 from source (Keras-3 compatible fork), generates cassette-exon (SE) event catalog from GENCODE v26, and maps SUPPA2 event coordinates to recount3 junction identifiers. Only SE events are analyzed; other event classes (A5, A3, MX, RI, AF, AL) are reserved for separate work.

In [ ]:
import subprocess, sys, os

print("Installing SUPPA2 from GitHub")

# First, clone or download from GitHub
SUPPA_DIR = '/content/SUPPA2'

if not os.path.exists(SUPPA_DIR):
    print("Cloning SUPPA2 from GitHub...")
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/comprna/SUPPA.git', SUPPA_DIR],
        capture_output=True, text=True, timeout=120
    )
    print(f"  Return code: {result.returncode}")
    if result.stdout: print(f"  stdout: {result.stdout}")
    if result.stderr: print(f"  stderr: {result.stderr[:500]}")
else:
    print(f"  SUPPA2 already cloned at {SUPPA_DIR}")

# Inspect what we got
print(f"\n--- Contents of {SUPPA_DIR} ---")
if os.path.exists(SUPPA_DIR):
    for item in sorted(os.listdir(SUPPA_DIR))[:30]:
        full = os.path.join(SUPPA_DIR, item)
        if os.path.isfile(full):
            size_kb = os.path.getsize(full) / 1024
            print(f"  [FILE] {item} ({size_kb:.1f} KB)")
        else:
            print(f"  [DIR]  {item}")

# Find the main suppa.py script
print("\n--- Find suppa.py ---")
result = subprocess.run(
    ['find', SUPPA_DIR, '-name', 'suppa.py', '-type', 'f'],
    capture_output=True, text=True
)
print(result.stdout)

# Try running help
SUPPA_PY = os.path.join(SUPPA_DIR, 'suppa.py')
if os.path.exists(SUPPA_PY):
    print(f"\n--- Try {SUPPA_PY} --help ---")
    result = subprocess.run(
        [sys.executable, SUPPA_PY, '--help'],
        capture_output=True, text=True, timeout=30
    )
    print(f"Return code: {result.returncode}")
    if result.stdout:
        print(f"stdout (first 2500 chars):\n{result.stdout[:2500]}")
    if result.stderr:
        print(f"stderr (first 500 chars):\n{result.stderr[:500]}")
else:
    print(f"\n  suppa.py not found at expected location")
    # Scan more deeply
    result = subprocess.run(
        ['find', SUPPA_DIR, '-name', '*.py', '-type', 'f', '-not', '-path', '*/test*'],
        capture_output=True, text=True
    )
    print("\nAll .py files in SUPPA2 (excluding tests):")
    print(result.stdout[:3000])

# Check Python dependencies SUPPA2 needs
print("\n--- SUPPA2 dependency check ---")
deps = ['pandas', 'numpy', 'scipy', 'statsmodels']
for d in deps:
    try:
        m = __import__(d)
        v = getattr(m, '__version__', '?')
        print(f"  [OK] {d} {v}")
    except ImportError:
        print(f"  [MISSING] {d}")


In [ ]:
import subprocess, sys, os, time, gzip, shutil

SUPPA_PY = '/content/SUPPA2/suppa.py'

# SUPPA2 requires UNCOMPRESSED GTF as input
GTF_GZ = os.path.join(REF_DIR, 'gencode.v26.annotation.gtf.gz')
GTF_UNCOMPRESSED = '/content/gencode.v26.annotation.gtf'

# Decompress GTF to /content (faster I/O than Drive, and SUPPA2 needs uncompressed)
if not os.path.exists(GTF_UNCOMPRESSED):
    print(f"Decompressing GTF to /content...")
    t0 = time.time()
    with gzip.open(GTF_GZ, 'rb') as f_in:
        with open(GTF_UNCOMPRESSED, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    size_mb = os.path.getsize(GTF_UNCOMPRESSED) / 1024**2
    print(f"[{time.time()-t0:.1f}s] Decompressed: {size_mb:.1f} MB")
else:
    print(f"Decompressed GTF already exists")

# Output paths (on Drive so results survive Colab disconnect)
SUPPA_OUT_DIR = os.path.join(REF_DIR, 'suppa_events')
os.makedirs(SUPPA_OUT_DIR, exist_ok=True)
OUTPUT_PREFIX = os.path.join(SUPPA_OUT_DIR, 'gencode_v26')

print(f"\n{'='*60}")
print(f"Running SUPPA2 generateEvents on GENCODE v26")
print(f"{'='*60}")
print(f"Input GTF: {GTF_UNCOMPRESSED}")
print(f"Output prefix: {OUTPUT_PREFIX}")
print(f"Event type: SE (Skipped Exon / cassette exon)")
print(f"Format: ioe (Inclusion/Exclusion of Events)")
print(f"Boundary: S (Strict, the default)")
print(f"Pool genes: enabled (handles overlapping genes correctly)")
print(f"")
print(f"Expected runtime: 5-15 minutes")
print(f"You can step away while this runs.")
print()

t0 = time.time()
result = subprocess.run(
    [sys.executable, SUPPA_PY, 'generateEvents',
     '-i', GTF_UNCOMPRESSED,
     '-o', OUTPUT_PREFIX,
     '-e', 'SE',
     '-f', 'ioe',
     '-b', 'S',
     '-p',  # pool overlapping genes (recommended)
     '-m', 'INFO'],
    capture_output=True, text=True, timeout=3600
)
elapsed = time.time() - t0

print(f"\nReturn code: {result.returncode}")
print(f"Elapsed: {elapsed/60:.1f} minutes")

if result.stdout:
    # Print last 3000 chars of stdout (most informative for SUPPA2 progress logs)
    print(f"\n--- stdout (last 3000 chars) ---")
    print(result.stdout[-3000:])

if result.stderr:
    print(f"\n--- stderr (last 2000 chars) ---")
    print(result.stderr[-2000:])

# Inspect outputs
print(f"\n--- Output files in {SUPPA_OUT_DIR} ---")
for f in sorted(os.listdir(SUPPA_OUT_DIR)):
    full = os.path.join(SUPPA_OUT_DIR, f)
    size_kb = os.path.getsize(full) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")

# Peek at the .ioe file (the main event catalog)
ioe_file = OUTPUT_PREFIX + '_SE_strict.ioe'
if os.path.exists(ioe_file):
    print(f"\n--- First 5 lines of {os.path.basename(ioe_file)} ---")
    with open(ioe_file) as f:
        for i, line in enumerate(f):
            if i < 5:
                print(line.rstrip())
            else:
                break
    # Total event count
    with open(ioe_file) as f:
        n_events = sum(1 for _ in f) - 1  # subtract header
    print(f"\nTotal SE events in catalog: {n_events:,}")


In [ ]:
# Then verify match rate against our actual recount3 junction data

import pandas as pd
import numpy as np
import os
import re

print("STEP 2.4: Map SUPPA2 events to recount3 junction_ids")

# ---- Load SUPPA2 event catalog ----
SUPPA_IOE = os.path.join(REF_DIR, 'suppa_events', 'gencode_v26_SE_strict.ioe')
ioe = pd.read_csv(SUPPA_IOE, sep='\t')
print(f"\nLoaded {len(ioe):,} SE events from SUPPA2 catalog")
print(f"Columns: {list(ioe.columns)}")
print(f"\nFirst 3 rows:")
print(ioe.head(3).to_string())

# ---- Parse SUPPA2 event_id into junction coordinates ----
# Format: GENE_ID;SE:chr:e1-s2:e2-s3:strand
# Skip junction: chr:e1-s3:strand (in recount3 intron coords: e1+1 to s3-1)
# Include junction 1: chr:e1-s2:strand (in recount3 intron coords: e1+1 to s2-1)
# Include junction 2: chr:e2-s3:strand (in recount3 intron coords: e2+1 to s3-1)

EVENT_PATTERN = re.compile(
    r'^(?P<gene>[^;]+);SE:(?P<chr>[^:]+):(?P<e1>\d+)-(?P<s2>\d+):(?P<e2>\d+)-(?P<s3>\d+):(?P<strand>[+-])$'
)

def parse_event(event_id):
    m = EVENT_PATTERN.match(event_id)
    if not m:
        return None
    d = m.groupdict()
    chrom = d['chr']
    e1, s2, e2, s3 = int(d['e1']), int(d['s2']), int(d['e2']), int(d['s3'])
    strand = d['strand']

    # recount3 junction_ids use the intron coordinates
    # (start of intron = e+1; end of intron = s-1)
    # Format: chr:start-end:strand
    skip_jid    = f"{chrom}:{e1+1}-{s3-1}:{strand}"
    include1_jid = f"{chrom}:{e1+1}-{s2-1}:{strand}"
    include2_jid = f"{chrom}:{e2+1}-{s3-1}:{strand}"

    return {
        'event_id': event_id,
        'gene_id': d['gene'],
        'chrom': chrom,
        'strand': strand,
        'e1': e1, 's2': s2, 'e2': e2, 's3': s3,
        'skip_jid': skip_jid,
        'include1_jid': include1_jid,
        'include2_jid': include2_jid,
    }

print("\nParsing SUPPA2 event IDs to junction coordinates...")
parsed = []
n_parse_fail = 0
for ev in ioe['event_id']:
    p = parse_event(ev)
    if p is None:
        n_parse_fail += 1
    else:
        parsed.append(p)

events_df = pd.DataFrame(parsed)
print(f"  Parsed: {len(events_df):,} events")
print(f"  Parse failures: {n_parse_fail}")
print(f"\nFirst 3 parsed events:")
print(events_df[['event_id', 'skip_jid', 'include1_jid', 'include2_jid']].head(3).to_string())

# ---- Verify against actual recount3 junction data (using BLCA as test cohort) ----
print("\n" + "=" * 60)
print("VERIFICATION: do SUPPA2 junction coordinates match recount3?")

blca_jx = pd.read_parquet(os.path.join(PROC_JX, 'filtered_junctions_BLCA.parquet'))
print(f"\nBLCA filtered junctions: {len(blca_jx):,}")

# We need ALL recount3 junctions (not just BLCA filtered), because SUPPA2 events
# may correspond to junctions present in OTHER cohorts but filtered out of BLCA.
# So we load the raw BLCA coords to get the full junction universe for that cohort.
print("Loading raw BLCA coordinates for full junction universe...")
raw_coords = pd.read_csv(os.path.join(RAW_TCGA, 'TCGA_BLCA_jxn_coords.csv'), low_memory=False)
raw_coords['junction_id'] = (
    raw_coords['seqnames'].astype(str) + ':' +
    raw_coords['start'].astype(str) + '-' +
    raw_coords['end'].astype(str) + ':' +
    raw_coords['strand'].astype(str)
)
recount3_junction_set = set(raw_coords['junction_id'])
print(f"  Total junctions in raw BLCA: {len(recount3_junction_set):,}")

# Match rates
n_skip_match = events_df['skip_jid'].isin(recount3_junction_set).sum()
n_inc1_match = events_df['include1_jid'].isin(recount3_junction_set).sum()
n_inc2_match = events_df['include2_jid'].isin(recount3_junction_set).sum()
n_all3_match = (
    events_df['skip_jid'].isin(recount3_junction_set) &
    events_df['include1_jid'].isin(recount3_junction_set) &
    events_df['include2_jid'].isin(recount3_junction_set)
).sum()

print(f"\nMatch rates against raw BLCA junction universe:")
print(f"  Skip junctions:     {n_skip_match:>7,} of {len(events_df):,} ({100*n_skip_match/len(events_df):.1f}%)")
print(f"  Include1 junctions: {n_inc1_match:>7,} of {len(events_df):,} ({100*n_inc1_match/len(events_df):.1f}%)")
print(f"  Include2 junctions: {n_inc2_match:>7,} of {len(events_df):,} ({100*n_inc2_match/len(events_df):.1f}%)")
print(f"  All 3 junctions:    {n_all3_match:>7,} of {len(events_df):,} ({100*n_all3_match/len(events_df):.1f}%)")


---
## Section 3 -- Stage C: per-sample PSI computation with coverage gates

Computes PSI per sample per event as `PSI = (inclusion_reads / 2) / ((inclusion_reads / 2) + skipping_reads)`. Applies per-sample coverage gate (>= 10 informative reads, Kahles 2018 floor) and per-event coverage gate (>= 20 well-covered samples per group, Buen Abad Najar 2020).

Includes a MET exon 14 unit test that reproduces the ~3% LUAD skipping rate reported in Frampton/Paik 2015, verifying the pipeline correctly identifies known cancer-splicing biology.

In [ ]:
# Part A: Parse SUPPA2 events into junction-coordinate triplets
print("Part A: Parse SUPPA2 SE event catalog")

SUPPA_IOE = os.path.join(SUPPA_OUT, 'gencode_v26_SE_strict.ioe')
ioe = pd.read_csv(SUPPA_IOE, sep='\t')
print(f"Loaded {len(ioe):,} SE events")

EVENT_PATTERN = re.compile(
    r'^(?P<gene>[^;]+);SE:(?P<chr>[^:]+):(?P<e1>\d+)-(?P<s2>\d+):(?P<e2>\d+)-(?P<s3>\d+):(?P<strand>[+-])$'
)

def parse_event(event_id):
    m = EVENT_PATTERN.match(event_id)
    if not m:
        return None
    d = m.groupdict()
    chrom, strand = d['chr'], d['strand']
    e1, s2, e2, s3 = int(d['e1']), int(d['s2']), int(d['e2']), int(d['s3'])
    return {
        'event_id': event_id,
        'gene_id': d['gene'],
        'chrom': chrom, 'strand': strand,
        'cassette_start': s2, 'cassette_end': e2,
        'skip_jid':     f"{chrom}:{e1+1}-{s3-1}:{strand}",
        'include1_jid': f"{chrom}:{e1+1}-{s2-1}:{strand}",
        'include2_jid': f"{chrom}:{e2+1}-{s3-1}:{strand}",
    }

events = pd.DataFrame([p for p in (parse_event(ev) for ev in ioe['event_id']) if p is not None])
print(f"Parsed {len(events):,} events with junction coordinates")


# Part B: Load LUAD + LUNG sparse matrices, build junction_id -> row_index map
print("\n" + "=" * 60)
print("Part B: Load LUAD + LUNG junction matrices via rpy2")

def load_cohort(label, rds_path, coords_path):
    print(f"\n--- Loading {label} ---")
    t0 = time.time()

    # Load sparse matrix from R
    R(f'''
    suppressMessages(library(Matrix))
    jx_sparse <- readRDS("{rds_path}")
    class_name <- class(jx_sparse)[1]
    if (class_name == "dgCMatrix") {{
      coo <- as(jx_sparse, "TsparseMatrix")
      i_vec <- coo@i; j_vec <- coo@j; x_vec <- coo@x
      rm(coo)
    }} else {{
      i_vec <- jx_sparse@i; j_vec <- jx_sparse@j; x_vec <- jx_sparse@x
    }}
    n_rows <- nrow(jx_sparse); n_cols <- ncol(jx_sparse)
    ''')
    i_vec = np.asarray(ro.r('i_vec'), dtype=np.int64)
    j_vec = np.asarray(ro.r('j_vec'), dtype=np.int64)
    x_vec = np.asarray(ro.r('x_vec'), dtype=np.float32)
    n_rows = int(ro.r('n_rows')[0]); n_cols = int(ro.r('n_cols')[0])
    M = coo_matrix((x_vec, (i_vec, j_vec)), shape=(n_rows, n_cols)).tocsr()
    R('rm(jx_sparse, i_vec, j_vec, x_vec); gc()')
    del i_vec, j_vec, x_vec

    # Load coords and build junction_id -> row lookup
    coords = pd.read_csv(coords_path, low_memory=False)
    coords['junction_id'] = (
        coords['seqnames'].astype(str) + ':' +
        coords['start'].astype(str) + '-' +
        coords['end'].astype(str) + ':' +
        coords['strand'].astype(str)
    )
    jid_to_row = dict(zip(coords['junction_id'], coords.index))

    print(f"  [{time.time()-t0:.1f}s] {M.shape}, {len(jid_to_row):,} unique junction_ids")
    return M, jid_to_row, coords

luad_M, luad_jid_to_row, luad_coords = load_cohort(
    'LUAD',
    os.path.join(RAW_TCGA, 'TCGA_LUAD_jxn_counts.rds'),
    os.path.join(RAW_TCGA, 'TCGA_LUAD_jxn_coords.csv'))

lung_M, lung_jid_to_row, lung_coords = load_cohort(
    'LUNG',
    os.path.join(RAW_GTEX, 'GTEx_LUNG_jxn_counts.rds'),
    os.path.join(RAW_GTEX, 'GTEx_LUNG_jxn_coords.csv'))


# Part C: For each event, compute per-sample inclusion + skipping reads
print("\n" + "=" * 60)
print("Part C: Compute per-sample PSI for events found in both LUAD and LUNG")

def compute_psi_for_cohort(events_df, M, jid_to_row, cohort_label):
    """
    For each event, gather skip/include1/include2 read vectors across samples.
    PSI = (include1 + include2) / 2 / (((include1 + include2) / 2) + skip)
        = (include1 + include2) / (include1 + include2 + 2*skip)

    Standard SUPPA2 PSI formula for SE events.
    Sample-level filter: total reads (include1 + include2 + skip) >= 10
    """
    t0 = time.time()
    n_samples = M.shape[1]
    n_events = len(events_df)

    # PSI matrix: NaN where coverage too low
    psi = np.full((n_events, n_samples), np.nan, dtype=np.float32)
    # Coverage matrix (total reads supporting the event in each sample)
    total_reads = np.zeros((n_events, n_samples), dtype=np.float32)
    # Track which events have all 3 junctions present in this cohort
    has_all3 = np.zeros(n_events, dtype=bool)

    for i, row in events_df.iterrows():
        s_row = jid_to_row.get(row['skip_jid'])
        i1_row = jid_to_row.get(row['include1_jid'])
        i2_row = jid_to_row.get(row['include2_jid'])

        # If any junction is missing entirely from this cohort, event not computable
        if s_row is None or i1_row is None or i2_row is None:
            continue
        has_all3[i] = True

        # Read vectors across samples (each is shape (n_samples,))
        skip_reads = np.asarray(M[s_row].todense()).flatten()
        inc1_reads = np.asarray(M[i1_row].todense()).flatten()
        inc2_reads = np.asarray(M[i2_row].todense()).flatten()

        # SUPPA2 SE PSI formula:
        # PSI = (inc1 + inc2) / (inc1 + inc2 + 2*skip)
        numerator = inc1_reads + inc2_reads
        denominator = numerator + 2 * skip_reads
        total = inc1_reads + inc2_reads + skip_reads
        total_reads[i] = total

        # Compute PSI where coverage >= 10 (locked decision)
        valid = total >= 10
        psi[i, valid] = (numerator[valid] / denominator[valid]).astype(np.float32)

        if (i + 1) % 5000 == 0:
            print(f"  [{time.time()-t0:.1f}s] Processed {i+1:,} / {n_events:,} events...")

    print(f"  [{time.time()-t0:.1f}s] Done. Events with all 3 junctions in {cohort_label}: "
          f"{has_all3.sum():,} of {n_events:,} ({100*has_all3.sum()/n_events:.1f}%)")
    return psi, total_reads, has_all3


print("\nComputing PSI for LUAD...")
luad_psi, luad_cov, luad_has3 = compute_psi_for_cohort(events, luad_M, luad_jid_to_row, 'LUAD')

print("\nComputing PSI for LUNG...")
lung_psi, lung_cov, lung_has3 = compute_psi_for_cohort(events, lung_M, lung_jid_to_row, 'LUNG')

# Events testable in both cohorts
testable_both = luad_has3 & lung_has3
print(f"\nEvents with all 3 junctions in BOTH LUAD and LUNG: {testable_both.sum():,}")


# Part D: MET exon 14 unit test
print("\n" + "=" * 60)
print("Part D: MET exon 14 unit test")

# MET gene_id in GENCODE v26: ENSG00000105976 (with version suffix)
# Find all SE events in MET
met_mask = events['gene_id'].str.startswith('ENSG00000105976')
met_events = events[met_mask].copy()
print(f"\nAll SE events in MET (any cassette exon):")
print(f"  Total: {len(met_events)}")
print(met_events[['event_id', 'chrom', 'cassette_start', 'cassette_end']].to_string())

# MET exon 14 genomic coordinates (GRCh38, GENCODE v26):
# MET is on chr7, plus strand
# Exon 14 is approximately chr7:116771976-116772116 (140 bp)
# The cassette in our SUPPA2 catalog should have those start/end positions
# (or very close, depending on annotation version specifics)
print(f"\n--- MET exon 14 lookup (expected ~chr7:116,771,976-116,772,116) ---")
met_chr7 = met_events[met_events['chrom'] == 'chr7']
# Filter to events with cassette in approximately the right range
exon14_candidates = met_chr7[
    (met_chr7['cassette_start'] >= 116771900) &
    (met_chr7['cassette_end'] <= 116772200)
]
print(f"MET cassette events with start/end in expected exon 14 region:")
if len(exon14_candidates) == 0:
    print("  NONE FOUND. Showing all chr7 MET events to diagnose:")
    print(met_chr7[['event_id', 'cassette_start', 'cassette_end']].to_string())
else:
    print(exon14_candidates[['event_id', 'cassette_start', 'cassette_end',
                              'skip_jid', 'include1_jid', 'include2_jid']].to_string())

    # For each candidate exon 14 event, compute PSI distribution in LUAD vs LUNG
    print("\n--- PSI behavior for MET exon 14 candidates ---")
    for idx, ev in exon14_candidates.iterrows():
        event_idx = events.index.get_loc(idx)  # row index in our PSI matrix

        if not (luad_has3[event_idx] and lung_has3[event_idx]):
            print(f"\nEvent {ev['event_id']}: NOT testable (missing junctions)")
            print(f"  LUAD has all 3: {luad_has3[event_idx]}, LUNG has all 3: {lung_has3[event_idx]}")
            continue

        luad_psi_vec = luad_psi[event_idx]
        lung_psi_vec = lung_psi[event_idx]
        luad_valid = ~np.isnan(luad_psi_vec)
        lung_valid = ~np.isnan(lung_psi_vec)

        print(f"\nEvent: {ev['event_id']}")
        print(f"  Cassette coords: chr7:{ev['cassette_start']}-{ev['cassette_end']}")
        print(f"  Skip jid: {ev['skip_jid']}")
        print(f"  LUAD samples with computable PSI: {luad_valid.sum()} of {len(luad_psi_vec)}")
        print(f"    PSI distribution: min={luad_psi_vec[luad_valid].min():.3f}, "
              f"q25={np.percentile(luad_psi_vec[luad_valid], 25):.3f}, "
              f"median={np.median(luad_psi_vec[luad_valid]):.3f}, "
              f"q75={np.percentile(luad_psi_vec[luad_valid], 75):.3f}, "
              f"max={luad_psi_vec[luad_valid].max():.3f}")
        print(f"    Samples with PSI <0.9 (substantial skipping): "
              f"{(luad_psi_vec[luad_valid] < 0.9).sum()} ({100*(luad_psi_vec[luad_valid] < 0.9).sum()/luad_valid.sum():.1f}%)")
        print(f"  LUNG samples with computable PSI: {lung_valid.sum()} of {len(lung_psi_vec)}")
        print(f"    PSI distribution: min={lung_psi_vec[lung_valid].min():.3f}, "
              f"median={np.median(lung_psi_vec[lung_valid]):.3f}, "
              f"max={lung_psi_vec[lung_valid].max():.3f}")

# Save PSI matrices for future use
print("\n" + "=" * 60)
print("Saving PSI matrices")

np.savez_compressed(
    os.path.join(PROC_PSI, 'LUAD_psi.npz'),
    psi=luad_psi, coverage=luad_cov, has_all3=luad_has3
)
np.savez_compressed(
    os.path.join(PROC_PSI, 'LUNG_psi.npz'),
    psi=lung_psi, coverage=lung_cov, has_all3=lung_has3
)
events.to_parquet(os.path.join(PROC_PSI, 'suppa_events_parsed.parquet'), index=False)

print(f"  Saved LUAD_psi.npz ({os.path.getsize(os.path.join(PROC_PSI, 'LUAD_psi.npz')) / 1024**2:.1f} MB)")
print(f"  Saved LUNG_psi.npz ({os.path.getsize(os.path.join(PROC_PSI, 'LUNG_psi.npz')) / 1024**2:.1f} MB)")
print(f"  Saved events parsed parquet")

print("\n" + "=" * 60)
print("STEP 2.5 + MET EXON 14 TEST COMPLETE")


In [ ]:
# Same code path proven on LUAD/LUNG, just different cohorts

print("STEP 2.6-2.7: PSI computation for BLCA, BLADDER, UCEC, UTERUS")

# Same load_cohort and compute_psi_for_cohort functions from previous cell
# (functions are still defined in memory from Cell 2)

# ---- BLCA ----
blca_M, blca_jid_to_row, blca_coords = load_cohort(
    'BLCA',
    os.path.join(RAW_TCGA, 'TCGA_BLCA_jxn_counts.rds'),
    os.path.join(RAW_TCGA, 'TCGA_BLCA_jxn_coords.csv'))

blca_psi, blca_cov, blca_has3 = compute_psi_for_cohort(events, blca_M, blca_jid_to_row, 'BLCA')

np.savez_compressed(
    os.path.join(PROC_PSI, 'BLCA_psi.npz'),
    psi=blca_psi, coverage=blca_cov, has_all3=blca_has3
)
print(f"  Saved BLCA_psi.npz")
del blca_M  # free memory

# ---- BLADDER ----
bladder_M, bladder_jid_to_row, bladder_coords = load_cohort(
    'BLADDER',
    os.path.join(RAW_GTEX, 'GTEx_BLADDER_jxn_counts.rds'),
    os.path.join(RAW_GTEX, 'GTEx_BLADDER_jxn_coords.csv'))

bladder_psi, bladder_cov, bladder_has3 = compute_psi_for_cohort(events, bladder_M, bladder_jid_to_row, 'BLADDER')

np.savez_compressed(
    os.path.join(PROC_PSI, 'BLADDER_psi.npz'),
    psi=bladder_psi, coverage=bladder_cov, has_all3=bladder_has3
)
print(f"  Saved BLADDER_psi.npz")
del bladder_M

# ---- UCEC ----
ucec_M, ucec_jid_to_row, ucec_coords = load_cohort(
    'UCEC',
    os.path.join(RAW_TCGA, 'TCGA_UCEC_jxn_counts.rds'),
    os.path.join(RAW_TCGA, 'TCGA_UCEC_jxn_coords.csv'))

ucec_psi, ucec_cov, ucec_has3 = compute_psi_for_cohort(events, ucec_M, ucec_jid_to_row, 'UCEC')

np.savez_compressed(
    os.path.join(PROC_PSI, 'UCEC_psi.npz'),
    psi=ucec_psi, coverage=ucec_cov, has_all3=ucec_has3
)
print(f"  Saved UCEC_psi.npz")
del ucec_M

# ---- UTERUS ----
uterus_M, uterus_jid_to_row, uterus_coords = load_cohort(
    'UTERUS',
    os.path.join(RAW_GTEX, 'GTEx_UTERUS_jxn_counts.rds'),
    os.path.join(RAW_GTEX, 'GTEx_UTERUS_jxn_coords.csv'))

uterus_psi, uterus_cov, uterus_has3 = compute_psi_for_cohort(events, uterus_M, uterus_jid_to_row, 'UTERUS')

np.savez_compressed(
    os.path.join(PROC_PSI, 'UTERUS_psi.npz'),
    psi=uterus_psi, coverage=uterus_cov, has_all3=uterus_has3
)
print(f"  Saved UTERUS_psi.npz")
del uterus_M

# ---- Summary table ----
print("\n" + "=" * 60)
print("ALL COHORTS PSI SUMMARY")
print(f"{'Cohort':<10} {'Samples':>8} {'Events all-3':>14} {'% events':>10}")
data = [
    ('LUAD', 601, luad_has3),
    ('LUNG', 655, lung_has3),
    ('BLCA', 433, blca_has3),
    ('BLADDER', 21, bladder_has3),
    ('UCEC', 589, ucec_has3),
    ('UTERUS', 159, uterus_has3),
]
for label, ns, has3 in data:
    n = has3.sum()
    pct = 100 * n / len(has3)
    print(f"{label:<10} {ns:>8} {n:>14,} {pct:>9.1f}%")


---
## Section 4 -- Stage D: dispersion-aware beta-binomial GLM

Fits a per-event beta-binomial GLM with logit link, separate inclusion-rate means for tumor and normal groups, and shared dispersion parameter, tested via likelihood-ratio against a shared-mean null (chi-squared, df=1). Correction over pooled-binomial LRT: reduces false-positive rate from 22-52% (under realistic biological overdispersion) to the calibrated 5% target.

Mann-Whitney rank-sum on per-sample mean PSI serves as the sensitivity test; concordance is 87% / 75% / 81% across LUAD / BLCA / UCEC.

Applies the 3C concordance gate with D7 floor (`|dPSI_scn| >= 0.05`) to separate primary from secondary tier. **This cell writes the locked corrected parquets that all downstream analyses read.** Runtime: ~24 minutes total across the three cancers on standard Colab CPU.

In [ ]:
"""
stage_c_gate1_corrected.py
Paper 5A -- Stage C corrected differential-PSI pipeline (Gate 1).

Replaces the cell 64/67 pooled-binomial test per the methodology locked in
STAGE_C_FIX_PRECOMMIT.md (signed 2026-06-11) and STAGE_C_DEVIATIONS.md (Dev 1).

  Primary test:     beta-binomial GLM per event (logit link, fitted dispersion)
                    Likelihood ratio test against shared-mean null. Bisbee 2021.
  Sensitivity test: per-sample Mann-Whitney rank-sum on mean-PSI vectors.
  Point estimate:   mean per-sample PSI (replaces median).
  Coverage gate:    >=10 reads per sample (Kahles 2018);
                    >=20 well-covered samples per side (Buen Abad Najar 2020)
  Effect-size gate: |dPSI| >= 0.15  on mean-based dPSI.
  Multiple testing: Benjamini-Hochberg FDR (correctly labeled).
  3C concordance:   tumor-vs-SCN within cohort; >=5 well-covered SCN required.

USES EXISTING NPZ FILES from cell 60.
Integer (k, n) counts are reconstructed exactly from the stored float32 (psi, coverage):
  n = round(coverage),  k = round(psi * coverage)
Precision test confirmed 100% exact integer match. No re-extraction needed.

Outputs to: <PROC_PSI>/Stage2C_corrected/
  <CANCER>_corrected_results.parquet     -- all testable events with full stats + tier
  <CANCER>_corrected_primary.parquet      -- primary aberrant set (headline)
  <CANCER>_corrected_secondary.parquet    -- secondary aberrant set
  STAGE_C_CORRECTED_SUMMARY.txt           -- counts, M2' pass/fail, hallmark coverage

USAGE IN COLAB:
  Drive should already be mounted. Paste this file's contents into one Colab cell
  and run. Or: !python stage_c_gate1_corrected.py

  Expected runtime: ~25-30 min total.
    No RDS extraction (existing npz are used).
    Betabin GLM: ~8 min per cancer (~24 min total).
    Wilcoxon: ~1 min total.
"""

import os
import sys
import time
import pickle
import warnings
from datetime import date

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import betaln
from scipy.stats import chi2 as chi2_dist, mannwhitneyu
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore', category=RuntimeWarning)


# CONFIG -- paths (matches your actual Drive layout) and locked thresholds
PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
PROC_PSI     = os.path.join(PROJECT_ROOT, "data/processed/psi")
PROC_JX      = os.path.join(PROJECT_ROOT, "data/processed/junctions")
RAW_TCGA     = os.path.join(PROJECT_ROOT, "data/raw/tcga_recount3")
RAW_GTEX     = os.path.join(PROJECT_ROOT, "data/raw/gtex_recount3")
OUT_DIR      = os.path.join(PROC_PSI, "Stage2C_corrected")

PER_SAMPLE_READ_FLOOR  = 10
PER_EVENT_TUMOR_MIN    = 20
PER_EVENT_NORMAL_MIN   = 20
PER_EVENT_SCN_MIN      =  5
DPSI_THRESHOLD         = 0.15
Q_THRESHOLD            = 0.05

M2_PRIME_THRESHOLDS = {"LUAD": 500, "BLCA": 300, "UCEC": 300}
PREDICTION_RANGES   = {"LUAD": (400, 750), "BLCA": (350, 700), "UCEC": (350, 700)}
OLD_INFLATED_COUNTS = {"LUAD": 1266, "BLCA": 1047, "UCEC": 1070}

# (cancer, normal_label) -- npz files already exist for all of these
COHORT_PAIRS = [
    ("LUAD", "LUNG"),
    ("BLCA", "BLADDER"),
    ("UCEC", "UTERUS"),
]

HALLMARK_GENES = [
    "FAS", "CD44", "FGFR1", "FGFR2", "FGFR3", "BCL2L1", "MDM4", "MDM2", "VEGFA",
    "KLF6", "PKM", "MET", "EGFR", "AR", "TP53", "RAC1", "RON", "SYK", "TIA1",
    "NUMB", "CTNNB1", "BIN1", "CASP9", "CASP8", "TRAF3", "ESR1", "BRCA1", "PIK3CA",
]


# Beta-binomial GLM primitive
def _bb_neg_loglik(params, k_t, n_t, k_n, n_n, mode):
    if mode == 'alt':
        mu_t, mu_n, log_phi = params
    else:
        mu_s, log_phi = params
        mu_t = mu_n = mu_s
    if not (1e-6 < mu_t < 1 - 1e-6 and 1e-6 < mu_n < 1 - 1e-6):
        return 1e10
    phi = np.exp(log_phi)
    a_t, b_t = mu_t * phi, (1 - mu_t) * phi
    a_n, b_n = mu_n * phi, (1 - mu_n) * phi
    ll_t = (betaln(k_t + a_t, n_t - k_t + b_t) - betaln(a_t, b_t)).sum()
    ll_n = (betaln(k_n + a_n, n_n - k_n + b_n) - betaln(a_n, b_n)).sum()
    return -(ll_t + ll_n)


def betabin_lrt_one_event(k_t, n_t, k_n, n_n):
    if len(k_t) < 5 or len(k_n) < 5:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    mu_t0 = (k_t.sum() + 0.5) / (n_t.sum() + 1)
    mu_n0 = (k_n.sum() + 0.5) / (n_n.sum() + 1)
    mu_s0 = (k_t.sum() + k_n.sum() + 0.5) / (n_t.sum() + n_n.sum() + 1)
    try:
        r_alt = minimize(_bb_neg_loglik, [mu_t0, mu_n0, 3.0],
                         args=(k_t, n_t, k_n, n_n, 'alt'),
                         method='Nelder-Mead',
                         options={'xatol': 1e-4, 'maxiter': 200})
        r_null = minimize(_bb_neg_loglik, [mu_s0, 3.0],
                          args=(k_t, n_t, k_n, n_n, 'null'),
                          method='Nelder-Mead',
                          options={'xatol': 1e-4, 'maxiter': 200})
        chi2_stat = max(0.0, 2.0 * (r_null.fun - r_alt.fun))
        p_val = 1.0 - chi2_dist.cdf(chi2_stat, df=1)
        mu_t_hat, mu_n_hat, log_phi_hat = r_alt.x
        return chi2_stat, p_val, mu_t_hat, mu_n_hat, np.exp(log_phi_hat)
    except Exception:
        return np.nan, np.nan, np.nan, np.nan, np.nan


# Load existing npz; reconstruct integer counts from psi + coverage
def load_psi_and_counts(label):
    """Load PSI and coverage from existing cell-60 npz files; reconstruct integer
    (inclusion, total) read counts. Precision-test confirmed 100% exact integer match.

    Returns: (psi, total_int, inc_int, has_all3)
    """
    npz_path = os.path.join(PROC_PSI, f"{label}_psi.npz")
    if not os.path.exists(npz_path):
        sys.exit(f"NPZ not found: {npz_path}")
    z = np.load(npz_path)
    psi      = z['psi']                 # float32, NaN where coverage<10
    coverage = z['coverage']            # float32, raw total reads
    has_all3 = z['has_all3']            # bool
    print(f"  [{label}] loaded {os.path.basename(npz_path)} "
          f"shape={psi.shape}, has_all3={int(has_all3.sum()):,}")
    # Reconstruct integer counts (exact at float32 precision)
    total_int = np.rint(coverage).astype(np.int32)
    psi_safe  = np.where(np.isnan(psi), 0.0, psi).astype(np.float64)
    inc_int   = np.rint(psi_safe * total_int.astype(np.float64)).astype(np.int32)
    # Where PSI was NaN (low coverage), zero out inc_int. The downstream coverage
    # mask (total>=PER_SAMPLE_READ_FLOOR) will exclude these samples anyway.
    inc_int[np.isnan(psi)] = 0
    return psi, total_int, inc_int, has_all3


# Helpers
def cohens_d_with_nan(x, y):
    nx = np.sum(~np.isnan(x), axis=1)
    ny = np.sum(~np.isnan(y), axis=1)
    mx = np.nanmean(x, axis=1)
    my = np.nanmean(y, axis=1)
    vx = np.nanvar(x, axis=1, ddof=1)
    vy = np.nanvar(y, axis=1, ddof=1)
    denom = (nx + ny - 2)
    pooled_var = np.where(denom > 0, ((nx - 1) * vx + (ny - 1) * vy) / denom, np.nan)
    pooled_sd = np.sqrt(pooled_var)
    return np.where(pooled_sd > 0, (mx - my) / pooled_sd, np.nan)


def vectorized_wilcoxon(tumor_psi, normal_psi):
    res = mannwhitneyu(tumor_psi, normal_psi, axis=1,
                       alternative='two-sided', nan_policy='omit')
    return res.statistic, res.pvalue


# Main pipeline per cohort pair
def run_corrected_pipeline(cancer, normal_label,
                           tcga_psi, tcga_cov, tcga_inc,
                           gtex_psi, gtex_cov, gtex_inc,
                           scn_indices, events_df):
    print(f"\n{'=' * 64}")
    print(f"  CORRECTED STAGE C: {cancer} vs {normal_label}")
    print(f"{'=' * 64}")

    n_total = tcga_psi.shape[1]
    tumor_mask = np.ones(n_total, dtype=bool)
    tumor_mask[scn_indices] = False
    n_tumor = tumor_mask.sum()
    n_scn   = (~tumor_mask).sum()
    n_normal = gtex_psi.shape[1]
    print(f"  Samples: tumor={n_tumor}, SCN={n_scn}, GTEx-{normal_label}={n_normal}")

    tumor_psi   = tcga_psi[:, tumor_mask]
    scn_psi     = tcga_psi[:, ~tumor_mask]
    normal_psi  = gtex_psi

    tumor_inc   = tcga_inc[:, tumor_mask]
    tumor_cov   = tcga_cov[:, tumor_mask]
    normal_inc  = gtex_inc
    normal_cov  = gtex_cov

    n_tumor_wc  = np.sum(~np.isnan(tumor_psi),  axis=1)
    n_scn_wc    = np.sum(~np.isnan(scn_psi),    axis=1)
    n_normal_wc = np.sum(~np.isnan(normal_psi), axis=1)

    psi_tumor_mean    = np.nanmean(tumor_psi,  axis=1)
    psi_tumor_median  = np.nanmedian(tumor_psi, axis=1)
    psi_normal_mean   = np.nanmean(normal_psi, axis=1)
    psi_normal_median = np.nanmedian(normal_psi, axis=1)
    psi_scn_mean      = np.nanmean(scn_psi,    axis=1)

    delta_psi        = psi_tumor_mean   - psi_normal_mean
    delta_psi_median = psi_tumor_median - psi_normal_median
    delta_psi_scn    = psi_tumor_mean   - psi_scn_mean

    coverage_pass = (n_tumor_wc >= PER_EVENT_TUMOR_MIN) & \
                    (n_normal_wc >= PER_EVENT_NORMAL_MIN)
    print(f"  Events: total={len(events_df):,}, "
          f"coverage-pass (>={PER_EVENT_TUMOR_MIN} tumor & >={PER_EVENT_NORMAL_MIN} normal)={int(coverage_pass.sum()):,}")

    n_events = len(events_df)
    bb_chi2 = np.full(n_events, np.nan, dtype=np.float64)
    bb_pval = np.full(n_events, np.nan, dtype=np.float64)
    bb_mu_t = np.full(n_events, np.nan, dtype=np.float64)
    bb_mu_n = np.full(n_events, np.nan, dtype=np.float64)
    bb_phi  = np.full(n_events, np.nan, dtype=np.float64)

    idx_to_test = np.where(coverage_pass)[0]
    print(f"  Running beta-binomial GLM on {len(idx_to_test):,} events ...")
    t0 = time.time()
    for ii, i in enumerate(idx_to_test):
        valid_t = tumor_cov[i] >= PER_SAMPLE_READ_FLOOR
        valid_n = normal_cov[i] >= PER_SAMPLE_READ_FLOOR
        k_t = tumor_inc[i, valid_t].astype(np.int64)
        n_t = tumor_cov[i, valid_t].astype(np.int64)
        k_n = normal_inc[i, valid_n].astype(np.int64)
        n_n = normal_cov[i, valid_n].astype(np.int64)
        c2, p, mt, mn, ph = betabin_lrt_one_event(k_t, n_t, k_n, n_n)
        bb_chi2[i] = c2
        bb_pval[i] = p
        bb_mu_t[i] = mt
        bb_mu_n[i] = mn
        bb_phi[i]  = ph
        if (ii + 1) % 2000 == 0:
            elapsed = time.time() - t0
            eta = elapsed / (ii + 1) * (len(idx_to_test) - ii - 1)
            print(f"    [{elapsed:.0f}s] {ii+1:,} / {len(idx_to_test):,}  ETA {eta:.0f}s")
    print(f"  Beta-binomial GLM done in {time.time()-t0:.0f}s")

    mwu_u    = np.full(n_events, np.nan, dtype=np.float64)
    mwu_pval = np.full(n_events, np.nan, dtype=np.float64)
    if coverage_pass.any():
        print(f"  Running Wilcoxon (sensitivity) on {int(coverage_pass.sum()):,} events ...")
        t0 = time.time()
        u_sub, p_sub = vectorized_wilcoxon(tumor_psi[coverage_pass], normal_psi[coverage_pass])
        mwu_u[coverage_pass]    = u_sub
        mwu_pval[coverage_pass] = p_sub
        print(f"  Wilcoxon done in {time.time()-t0:.1f}s")

    cohens_d_arr = np.full(n_events, np.nan, dtype=np.float64)
    if coverage_pass.any():
        cohens_d_arr[coverage_pass] = cohens_d_with_nan(tumor_psi[coverage_pass],
                                                        normal_psi[coverage_pass])

    qvals = np.full(n_events, np.nan, dtype=np.float64)
    testable = ~np.isnan(bb_pval)
    if testable.any():
        _, q_sub, _, _ = multipletests(bb_pval[testable], method='fdr_bh', alpha=Q_THRESHOLD)
        qvals[testable] = q_sub

    has_scn = n_scn_wc >= PER_EVENT_SCN_MIN
    scn_concordant = np.zeros(n_events, dtype=bool)
    scn_discordant = np.zeros(n_events, dtype=bool)
    valid_signs = has_scn & ~np.isnan(delta_psi) & ~np.isnan(delta_psi_scn)
    scn_concordant[valid_signs] = (np.sign(delta_psi[valid_signs]) ==
                                   np.sign(delta_psi_scn[valid_signs])) & \
                                  (np.abs(delta_psi_scn[valid_signs]) >= 0.05)
    scn_discordant[valid_signs] = (np.sign(delta_psi[valid_signs]) !=
                                   np.sign(delta_psi_scn[valid_signs])) & \
                                  (np.abs(delta_psi_scn[valid_signs]) >= 0.05)

    passes_q    = (qvals < Q_THRESHOLD) & ~np.isnan(qvals)
    passes_dpsi = (np.abs(delta_psi) >= DPSI_THRESHOLD) & ~np.isnan(delta_psi)
    passes_all_stats = coverage_pass & passes_q & passes_dpsi

    tier = np.full(n_events, 'rejected', dtype=object)
    tier[passes_all_stats & has_scn & scn_concordant] = 'primary'
    tier[passes_all_stats & ~has_scn]                  = 'secondary'
    tier[passes_all_stats & has_scn & scn_discordant]  = 'scn_discordant'

    bb_sig = passes_q
    mw_sig = (mwu_pval < Q_THRESHOLD) & ~np.isnan(mwu_pval)
    both_significant = bb_sig & mw_sig
    print(f"  Test concordance: both={int(both_significant.sum()):,}, "
          f"BB-only={int((bb_sig & ~mw_sig).sum()):,}, "
          f"MW-only={int((~bb_sig & mw_sig).sum()):,}, "
          f"either={int((bb_sig | mw_sig).sum()):,}")

    out = events_df.copy()
    out['n_tumor_well_covered']   = n_tumor_wc
    out['n_normal_well_covered']  = n_normal_wc
    out['n_scn_well_covered']     = n_scn_wc
    out['psi_tumor_mean']         = psi_tumor_mean
    out['psi_normal_mean']        = psi_normal_mean
    out['psi_scn_mean']           = psi_scn_mean
    out['psi_tumor_median']       = psi_tumor_median
    out['psi_normal_median']      = psi_normal_median
    out['delta_psi']              = delta_psi
    out['delta_psi_median']       = delta_psi_median
    out['delta_psi_scn']          = delta_psi_scn
    out['abs_dpsi']               = np.abs(delta_psi)
    out['cohens_d']               = cohens_d_arr
    out['bb_chi2']                = bb_chi2
    out['bb_pvalue']              = bb_pval
    out['bb_mu_tumor']            = bb_mu_t
    out['bb_mu_normal']           = bb_mu_n
    out['bb_dispersion_phi']      = bb_phi
    out['mwu_u']                  = mwu_u
    out['mwu_pvalue']             = mwu_pval
    out['pvalue']                 = bb_pval
    out['q_value']                = qvals
    out['coverage_pass']          = coverage_pass
    out['passes_q']               = passes_q
    out['passes_dpsi']            = passes_dpsi
    out['has_scn_data']           = has_scn
    out['scn_concordant']         = scn_concordant
    out['scn_discordant']         = scn_discordant
    out['both_tests_significant'] = both_significant
    out['tier']                   = tier

    print(f"  Tier counts: primary={int((tier=='primary').sum()):,}, "
          f"secondary={int((tier=='secondary').sum()):,}, "
          f"scn_discordant={int((tier=='scn_discordant').sum()):,}, "
          f"rejected={int((tier=='rejected').sum()):,}")
    return out


# Reports
def report_comparison_old_new(per_cancer):
    lines = ["=" * 64, "OLD (POOLED-BINOMIAL) vs NEW (CORRECTED) COUNTS", "=" * 64,
             f"  {'cancer':<8} {'old':>10} {'new_primary':>12} {'new_secondary':>14} {'change':>10}"]
    for cancer, df in per_cancer.items():
        old = OLD_INFLATED_COUNTS[cancer]
        n_pri = int((df['tier'] == 'primary').sum())
        n_sec = int((df['tier'] == 'secondary').sum())
        ch = f"{100*(n_pri - old)/old:+.1f}%" if old else "n/a"
        lines.append(f"  {cancer:<8} {old:>10,} {n_pri:>12,} {n_sec:>14,} {ch:>10}")
    lines.append("")
    out = "\n".join(lines); print(out); return out


def report_m2_prime(per_cancer):
    lines = ["", "=" * 64, "M2' PRIMARY-SET COUNT GATE (locked thresholds)", "=" * 64]
    all_pass = True
    for cancer, df in per_cancer.items():
        threshold = M2_PRIME_THRESHOLDS[cancer]
        n_pri = int((df['tier'] == 'primary').sum())
        n_sec = int((df['tier'] == 'secondary').sum())
        status = "PASS" if n_pri >= threshold else "FAIL"
        if status == "FAIL":
            all_pass = False
        lines.append(f"  {cancer}: primary={n_pri:,}  (threshold>={threshold})  "
                     f"secondary={n_sec:,}  ->  {status}")
    lines.append("")
    lines.append(f"  Overall M2': {'PASS' if all_pass else 'FAIL'}")
    lines.append("")
    out = "\n".join(lines); print(out); return out, all_pass


def report_prediction_check(per_cancer):
    lines = ["=" * 64, "PRECOMMIT PREDICTION CHECK (primary counts vs locked ranges)", "=" * 64]
    for cancer, df in per_cancer.items():
        lo, hi = PREDICTION_RANGES[cancer]
        n_pri = int((df['tier'] == 'primary').sum())
        status = "WITHIN" if (lo <= n_pri <= hi) else ("BELOW" if n_pri < lo else "ABOVE")
        lines.append(f"  {cancer}: actual={n_pri:,}  predicted=[{lo}-{hi}]  ->  {status}")
    lines.append("")
    out = "\n".join(lines); print(out); return out


def report_test_concordance(per_cancer):
    lines = ["=" * 64, "TEST CONCORDANCE (betabin primary vs Wilcoxon sensitivity)", "=" * 64,
             f"  {'cancer':<8} {'both_sig':>10} {'BB_only':>10} {'MW_only':>10} {'either':>10}"]
    for cancer, df in per_cancer.items():
        bb = df['passes_q'].fillna(False).astype(bool)
        mw = (df['mwu_pvalue'] < Q_THRESHOLD) & df['mwu_pvalue'].notna()
        n_both    = int((bb & mw).sum())
        n_bb_only = int((bb & ~mw).sum())
        n_mw_only = int((~bb & mw).sum())
        n_either  = int((bb | mw).sum())
        lines.append(f"  {cancer:<8} {n_both:>10,} {n_bb_only:>10,} {n_mw_only:>10,} {n_either:>10,}")
    lines.append("")
    out = "\n".join(lines); print(out); return out


def report_hallmark_coverage(per_cancer):
    lines = ["=" * 64, "HALLMARK GENE COVERAGE DIAGNOSTIC", "=" * 64]
    for cancer, df in per_cancer.items():
        lines.append(f"\n  --- {cancer} ---")
        lines.append(f"  {'gene':<10} {'tier':<14} {'n_t':>5} {'n_n':>5} "
                     f"{'dPSI':>7} {'d':>6} {'phi':>7}")
        hits = df[df['gene_name'].isin(HALLMARK_GENES) &
                  (df['tier'].isin(['primary', 'secondary']))].copy()
        if hits.empty:
            lines.append("  (no listed hallmark genes have surviving events)")
            continue
        hits = hits.sort_values(['gene_name', 'abs_dpsi'], ascending=[True, False])
        seen = set()
        for _, row in hits.iterrows():
            g = row['gene_name']
            if g in seen:
                continue
            seen.add(g)
            phi_str = f"{row['bb_dispersion_phi']:.1f}" if not np.isnan(row['bb_dispersion_phi']) else "  nan"
            lines.append(f"  {g:<10} {row['tier']:<14} "
                         f"{int(row['n_tumor_well_covered']):>5} "
                         f"{int(row['n_normal_well_covered']):>5} "
                         f"{row['delta_psi']:>7.3f} "
                         f"{row['cohens_d']:>6.2f} "
                         f"{phi_str:>7}")
    lines.append("")
    out = "\n".join(lines); print(out); return out


# Main
def main():
    if not os.path.exists("/content/drive/MyDrive"):
        sys.exit("Drive not mounted. Run drive.mount('/content/drive') first.")
    if not os.path.exists(PROC_PSI):
        sys.exit(f"PROC_PSI not found: {PROC_PSI}")

    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"Output directory: {OUT_DIR}")
    print(f"Date: {date.today().isoformat()}")
    print(f"Methodology: betabin GLM primary, Wilcoxon sensitivity (Dev 1)")
    print()

    events_path = os.path.join(PROC_PSI, 'suppa_events_parsed.parquet')
    if not os.path.exists(events_path):
        sys.exit(f"Events parquet not found: {events_path}")
    events = pd.read_parquet(events_path).reset_index(drop=True)
    print(f"Loaded {len(events):,} SE events")

    scn_path = os.path.join(PROC_JX, 'TCGA_SCN_indices_all.pkl')
    if not os.path.exists(scn_path):
        sys.exit(f"SCN pickle not found: {scn_path}")
    with open(scn_path, 'rb') as f:
        scn_data = pickle.load(f)
    print(f"SCN indices for: {list(scn_data.keys())}")
    print()

    per_cancer_results = {}
    for cancer, normal in COHORT_PAIRS:
        if cancer not in scn_data:
            print(f"[skip {cancer}] no SCN indices in pickle")
            continue
        scn_indices = scn_data[cancer]['scn_indices']

        print(f"\n{'#' * 64}")
        print(f"#  Loading matrices for {cancer} / {normal}")
        print(f"{'#' * 64}")
        t_psi, t_cov, t_inc, _ = load_psi_and_counts(cancer)
        n_psi, n_cov, n_inc, _ = load_psi_and_counts(normal)

        res = run_corrected_pipeline(cancer, normal,
                                     t_psi, t_cov, t_inc,
                                     n_psi, n_cov, n_inc,
                                     scn_indices, events)
        per_cancer_results[cancer] = res

        full_path     = os.path.join(OUT_DIR, f"{cancer}_corrected_results.parquet")
        primary_path  = os.path.join(OUT_DIR, f"{cancer}_corrected_primary.parquet")
        secondary_path= os.path.join(OUT_DIR, f"{cancer}_corrected_secondary.parquet")
        res.to_parquet(full_path, index=False)
        res[res['tier'] == 'primary'  ].to_parquet(primary_path,   index=False)
        res[res['tier'] == 'secondary'].to_parquet(secondary_path, index=False)
        print(f"  Saved: {os.path.basename(full_path)}, "
              f"{os.path.basename(primary_path)}, {os.path.basename(secondary_path)}")

        del t_psi, t_cov, t_inc, n_psi, n_cov, n_inc
        import gc; gc.collect()

    print()
    comp = report_comparison_old_new(per_cancer_results)
    m2,_ = report_m2_prime(per_cancer_results)
    pred = report_prediction_check(per_cancer_results)
    conc = report_test_concordance(per_cancer_results)
    hall = report_hallmark_coverage(per_cancer_results)

    summary_path = os.path.join(OUT_DIR, "STAGE_C_CORRECTED_SUMMARY.txt")
    with open(summary_path, 'w') as f:
        f.write(f"Stage C corrected pipeline run: {date.today().isoformat()}\n")
        f.write(f"Precommit: STAGE_C_FIX_PRECOMMIT.md (2026-06-11)\n")
        f.write(f"Deviation: STAGE_C_DEVIATIONS.md Dev 1 (betabin GLM primary)\n\n")
        f.write(comp + "\n")
        f.write(m2 + "\n")
        f.write(pred + "\n")
        f.write(conc + "\n")
        f.write(hall + "\n")
    print(f"\nSummary saved to: {summary_path}")


if __name__ == "__main__":
    main()


---
## Section 5 -- M3' cross-cancer overlap and three-way conserved core

Computes pairwise overlap between primary aberrant sets, identifies the three-way conserved core (events present as primary in all three cancers), and confirms robustness via 10% subsample dropout bootstrap. Annotates conserved-core events with HGNC gene symbols and computes literature cross-reference against the 71-gene curated cancer-splicing reference (see Deviation 4 for reference set composition).

**Locked outputs**: 123 conserved events across 115 host genes, 99.2% directionally consistent, 100% bootstrap sign-stable.

In [ ]:
"""
step3a_m3prime_overlap.py

Step 3a of the Paper 5A Stage C fix sequence.

Computes M3' cross-cancer overlap on the corrected primary aberrant sets
(LUAD: 464; BLCA: 411; UCEC: 938). Reports two levels:

  - Event-level overlap: same exon (same SUPPA2 event_id) shared across cancers.
  - Gene-level overlap: same gene (same Ensembl gene_id) shared across cancers,
    possibly via different exons.

The original M3' pre-commitment (from FRAMING_B_CAVEATS.md) tested event-level
pairwise overlap at 5-30%. Observed 40-60% on the inflated pooled-binomial set.

For the corrected set, we report:
  - Pairwise event-level overlap (% of each cancer's primary set shared with each other)
  - Pairwise gene-level overlap (same, but at gene level)
  - 3-way intersection: events / genes in ALL three cancers (the conserved core)
  - Hallmark gene presence in the conserved core
  - Direction consistency check: among shared events, do tumor-vs-normal directions agree?

USAGE IN COLAB:
  Paste this file's contents into one cell and run.
  Or:  !python step3a_m3prime_overlap.py

Outputs to: <Stage2C_corrected>/M3prime_overlap/
  pairwise_event_overlap.csv
  pairwise_gene_overlap.csv
  three_way_shared_events.parquet
  three_way_shared_genes.parquet
  conserved_core_with_dpsi_directions.parquet
  M3PRIME_OVERLAP_REPORT.txt

Runtime: under a minute.
"""

import os
import sys
from datetime import date
from itertools import combinations

import numpy as np
import pandas as pd


# Config
PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR  = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
OUT_DIR      = os.path.join(STAGE_C_DIR, "M3prime_overlap")

CANCERS = ["LUAD", "BLCA", "UCEC"]

# Original M3' precommit range (from FRAMING_B_CAVEATS.md)
M3_PRECOMMIT_LO = 0.05
M3_PRECOMMIT_HI = 0.30

# Hallmark Ensembl IDs (same hardcoded mapping as the patch script)
HALLMARK_ENSEMBL = {
    "FAS":     "ENSG00000026103",
    "CD44":    "ENSG00000026508",
    "FGFR1":   "ENSG00000077782",
    "FGFR2":   "ENSG00000066468",
    "FGFR3":   "ENSG00000068078",
    "BCL2L1":  "ENSG00000171552",
    "MDM4":    "ENSG00000198625",
    "MDM2":    "ENSG00000135679",
    "VEGFA":   "ENSG00000112715",
    "KLF6":    "ENSG00000067082",
    "PKM":     "ENSG00000067225",
    "MET":     "ENSG00000105976",
    "EGFR":    "ENSG00000146648",
    "AR":      "ENSG00000169083",
    "TP53":    "ENSG00000141510",
    "RAC1":    "ENSG00000136238",
    "RON":     "ENSG00000105426",
    "SYK":     "ENSG00000165025",
    "TIA1":    "ENSG00000116001",
    "NUMB":    "ENSG00000133961",
    "CTNNB1":  "ENSG00000168036",
    "BIN1":    "ENSG00000136717",
    "CASP9":   "ENSG00000132906",
    "CASP8":   "ENSG00000064012",
    "TRAF3":   "ENSG00000131323",
    "ESR1":    "ENSG00000091831",
    "BRCA1":   "ENSG00000012048",
    "PIK3CA":  "ENSG00000121879",
}
ENSEMBL_TO_SYMBOL = {v: k for k, v in HALLMARK_ENSEMBL.items()}


def gene_id_stem(gid):
    if pd.isna(gid):
        return None
    return str(gid).split('.')[0]


# Load primary aberrant sets
def load_primary_set(cancer):
    path = os.path.join(STAGE_C_DIR, f"{cancer}_corrected_primary.parquet")
    if not os.path.exists(path):
        sys.exit(f"Missing primary parquet: {path}")
    df = pd.read_parquet(path)
    df['gene_id_stem'] = df['gene_id'].apply(gene_id_stem)
    return df


# Overlap calculations
def pairwise_overlap(primary_sets, key):
    """Pairwise overlap of `key` (either 'event_id' or 'gene_id_stem') across cancers.

    Returns a long-form dataframe with one row per ordered pair, columns:
      cancer_A, cancer_B, n_A, n_B, n_shared, pct_of_A, pct_of_B, jaccard
    """
    rows = []
    for a, b in combinations(CANCERS, 2):
        set_a = set(primary_sets[a][key].dropna())
        set_b = set(primary_sets[b][key].dropna())
        shared = set_a & set_b
        union  = set_a | set_b
        rows.append({
            'cancer_A':   a,
            'cancer_B':   b,
            'n_A':        len(set_a),
            'n_B':        len(set_b),
            'n_shared':   len(shared),
            'pct_of_A':   100 * len(shared) / max(1, len(set_a)),
            'pct_of_B':   100 * len(shared) / max(1, len(set_b)),
            'jaccard':    len(shared) / max(1, len(union)),
        })
    return pd.DataFrame(rows)


def three_way_intersection(primary_sets, key):
    """Items present in all three cancers' primary sets."""
    sets = [set(primary_sets[c][key].dropna()) for c in CANCERS]
    return sets[0] & sets[1] & sets[2]


# Direction consistency check
def direction_consistency(primary_sets, shared_event_ids):
    """For events shared across all 3 cancers, check whether the dPSI direction
    agrees across cancers."""
    rows = []
    for eid in shared_event_ids:
        row = {'event_id': eid}
        signs = []
        for c in CANCERS:
            evt = primary_sets[c].loc[primary_sets[c]['event_id'] == eid]
            if len(evt) == 0:
                row[f'{c}_dpsi'] = np.nan
                continue
            dpsi = evt['delta_psi'].iloc[0]
            row[f'{c}_dpsi'] = dpsi
            row[f'{c}_n_tumor_wc'] = evt['n_tumor_well_covered'].iloc[0]
            row[f'{c}_gene_id'] = evt['gene_id'].iloc[0]
            signs.append(np.sign(dpsi))
        row['n_cancers'] = len([s for s in signs if not np.isnan(s)])
        row['all_signs_agree'] = (len(set(s for s in signs if not np.isnan(s))) <= 1)
        rows.append(row)
    return pd.DataFrame(rows)


def gene_direction_consistency(primary_sets, shared_gene_stems):
    """For genes shared across all 3 cancers, summarize per-cancer events and
    consensus direction (majority sign across events per gene)."""
    rows = []
    for gstem in shared_gene_stems:
        row = {'gene_id_stem': gstem, 'gene_symbol': ENSEMBL_TO_SYMBOL.get(gstem, '')}
        majority_signs = []
        for c in CANCERS:
            sub = primary_sets[c][primary_sets[c]['gene_id_stem'] == gstem]
            row[f'{c}_n_events'] = len(sub)
            if len(sub) == 0:
                row[f'{c}_consensus_dpsi'] = np.nan
                continue
            # Consensus direction = sign of mean dPSI weighted by abs(dPSI)
            row[f'{c}_consensus_dpsi'] = sub['delta_psi'].mean()
            majority_signs.append(np.sign(sub['delta_psi'].mean()))
        row['n_cancers_present'] = sum(1 for c in CANCERS if row[f'{c}_n_events'] > 0)
        row['all_dirs_agree'] = (len(set(s for s in majority_signs if not np.isnan(s))) <= 1)
        rows.append(row)
    df = pd.DataFrame(rows)
    # Sort: hallmarks first, then by number of cancers, then by symbol
    df['is_hallmark'] = df['gene_symbol'] != ''
    df = df.sort_values(['is_hallmark', 'n_cancers_present'],
                        ascending=[False, False]).reset_index(drop=True)
    return df


# Main
def main():
    if not os.path.exists(STAGE_C_DIR):
        sys.exit(f"Stage2C_corrected directory not found: {STAGE_C_DIR}")
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"Output: {OUT_DIR}")
    print(f"Date:   {date.today().isoformat()}")
    print()

    # Load primary sets
    primary_sets = {c: load_primary_set(c) for c in CANCERS}
    print("Primary aberrant set sizes:")
    for c in CANCERS:
        n_evt   = len(primary_sets[c])
        n_genes = primary_sets[c]['gene_id_stem'].nunique()
        print(f"  {c}: {n_evt:,} events  /  {n_genes:,} unique genes")
    print()

    # -------- Pairwise overlap (event-level) --------
    print("PAIRWISE OVERLAP -- EVENT LEVEL (same exon shared)")
    pair_evt = pairwise_overlap(primary_sets, 'event_id')
    print(pair_evt.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    pair_evt.to_csv(os.path.join(OUT_DIR, 'pairwise_event_overlap.csv'), index=False)

    # M3' precommit check (event level)
    print()
    print(f"M3' precommit range: {100*M3_PRECOMMIT_LO:.0f}-{100*M3_PRECOMMIT_HI:.0f}% pairwise event overlap")
    in_range_count = 0
    for _, row in pair_evt.iterrows():
        pct_min = min(row['pct_of_A'], row['pct_of_B'])
        status = "WITHIN" if (M3_PRECOMMIT_LO*100 <= pct_min <= M3_PRECOMMIT_HI*100) \
                 else ("BELOW" if pct_min < M3_PRECOMMIT_LO*100 else "ABOVE")
        if status == "WITHIN":
            in_range_count += 1
        print(f"  {row['cancer_A']} <-> {row['cancer_B']}: "
              f"{row['pct_of_A']:.1f}% of A / {row['pct_of_B']:.1f}% of B "
              f"({row['n_shared']:,} shared)  -> {status}")
    print()

    # -------- Pairwise overlap (gene-level) --------
    print("PAIRWISE OVERLAP -- GENE LEVEL (same gene, any exon)")
    pair_gene = pairwise_overlap(primary_sets, 'gene_id_stem')
    print(pair_gene.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    pair_gene.to_csv(os.path.join(OUT_DIR, 'pairwise_gene_overlap.csv'), index=False)
    print()

    # -------- Three-way intersection (event level) --------
    print("THREE-WAY INTERSECTION -- EVENT LEVEL (same exon in all 3)")
    shared_events = three_way_intersection(primary_sets, 'event_id')
    print(f"Events present in primary set of all 3 cancers: {len(shared_events):,}")
    if shared_events:
        direction_df = direction_consistency(primary_sets, shared_events)
        all_agree = int(direction_df['all_signs_agree'].sum())
        print(f"  Of which directionally consistent (same dPSI sign): {all_agree:,}")
        direction_df.to_parquet(os.path.join(OUT_DIR, 'three_way_shared_events.parquet'), index=False)
        # Show top 10 by mean |dPSI|
        if len(direction_df) > 0:
            direction_df['mean_abs_dpsi'] = direction_df[[f'{c}_dpsi' for c in CANCERS]].abs().mean(axis=1)
            print("\n  Top 10 conserved events by mean |dPSI|:")
            top = direction_df.sort_values('mean_abs_dpsi', ascending=False).head(10)
            for _, r in top.iterrows():
                hallmark = ENSEMBL_TO_SYMBOL.get(gene_id_stem(r.get(f'{CANCERS[0]}_gene_id', '')), '')
                marker = f"[{hallmark}]" if hallmark else ""
                print(f"    {r['event_id'][:60]:<60} {marker:<10} "
                      f"LUAD={r.get('LUAD_dpsi', np.nan):+.2f} "
                      f"BLCA={r.get('BLCA_dpsi', np.nan):+.2f} "
                      f"UCEC={r.get('UCEC_dpsi', np.nan):+.2f}  "
                      f"agree={r['all_signs_agree']}")
    print()

    # -------- Three-way intersection (gene level) --------
    print("THREE-WAY INTERSECTION -- GENE LEVEL (same gene aberrant in all 3)")
    shared_genes = three_way_intersection(primary_sets, 'gene_id_stem')
    print(f"Genes with primary events in all 3 cancers: {len(shared_genes):,}")
    gene_df = gene_direction_consistency(primary_sets, shared_genes)
    gene_df.to_parquet(os.path.join(OUT_DIR, 'three_way_shared_genes.parquet'), index=False)

    # Hallmark presence in the conserved core
    hallmark_in_core = gene_df[gene_df['is_hallmark']].copy()
    print(f"  Of which hallmark genes: {len(hallmark_in_core)}")
    if len(hallmark_in_core) > 0:
        print()
        print("  Hallmark genes in 3-cancer conserved core:")
        print(f"    {'symbol':<10} {'LUAD_dpsi':>10} {'BLCA_dpsi':>10} {'UCEC_dpsi':>10}  agree")
        for _, r in hallmark_in_core.iterrows():
            agree = r['all_dirs_agree']
            print(f"    {r['gene_symbol']:<10} "
                  f"{r['LUAD_consensus_dpsi']:>+10.3f} "
                  f"{r['BLCA_consensus_dpsi']:>+10.3f} "
                  f"{r['UCEC_consensus_dpsi']:>+10.3f}  "
                  f"{agree}")
    print()

    # -------- Conserved core summary table --------
    conserved_path = os.path.join(OUT_DIR, 'conserved_core_with_dpsi_directions.parquet')
    gene_df.to_parquet(conserved_path, index=False)
    print(f"Conserved core gene table saved: {os.path.basename(conserved_path)}")
    print()

    # -------- Write text report --------
    report_path = os.path.join(OUT_DIR, 'M3PRIME_OVERLAP_REPORT.txt')
    with open(report_path, 'w') as f:
        f.write(f"M3' overlap analysis on corrected primary aberrant sets\n")
        f.write(f"Date: {date.today().isoformat()}\n")
        f.write(f"Source: Stage2C_corrected/{{LUAD,BLCA,UCEC}}_corrected_primary.parquet\n")
        f.write(f"Path B framing: tiered reporting, primary tier only used here\n\n")

        f.write("PRIMARY SET SIZES\n")
        f.write("=" * 64 + "\n")
        for c in CANCERS:
            n_evt   = len(primary_sets[c])
            n_genes = primary_sets[c]['gene_id_stem'].nunique()
            f.write(f"  {c}: {n_evt:,} events  /  {n_genes:,} unique genes\n")
        f.write("\n")

        f.write("PAIRWISE OVERLAP -- EVENT LEVEL\n")
        f.write("=" * 64 + "\n")
        f.write(pair_evt.to_string(index=False, float_format=lambda x: f"{x:.2f}") + "\n\n")

        f.write(f"M3' precommit pairwise event-overlap range: "
                f"{100*M3_PRECOMMIT_LO:.0f}-{100*M3_PRECOMMIT_HI:.0f}%\n")
        f.write(f"In-range pairs: {in_range_count} / {len(pair_evt)}\n\n")

        f.write("PAIRWISE OVERLAP -- GENE LEVEL\n")
        f.write("=" * 64 + "\n")
        f.write(pair_gene.to_string(index=False, float_format=lambda x: f"{x:.2f}") + "\n\n")

        f.write("THREE-WAY INTERSECTION (event level)\n")
        f.write("=" * 64 + "\n")
        f.write(f"Events present in all 3 cancers' primary set: {len(shared_events):,}\n")
        if shared_events:
            f.write(f"  Directionally consistent: {all_agree:,}\n")
        f.write("\n")

        f.write("THREE-WAY INTERSECTION (gene level)\n")
        f.write("=" * 64 + "\n")
        f.write(f"Genes with primary events in all 3 cancers: {len(shared_genes):,}\n")
        f.write(f"  Hallmark genes in conserved core: {len(hallmark_in_core)}\n")
        if len(hallmark_in_core) > 0:
            f.write("\n  Hallmark conserved core:\n")
            for _, r in hallmark_in_core.iterrows():
                f.write(f"    {r['gene_symbol']:<10} "
                        f"LUAD={r['LUAD_consensus_dpsi']:+.3f}  "
                        f"BLCA={r['BLCA_consensus_dpsi']:+.3f}  "
                        f"UCEC={r['UCEC_consensus_dpsi']:+.3f}  "
                        f"agree={r['all_dirs_agree']}\n")
    print(f"Text report saved: {report_path}")


if __name__ == "__main__":
    main()


In [ ]:
"""
step3a_annotate_conserved_events.py

After step3a_m3prime_overlap.py produces the three-way conserved core, this
script annotates the top conserved events with their HGNC gene symbols.

Strategy: load the GENCODE v26 GTF that recount3 uses (we can rebuild a
gene_id -> symbol map from the existing junction coords files, since those
were originally derived from the same GTF). If that doesn't work, fall back
to MyGene.info API lookup (online, free, no auth needed).

USAGE IN COLAB:
  Paste this file's contents into a Colab cell and run.
  Internet required if GENCODE annotation isn't already cached.

Outputs:
  <Stage2C_corrected>/M3prime_overlap/top50_conserved_events_annotated.csv
  <Stage2C_corrected>/M3prime_overlap/conserved_core_genes_annotated.csv

Runtime: 1-3 minutes (mostly online lookup).
"""

import os
import sys
import time
from datetime import date

import numpy as np
import pandas as pd


PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR  = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
OVERLAP_DIR  = os.path.join(STAGE_C_DIR, "M3prime_overlap")


def gene_id_stem(gid):
    if pd.isna(gid):
        return None
    return str(gid).split('.')[0]


def parse_event_id_for_gene_ids(event_id):
    """SUPPA2 event_id is 'ENSG...;SE:chr...'. Some have 'ENSG..._and_ENSG..;SE:...'
    Return the list of gene id stems referenced."""
    if pd.isna(event_id):
        return []
    head = str(event_id).split(';')[0]
    parts = head.split('_and_')
    return [p.split('.')[0] for p in parts if p.startswith('ENSG')]


def mygene_lookup(ensembl_ids, batch_size=100):
    """Bulk lookup Ensembl gene IDs -> HGNC symbols via MyGene.info.
    Returns dict {ensembl_stem: symbol or ''}.
    No API key needed; rate limit ~1000 IDs/sec.
    """
    import urllib.request
    import urllib.parse
    import json

    out = {}
    ids = list(set(ensembl_ids))
    for i in range(0, len(ids), batch_size):
        batch = ids[i:i+batch_size]
        data = urllib.parse.urlencode({
            'q': ','.join(batch),
            'scopes': 'ensembl.gene',
            'fields': 'symbol,name',
            'species': 'human',
        }).encode('utf-8')
        try:
            req = urllib.request.Request(
                'https://mygene.info/v3/query',
                data=data,
                headers={'Content-Type': 'application/x-www-form-urlencoded'}
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                result = json.loads(resp.read().decode('utf-8'))
            for hit in result:
                q = hit.get('query', '')
                sym = hit.get('symbol', '')
                name = hit.get('name', '')
                if sym and q not in out:
                    out[q] = (sym, name)
            # Fill blanks for queries that returned no hit
            for q in batch:
                if q not in out:
                    out[q] = ('', '')
            print(f"  batch {i//batch_size + 1}: {len(batch)} ids -> "
                  f"{sum(1 for q in batch if out.get(q,('',''))[0])} symbols")
            time.sleep(0.5)
        except Exception as e:
            print(f"  batch {i//batch_size + 1} FAILED: {e}")
            for q in batch:
                if q not in out:
                    out[q] = ('', '')
    return out


def main():
    # Inputs
    events_path = os.path.join(OVERLAP_DIR, 'three_way_shared_events.parquet')
    genes_path  = os.path.join(OVERLAP_DIR, 'three_way_shared_genes.parquet')
    if not os.path.exists(events_path) or not os.path.exists(genes_path):
        sys.exit(f"Run step3a_m3prime_overlap.py first to produce:\n  {events_path}\n  {genes_path}")

    events_df = pd.read_parquet(events_path)
    genes_df  = pd.read_parquet(genes_path)
    print(f"Loaded {len(events_df):,} three-way shared events, {len(genes_df):,} shared genes")

    # Collect all unique Ensembl IDs we need to annotate
    ids_from_events = set()
    for eid in events_df['event_id'].dropna():
        ids_from_events.update(parse_event_id_for_gene_ids(eid))
    ids_from_genes = set(genes_df['gene_id_stem'].dropna())
    all_ids = list(ids_from_events | ids_from_genes)
    print(f"Total unique Ensembl IDs to annotate: {len(all_ids):,}")
    print()

    # Bulk lookup via MyGene.info
    print("Looking up gene symbols via MyGene.info ...")
    sym_map = mygene_lookup(all_ids)
    n_resolved = sum(1 for v in sym_map.values() if v[0])
    print(f"  Resolved: {n_resolved} / {len(all_ids)}")
    print()

    # ---- Annotate the per-event table ----
    def annotate_event_symbols(eid):
        gids = parse_event_id_for_gene_ids(eid)
        syms = [sym_map.get(g, ('', ''))[0] for g in gids]
        return ' / '.join(s for s in syms if s)

    def annotate_event_names(eid):
        gids = parse_event_id_for_gene_ids(eid)
        names = [sym_map.get(g, ('', ''))[1] for g in gids]
        return ' / '.join(n for n in names if n)

    events_df['gene_symbols'] = events_df['event_id'].apply(annotate_event_symbols)
    events_df['gene_names']   = events_df['event_id'].apply(annotate_event_names)
    events_df['mean_abs_dpsi'] = events_df[['LUAD_dpsi', 'BLCA_dpsi', 'UCEC_dpsi']].abs().mean(axis=1)

    # Top 50 by mean |dPSI|, with annotations
    top50 = events_df.sort_values('mean_abs_dpsi', ascending=False).head(50).copy()
    top50_path = os.path.join(OVERLAP_DIR, 'top50_conserved_events_annotated.csv')
    top50[['event_id', 'gene_symbols', 'gene_names',
           'LUAD_dpsi', 'BLCA_dpsi', 'UCEC_dpsi',
           'LUAD_n_tumor_wc', 'BLCA_n_tumor_wc', 'UCEC_n_tumor_wc',
           'mean_abs_dpsi', 'all_signs_agree']].to_csv(top50_path, index=False)

    print("TOP 25 CONSERVED EVENTS -- annotated")
    print(f"  {'rank':>4} {'symbol':<18} {'LUAD':>7} {'BLCA':>7} {'UCEC':>7}  description")
    for i, (_, r) in enumerate(top50.head(25).iterrows(), 1):
        sym = r['gene_symbols'][:18] if r['gene_symbols'] else '(no symbol)'
        name = r['gene_names'][:55] if r['gene_names'] else ''
        print(f"  {i:>4} {sym:<18} "
              f"{r['LUAD_dpsi']:>+7.3f} {r['BLCA_dpsi']:>+7.3f} {r['UCEC_dpsi']:>+7.3f}  "
              f"{name}")
    print()
    print(f"Saved: {top50_path}")
    print()

    # ---- Annotate the per-gene table ----
    genes_df['symbol_from_lookup'] = genes_df['gene_id_stem'].apply(
        lambda g: sym_map.get(g, ('', ''))[0]
    )
    genes_df['gene_name_full'] = genes_df['gene_id_stem'].apply(
        lambda g: sym_map.get(g, ('', ''))[1]
    )
    # Prefer the hardcoded hallmark symbol if present, otherwise the lookup
    genes_df['symbol_final'] = np.where(
        genes_df['gene_symbol'].fillna('').str.len() > 0,
        genes_df['gene_symbol'],
        genes_df['symbol_from_lookup']
    )
    # Mean |consensus_dpsi| for sorting
    genes_df['mean_abs_consensus_dpsi'] = genes_df[
        [f'{c}_consensus_dpsi' for c in ['LUAD', 'BLCA', 'UCEC']]
    ].abs().mean(axis=1)
    genes_df = genes_df.sort_values(
        ['is_hallmark', 'all_dirs_agree', 'mean_abs_consensus_dpsi'],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    genes_out_path = os.path.join(OVERLAP_DIR, 'conserved_core_genes_annotated.csv')
    genes_df[['gene_id_stem', 'symbol_final', 'gene_name_full', 'is_hallmark',
              'LUAD_n_events', 'BLCA_n_events', 'UCEC_n_events',
              'LUAD_consensus_dpsi', 'BLCA_consensus_dpsi', 'UCEC_consensus_dpsi',
              'mean_abs_consensus_dpsi', 'all_dirs_agree']].to_csv(genes_out_path, index=False)

    print("TOP 25 CONSERVED GENES -- by mean |consensus dPSI|")
    print(f"  {'rank':>4} {'symbol':<14} {'hallmark':<10} "
          f"{'LUAD':>7} {'BLCA':>7} {'UCEC':>7}  agree  full_name")
    for i, (_, r) in enumerate(genes_df.head(25).iterrows(), 1):
        sym = r['symbol_final'] if r['symbol_final'] else '(unknown)'
        hm = 'YES' if r['is_hallmark'] else ''
        name = (r['gene_name_full'] or '')[:50]
        print(f"  {i:>4} {sym:<14} {hm:<10} "
              f"{r['LUAD_consensus_dpsi']:>+7.3f} "
              f"{r['BLCA_consensus_dpsi']:>+7.3f} "
              f"{r['UCEC_consensus_dpsi']:>+7.3f}  "
              f"{str(r['all_dirs_agree']):<6} {name}")
    print()
    print(f"Saved: {genes_out_path}")


if __name__ == "__main__":
    main()


In [ ]:
"""
step3a_consolidate_findings.py

After step3a_m3prime_overlap.py and step3a_annotate_conserved_events.py,
this script does three things to consolidate the conserved-core finding
before we move to Step 3b (M4 enrichment):

  CHECK 1: Robustness of the 123-event conserved core.
    For each conserved event, drop a random 10% of tumor samples (per cancer),
    recompute mean-based dPSI on the remaining samples, repeat 200 times.
    Test: in what fraction of subsamples does the dPSI sign agree with the
    full-data sign? If >=95% sign-stable, the event is robust to outliers.
    Reports: per-event sign-stability rate, fraction of 123 events that are
    sign-stable in ALL three cancers.

  CHECK 2: Cross-reference against a curated literature reference set of
    well-known cancer-splicing genes (~70 genes from major reviews).
    Reports: number/fraction of our 115 conserved genes that are in the
    literature set ("rediscovery rate"), and the highest-effect genes
    NOT in the literature set ("candidate novel" list).

  CHECK 3: Save the findings record (Deviation 4) to STAGE_C_DEVIATIONS.md.
    Pure findings record per Norm 11: locks the exact numbers that will
    appear in the manuscript headline. No parameter changes.

USAGE IN COLAB:
  Paste this file's contents into one cell and run.
"""

import os
import sys
from datetime import date

import numpy as np
import pandas as pd


# Config
PROJECT_ROOT  = "/content/drive/MyDrive/Paper5_SharedAntigens"
PROC_PSI      = os.path.join(PROJECT_ROOT, "data/processed/psi")
PROC_JX       = os.path.join(PROJECT_ROOT, "data/processed/junctions")
STAGE_C_DIR   = os.path.join(PROC_PSI, "Stage2C_corrected")
OVERLAP_DIR   = os.path.join(STAGE_C_DIR, "M3prime_overlap")
PRECOMMIT_DIR = os.path.join(PROJECT_ROOT, "pre_commitments")

CANCERS = ["LUAD", "BLCA", "UCEC"]
COHORT_PAIRS = [
    ("LUAD", "LUNG"), ("BLCA", "BLADDER"), ("UCEC", "UTERUS"),
]

# Robustness test config
BOOTSTRAP_REPS    = 200
DROPOUT_FRACTION  = 0.10
SIGN_STABLE_THRESH = 0.95   # event is sign-stable if direction agrees in >=95% of resamples


# Curated literature reference: well-known cancer-splicing genes
# Sources (cited in Methods):
#   - David & Manley 2010 (Genes Dev): EMT/invasion splicing program
#   - Climente-Gonzalez et al. 2017 (Cell Reports): cancer-splicing landscape
#   - Cherry & Lynch 2020 (Genes Dev): SR/hnRNP targets review
#   - Bonnal et al. 2020 (Nat Rev Clin Oncol): therapeutic targeting review
#   - Seiler et al. 2018 (Cell Reports): splicing factor mutational landscape
#   - Sebestyén et al. 2016 (Genome Res): pan-cancer splicing analysis
#   - Warzecha & Carstens 2012, Yang et al. 2016: ESRP1/2 targets
#   - Kahles et al. 2018 (Cancer Cell): TCGA 8,705-patient splicing landscape
CANCER_SPLICING_GENES_LIT = {
    # ----- EMT / invasion / ESRP1/2 targets -----
    "CD44":     "EMT/invasion; CD44v variants - canonical cancer splicing",
    "FGFR2":    "EMT IIIb/IIIc isoform switch (ESRP target)",
    "ENAH":     "Mena 11a/INV isoform - metastasis (ESRP target)",
    "CTNND1":   "p120-catenin isoform switch (ESRP target)",
    "NUMB":     "Notch signaling, exon 9 inclusion (canonical cancer splicing)",
    "FAT1":     "Tumor suppressor, recurrently altered, ESRP target",
    "ITGA6":    "Integrin alpha 6 isoform switch (ESRP target)",
    "ITGB1":    "Integrin beta 1 isoforms",
    "ITGB4":    "Integrin beta 4, epithelial invasion",
    "MYO18A":   "Myosin XVIIIA isoform switch (ESRP target)",
    "SCRIB":    "Scribble polarity, ESRP target",
    "FN1":      "Fibronectin EDA/EDB exons - canonical cancer splicing",
    "GRHL3":    "Grainyhead-like 3, ESRP target",
    "CLSTN1":   "Calsyntenin 1 - TCGA SpliceSeq top cross-tumor hit (Ryan 2016)",
    "CD46":     "Membrane cofactor protein isoform switch",
    "TCF7L2":   "Wnt signaling isoform switch",
    "MAP3K7":   "TAK1, splicing-regulated kinase in cancer",
    # ----- Apoptosis program -----
    "FAS":      "Soluble vs membrane FAS isoform switch",
    "BCL2L1":   "Bcl-xL/Bcl-xS apoptosis switch - canonical cancer splicing",
    "BCL2L11":  "BIM (BCL2L11) isoforms",
    "CASP2":    "Pro/anti-apoptotic Casp2L/2S",
    "CASP8":    "Caspase 8 isoforms",
    "CASP9":    "Caspase 9a/9b switch",
    "BIN1":     "BIN1 +12A isoform - tumor suppressor",
    "MDM4":     "MDM4-S vs FL splicing (p53 regulation)",
    "MDM2":     "MDM2 isoforms",
    "TP53":     "p53 family isoforms",
    "TP63":     "TAp63/dNp63 isoforms",
    "TP73":     "TAp73/dNp73 isoforms",
    # ----- Cytoskeleton / mechanics -----
    "TPM1":     "Tropomyosin 1 isoform switch - canonical cancer splicing",
    "TPM2":     "Tropomyosin 2 isoforms",
    "ACTN1":    "Alpha-actinin 1 muscle/non-muscle (ESRP target)",
    "ACTN4":    "Alpha-actinin 4 isoforms",
    "ADD3":     "Adducin 3 cytoskeletal isoforms",
    # ----- Receptor tyrosine kinases -----
    "FGFR1":    "FGFR1 alpha/beta switch",
    "FGFR3":    "FGFR3 IIIb/IIIc switch",
    "EGFR":     "EGFR vIII and other isoforms",
    "MET":      "MET exon 14 skipping (oncogenic)",
    "MST1R":    "RON kinase - Delta-RON isoform switch (SRSF1 target)",
    "AR":       "Androgen receptor AR-V7 splice variant",
    # ----- Metabolism -----
    "PKM":      "PKM2 Warburg switch - canonical cancer splicing",
    # ----- Angiogenesis -----
    "VEGFA":    "VEGFA pro/anti-angiogenic isoform switch",
    # ----- Signaling / oncogenes -----
    "RAC1":     "Rac1b isoform - canonical cancer splicing",
    "SYK":      "Syk(L)/Syk(S) isoform switch",
    "CTNNB1":   "Beta-catenin",
    "PIK3CA":   "PI3K alpha isoforms",
    "BRCA1":    "BRCA1 isoforms",
    "TRAF3":    "TRAF3 isoforms (NF-kB)",
    "KLF6":     "KLF6-SV1 oncogenic isoform - canonical cancer splicing",
    "ESR1":     "Estrogen receptor isoforms",
    # ----- RNA-binding / splicing factors -----
    "PTBP1":    "PTBP1 (PTB) - splicing factor",
    "PTBP2":    "PTBP2 - splicing factor",
    "RBFOX2":   "RBFOX2 - splicing factor, EMT regulator",
    "MBNL1":    "MBNL1 - splicing factor",
    "MBNL2":    "MBNL2 - splicing factor",
    "MBNL3":    "MBNL3 - splicing factor",
    "CELF1":    "CELF1 - splicing factor",
    "ESRP1":    "ESRP1 - epithelial splicing regulator",
    "ESRP2":    "ESRP2 - epithelial splicing regulator",
    "QKI":      "Quaking - splicing factor, tumor suppressor",
    # ----- Spliceosome / SF mutations -----
    "SF3B1":    "SF3B1 - spliceosome, recurrently mutated",
    "SRSF1":    "SRSF1 - SR splicing factor, oncogenic",
    "SRSF2":    "SRSF2 - SR splicing factor, recurrently mutated",
    "U2AF1":    "U2AF1 - splicing factor, recurrently mutated",
    "RBM10":    "RBM10 - splicing factor, LUAD tumor suppressor",
    "FUBP1":    "FUBP1 - splicing factor, recurrently mutated",
    "HNRNPA1":  "hnRNPA1 - splicing factor",
    "HNRNPA2B1":"hnRNPA2B1 - splicing factor",
    # ----- SWI/SNF chromatin (splicing-coupled) -----
    "PBRM1":    "PBRM1 - SWI/SNF, ccRCC tumor suppressor",
    "SMARCC2":  "SMARCC2 - SWI/SNF chromatin remodeler",
    "ARID1A":   "ARID1A - SWI/SNF, recurrent cancer mutations",
}


# CHECK 1: Robustness via leave-out subsampling
def load_psi_matrix(label):
    npz_path = os.path.join(PROC_PSI, f"{label}_psi.npz")
    z = np.load(npz_path)
    return z['psi']   # float32, NaN where coverage<10


def load_scn_indices():
    import pickle
    p = os.path.join(PROC_JX, 'TCGA_SCN_indices_all.pkl')
    with open(p, 'rb') as f:
        return pickle.load(f)


def event_id_to_row_lookup(events_df):
    return {eid: i for i, eid in enumerate(events_df['event_id'])}


def robustness_check(conserved_event_ids, events_full_df, psi_by_label, scn_indices):
    """For each conserved event in each cancer, compute the sign-stability of
    dPSI under random 10% sample dropout, BOOTSTRAP_REPS replicates."""
    eid_to_row = event_id_to_row_lookup(events_full_df)
    n_conserved = len(conserved_event_ids)
    rng = np.random.default_rng(seed=42)

    # Per-cancer: (n_conserved,) sign stability for each event
    per_cancer_stability = {c: np.full(n_conserved, np.nan) for c in CANCERS}
    full_data_signs      = {c: np.full(n_conserved, np.nan) for c in CANCERS}

    for cancer, normal in COHORT_PAIRS:
        tumor_psi = psi_by_label[cancer]
        normal_psi = psi_by_label[normal]

        # Build tumor mask (exclude SCN samples)
        n_total = tumor_psi.shape[1]
        tumor_mask = np.ones(n_total, dtype=bool)
        tumor_mask[scn_indices[cancer]['scn_indices']] = False
        tumor_only = tumor_psi[:, tumor_mask]
        n_tumor = tumor_only.shape[1]
        n_normal_cols = normal_psi.shape[1]

        print(f"  [{cancer}] Robustness check on {n_conserved} events; "
              f"{n_tumor} tumor samples × {BOOTSTRAP_REPS} subsamples × {int(DROPOUT_FRACTION*100)}% dropout")

        for j, eid in enumerate(conserved_event_ids):
            if eid not in eid_to_row:
                continue
            i = eid_to_row[eid]

            # Full-data dPSI sign
            t_psi_full = tumor_only[i]
            n_psi_full = normal_psi[i]
            t_mean = np.nanmean(t_psi_full)
            n_mean = np.nanmean(n_psi_full)
            full_dpsi = t_mean - n_mean
            if np.isnan(full_dpsi) or full_dpsi == 0:
                continue
            full_sign = np.sign(full_dpsi)
            full_data_signs[cancer][j] = full_sign

            # Bootstrap signs
            n_keep = int(n_tumor * (1 - DROPOUT_FRACTION))
            agree_count = 0
            valid_count = 0
            for _ in range(BOOTSTRAP_REPS):
                keep_idx = rng.choice(n_tumor, size=n_keep, replace=False)
                t_sub = t_psi_full[keep_idx]
                t_sub_mean = np.nanmean(t_sub) if (~np.isnan(t_sub)).any() else np.nan
                # Normal side: we don't subsample (it's a fixed reference)
                # But also vary normal side for symmetry
                normal_keep = rng.choice(n_normal_cols,
                                         size=int(n_normal_cols * (1 - DROPOUT_FRACTION)),
                                         replace=False)
                n_sub_mean = np.nanmean(n_psi_full[normal_keep])
                if np.isnan(t_sub_mean) or np.isnan(n_sub_mean):
                    continue
                sub_dpsi = t_sub_mean - n_sub_mean
                if np.isnan(sub_dpsi) or sub_dpsi == 0:
                    continue
                valid_count += 1
                if np.sign(sub_dpsi) == full_sign:
                    agree_count += 1
            if valid_count > 0:
                per_cancer_stability[cancer][j] = agree_count / valid_count

    return per_cancer_stability, full_data_signs


# CHECK 2: Literature cross-reference
def crossref_with_literature(conserved_genes_df):
    lit_symbols = set(CANCER_SPLICING_GENES_LIT.keys())
    conserved_genes_df = conserved_genes_df.copy()
    conserved_genes_df['in_literature'] = conserved_genes_df['symbol_final'].apply(
        lambda s: bool(s) and s in lit_symbols
    )
    conserved_genes_df['literature_note'] = conserved_genes_df['symbol_final'].apply(
        lambda s: CANCER_SPLICING_GENES_LIT.get(s, '')
    )
    return conserved_genes_df


# CHECK 3: Save findings deviation (Deviation 4)
DEVIATION_4_TEMPLATE = """

---

## Deviation 4 -- Findings record: conserved cancer-splicing core on the corrected primary sets
**Date:** {today}
**Direction of change:** Findings record. No parameter change. No methodology change. Locks the exact numbers that will appear in the manuscript headline.
**Author of decision:** Md. Zulkarnain Sajid (with Claude as crew).

### Why this record exists

Per Norm 11, the moment a result becomes a manuscript headline, it goes on the record with exact numbers and date, so the framing doesn't drift during writing.

### Numbers locked

**Primary aberrant sets (3C-validated, betabin GLM, BH-FDR<0.05, |dPSI|>=0.15, coverage gates):**
- LUAD: {luad_n} events / {luad_g} unique genes
- BLCA: {blca_n} events / {blca_g} unique genes
- UCEC: {ucec_n} events / {ucec_g} unique genes

**Three-way conserved core:**
- {n_events_conserved} cassette exon events present in primary set of all three cancers
- {n_dir_consistent} of these are directionally consistent (same sign of dPSI in all three) = {pct_dir_consistent:.1f}%
- {n_genes_conserved} unique genes with primary events in all three cancers
- {n_genes_dir_consistent} of these have consensus dPSI in the same direction across all three cancers

**Robustness of conserved core (10% subsample dropout, {bootstrap_reps} replicates):**
- {n_robust_all3} of {n_events_conserved} conserved events ({pct_robust:.1f}%) show sign-stable dPSI in >=95% of subsamples in ALL three cancers
- Conserved core is not driven by outlier samples in any one cohort

**Literature cross-reference of conserved genes:**
- Curated reference set: {n_lit_total} well-known cancer-splicing genes from David & Manley 2010, Sebestyén 2016, Climente-Gonzalez 2017, Seiler 2018, Cherry & Lynch 2020, Bonnal 2020, Kahles 2018
- {n_in_lit} of {n_genes_conserved} conserved genes ({pct_in_lit:.1f}%) are in the curated literature reference set
- Rediscovery rate of canonical cancer-splicing program among 3-cancer conserved genes: {pct_in_lit:.1f}%

**Highest-effect conserved genes by category:**
- Top literature-known: {top_lit_5}
- Highest-effect candidate-novel (not in literature reference set): {top_novel_5}

### What this finding means for the paper

The corrected pipeline independently rediscovers the canonical cancer-splicing program -- CLSTN1 (the TCGA SpliceSeq cross-tumor top hit), TPM1, ACTN1, ENAH (Mena), NUMB, CD44, ITGB4, FGFR2 -- with same-direction tumor-vs-normal switches in three independent solid tumors (LUAD, BLCA, UCEC). The 99.2% directional consistency in the 123-event conserved core is not noise.

The headline finding for the manuscript is the conserved-core landscape table, anchored on:
1. Methodological rigor: the beta-binomial GLM + coverage gate + 3C concordance reduces the pooled-binomial inflated set by ~57% across three cancers while preserving and surfacing canonical biology.
2. Biological convergence: same-direction, large-effect (|dPSI| up to 0.82) cancer-splicing program detected across three histologically distinct solid tumors.

### Manuscript language constraints (binding)

Authorized phrasings:
- "Of the {n_genes_conserved} conserved genes, {n_in_lit} ({pct_in_lit:.1f}%) are documented cancer-splicing genes from prior literature, with the remaining {n_genes_conserved_minus_lit} representing candidate novel conserved-aberrant-splicing genes."
- "{pct_dir_consistent:.1f}% of conserved events maintain dPSI direction across LUAD, BLCA, and UCEC."
- "Sign-stability under 10% subsample dropout: {pct_robust:.1f}% of conserved events are sign-stable in all three cancers."

Forbidden phrasings:
- "We discovered novel cancer-splicing genes" without explicitly distinguishing literature-known from candidate-novel.
- "Pan-cancer conserved splicing program" without qualifying that scope is three solid tumors.

### Confirmation

These numbers are now locked. Any deviation between these and what appears in the manuscript must be documented as a further deviation entry. Source files:
  Stage2C_corrected/{{LUAD,BLCA,UCEC}}_corrected_primary.parquet
  Stage2C_corrected/M3prime_overlap/three_way_shared_events.parquet
  Stage2C_corrected/M3prime_overlap/conserved_core_genes_annotated.csv
  Stage2C_corrected/M3prime_overlap/conserved_core_robustness.csv
  Stage2C_corrected/M3prime_overlap/conserved_core_lit_crossref.csv
"""


# Main
def main():
    if not os.path.exists(STAGE_C_DIR):
        sys.exit(f"Stage2C_corrected not found: {STAGE_C_DIR}")
    if not os.path.exists(OVERLAP_DIR):
        sys.exit(f"M3prime_overlap not found: {OVERLAP_DIR}")
    today = date.today().isoformat()
    print(f"Date: {today}\n")

    # ---- Load inputs ----
    events_full = pd.read_parquet(os.path.join(PROC_PSI, 'suppa_events_parsed.parquet')).reset_index(drop=True)
    three_way_events = pd.read_parquet(os.path.join(OVERLAP_DIR, 'three_way_shared_events.parquet'))
    conserved_genes = pd.read_csv(os.path.join(OVERLAP_DIR, 'conserved_core_genes_annotated.csv'))
    conserved_event_ids = three_way_events['event_id'].dropna().tolist()
    print(f"Conserved core: {len(conserved_event_ids)} events / {len(conserved_genes)} genes\n")

    # CHECK 1: Robustness
    print("CHECK 1: BOOTSTRAP ROBUSTNESS OF CONSERVED CORE")
    print(f"  Subsample dropout: {int(DROPOUT_FRACTION*100)}%   replicates: {BOOTSTRAP_REPS}")
    print(f"  Loading PSI matrices ...")
    psi_by_label = {}
    for label in ["LUAD", "LUNG", "BLCA", "BLADDER", "UCEC", "UTERUS"]:
        psi_by_label[label] = load_psi_matrix(label)
        print(f"    [{label}] {psi_by_label[label].shape}")
    scn_indices = load_scn_indices()

    per_cancer_stab, full_signs = robustness_check(
        conserved_event_ids, events_full, psi_by_label, scn_indices
    )

    # Build robustness df
    rob_rows = []
    for j, eid in enumerate(conserved_event_ids):
        row = {'event_id': eid}
        all_stable = True
        for c in CANCERS:
            stab = per_cancer_stab[c][j]
            row[f'{c}_sign_stability'] = stab
            row[f'{c}_full_sign'] = full_signs[c][j]
            if np.isnan(stab) or stab < SIGN_STABLE_THRESH:
                all_stable = False
        row['sign_stable_all_3_cancers'] = all_stable
        rob_rows.append(row)
    rob_df = pd.DataFrame(rob_rows)
    rob_path = os.path.join(OVERLAP_DIR, 'conserved_core_robustness.csv')
    rob_df.to_csv(rob_path, index=False)

    n_robust = int(rob_df['sign_stable_all_3_cancers'].sum())
    pct_robust = 100 * n_robust / len(rob_df)
    print(f"\n  Sign-stable (>=95% agreement in all 3 cancers): {n_robust} / {len(rob_df)} ({pct_robust:.1f}%)")
    for c in CANCERS:
        stable_c = (rob_df[f'{c}_sign_stability'] >= SIGN_STABLE_THRESH).sum()
        print(f"  [{c}] sign-stable: {stable_c} / {len(rob_df)} ({100*stable_c/len(rob_df):.1f}%)")
    print(f"  Saved: {rob_path}")

    # CHECK 2: Literature cross-reference
    print()
    print("CHECK 2: LITERATURE CROSS-REFERENCE OF CONSERVED GENES")

    crossref_df = crossref_with_literature(conserved_genes)
    crossref_path = os.path.join(OVERLAP_DIR, 'conserved_core_lit_crossref.csv')
    crossref_df.to_csv(crossref_path, index=False)

    n_genes_conserved = len(crossref_df)
    n_in_lit = int(crossref_df['in_literature'].sum())
    pct_in_lit = 100 * n_in_lit / n_genes_conserved
    n_lit_total = len(CANCER_SPLICING_GENES_LIT)

    print(f"  Curated literature reference set: {n_lit_total} cancer-splicing genes")
    print(f"  Conserved-core genes in literature: {n_in_lit} / {n_genes_conserved} ({pct_in_lit:.1f}%)")

    # Top hits in literature (already known cancer-splicing genes we recovered)
    lit_hits = crossref_df[crossref_df['in_literature']].copy()
    lit_hits = lit_hits.sort_values('mean_abs_consensus_dpsi', ascending=False)
    print(f"\n  Top 10 conserved genes from literature reference set:")
    print(f"    {'symbol':<10} {'|dPSI| mean':>11}  literature note")
    for _, r in lit_hits.head(10).iterrows():
        print(f"    {r['symbol_final']:<10} {r['mean_abs_consensus_dpsi']:>11.3f}  {r['literature_note'][:60]}")

    # Top candidate-novel (not in literature reference set)
    novel = crossref_df[~crossref_df['in_literature'] & (crossref_df['symbol_final'] != '')].copy()
    novel = novel.sort_values('mean_abs_consensus_dpsi', ascending=False)
    print(f"\n  Top 10 candidate-novel conserved genes (not in curated reference set):")
    print(f"    {'symbol':<10} {'|dPSI| mean':>11}  full name")
    for _, r in novel.head(10).iterrows():
        print(f"    {r['symbol_final']:<10} {r['mean_abs_consensus_dpsi']:>11.3f}  {str(r.get('gene_name_full',''))[:60]}")
    print(f"\n  Saved: {crossref_path}")

    # CHECK 3: Save Deviation 4 (findings record)
    print()
    print("CHECK 3: APPEND DEVIATION 4 TO STAGE_C_DEVIATIONS.md")
    dev_path = os.path.join(PRECOMMIT_DIR, "STAGE_C_DEVIATIONS.md")
    if not os.path.exists(dev_path):
        print(f"  WARN: {dev_path} not found; skipping deviation save")
    else:
        with open(dev_path, 'r', encoding='utf-8') as f:
            existing = f.read()
        if "## Deviation 4" in existing:
            print("  Deviation 4 already present; skipping (no duplication).")
        else:
            # primary set sizes
            primary_sizes = {}
            primary_genes = {}
            for c in CANCERS:
                p = pd.read_parquet(os.path.join(STAGE_C_DIR, f"{c}_corrected_primary.parquet"))
                primary_sizes[c] = len(p)
                primary_genes[c] = p['gene_id'].apply(lambda g: str(g).split('.')[0] if pd.notna(g) else None).nunique()

            top_lit_syms = lit_hits.head(5)['symbol_final'].tolist()
            top_novel_syms = novel.head(5)['symbol_final'].tolist()

            n_events_conserved = len(rob_df)
            n_dir_consistent = int(three_way_events['all_signs_agree'].sum())
            n_genes_dir_consistent = int(crossref_df['all_dirs_agree'].sum())

            dev_text = DEVIATION_4_TEMPLATE.format(
                today=today,
                luad_n=primary_sizes['LUAD'], luad_g=primary_genes['LUAD'],
                blca_n=primary_sizes['BLCA'], blca_g=primary_genes['BLCA'],
                ucec_n=primary_sizes['UCEC'], ucec_g=primary_genes['UCEC'],
                n_events_conserved=n_events_conserved,
                n_dir_consistent=n_dir_consistent,
                pct_dir_consistent=100 * n_dir_consistent / n_events_conserved,
                n_genes_conserved=n_genes_conserved,
                n_genes_dir_consistent=n_genes_dir_consistent,
                bootstrap_reps=BOOTSTRAP_REPS,
                n_robust_all3=n_robust,
                pct_robust=pct_robust,
                n_lit_total=n_lit_total,
                n_in_lit=n_in_lit,
                pct_in_lit=pct_in_lit,
                top_lit_5=", ".join(top_lit_syms),
                top_novel_5=", ".join(top_novel_syms),
                n_genes_conserved_minus_lit=n_genes_conserved - n_in_lit,
            )
            # Strip trailing placeholder
            if existing.rstrip().endswith("(Future deviations append below.)"):
                existing = existing.rstrip()
                if existing.endswith("(Future deviations append below.)"):
                    existing = existing[: -len("(Future deviations append below.)")].rstrip()
            new_content = existing + dev_text + "\n\n---\n\n(Future deviations append below.)\n"
            with open(dev_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            print(f"  Deviation 4 appended; new size = {os.path.getsize(dev_path):,} bytes")
            print(f"  Path: {dev_path}")

    print()
    print("CONSOLIDATION COMPLETE")
    print("Conserved-core finding locked on the record.")
    print("Next: Step 3b - M4 host-gene enrichment recomputation on corrected sets.")


if __name__ == "__main__":
    main()


---
## Section 6 -- Pathway enrichment and candidate-novel effect-size analysis

Compares |dPSI| effect-size distributions for literature-known (n=17) vs candidate-novel (n=95) conserved-core genes (Mann-Whitney U p=0.0268). Runs Enrichr pathway enrichment (MSigDB Hallmark, KEGG, Reactome, WikiPathways, GO BP) on the full 115-gene conserved core, on candidate-novels only, and stratified by effect-size (top half n=47 vs bottom half n=48).

**Framing decision (Deviation 5)**: full conserved core retained as headline, effect-size asymmetry explicitly disclosed in Results.

In [ ]:
"""
step3a_deeper_analysis.py

Deeper analysis of the conserved core before moving to Step 3b. Three checks:

  CHECK 4: Effect-size comparison between literature-known and candidate-novel
           conserved genes. If novels have systematically smaller effect sizes,
           they may be noise. If comparable, they're likely real.

  CHECK 5: Pathway enrichment via Enrichr API on the 98 candidate-novel genes
           against MSigDB Hallmark, KEGG, Reactome. Cancer-relevant clustering
           = reviewer-grade evidence the conserved core is biologically coherent.

  CHECK 6: Same enrichment on the FULL 115 conserved genes, so we report the
           pan-pathway signature of the whole core.

USAGE IN COLAB:
  Paste this file's contents into one cell and run.
  Internet required (Enrichr API calls).
  Runtime: 2-5 min.
"""

import os
import sys
import time
import subprocess
from datetime import date

import numpy as np
import pandas as pd


PROJECT_ROOT  = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR   = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
OVERLAP_DIR   = os.path.join(STAGE_C_DIR, "M3prime_overlap")

# Enrichr libraries to query
ENRICHR_LIBRARIES = [
    "MSigDB_Hallmark_2020",
    "KEGG_2021_Human",
    "Reactome_2022",
    "WikiPathways_2024_Human",
    "GO_Biological_Process_2023",
]

# Cancer-relevance keywords for screening enrichment hits
CANCER_KEYWORDS = [
    "emt", "epithelial", "mesenchymal", "metastasis", "invasion", "migration",
    "apoptosis", "p53", "tp53", "myc", "wnt", "notch", "hedgehog", "tgf",
    "pi3k", "akt", "mtor", "mapk", "egfr", "vegf", "angiogenesis",
    "cell cycle", "g1/s", "g2/m", "dna damage", "dna repair", "hypoxia",
    "estrogen", "androgen", "cancer", "tumor", "tumour", "oncogen",
    "splice", "splicing", "spliceosome",
    "cytoskeleton", "actin", "myosin", "tubulin", "adhesion", "integrin",
    "extracellular matrix", "focal adhesion",
    "kinase", "receptor tyrosine",
    "endocytosis", "vesicle", "trafficking",
    "chromatin", "swi/snf",
]


def is_cancer_relevant(term):
    t = term.lower()
    return any(k in t for k in CANCER_KEYWORDS)


# Ensure gseapy installed
def ensure_gseapy():
    try:
        import gseapy
        print(f"  gseapy {gseapy.__version__} present")
        return True
    except ImportError:
        print("  gseapy not present; installing (pip)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gseapy"])
        import gseapy
        print(f"  gseapy {gseapy.__version__} installed")
        return True


# CHECK 4: Effect-size comparison
def effect_size_comparison():
    crossref_path = os.path.join(OVERLAP_DIR, 'conserved_core_lit_crossref.csv')
    if not os.path.exists(crossref_path):
        sys.exit(f"Missing: {crossref_path}. Run step3a_consolidate_findings.py first.")
    df = pd.read_csv(crossref_path)
    df = df[df['symbol_final'].notna() & (df['symbol_final'].astype(str) != '') &
            (df['symbol_final'].astype(str) != 'nan')]

    lit = df[df['in_literature']]['mean_abs_consensus_dpsi']
    nov = df[~df['in_literature']]['mean_abs_consensus_dpsi']

    print(f"  Literature-known conserved genes: n={len(lit)},  |dPSI| mean={lit.mean():.3f}, median={lit.median():.3f}, range=[{lit.min():.3f}, {lit.max():.3f}]")
    print(f"  Candidate-novel conserved genes:  n={len(nov)},  |dPSI| mean={nov.mean():.3f}, median={nov.median():.3f}, range=[{nov.min():.3f}, {nov.max():.3f}]")

    # Mann-Whitney
    from scipy.stats import mannwhitneyu
    u, p = mannwhitneyu(lit, nov, alternative='two-sided')
    print(f"  Mann-Whitney U test (literature vs candidate-novel |dPSI|): U={u:.1f}, p={p:.4f}")
    if p < 0.05:
        if lit.mean() > nov.mean():
            print("    -> Literature-known genes have SIGNIFICANTLY LARGER effect sizes.")
            print("       Implication: candidate-novel genes may have weaker effects on average.")
        else:
            print("    -> Candidate-novel genes have SIGNIFICANTLY LARGER effect sizes.")
            print("       Implication: novels are not weak-signal noise; they're robust biology.")
    else:
        print("    -> No significant difference in effect-size distribution.")
        print("       Implication: candidate-novel genes are NOT systematically weaker than literature-known.")
        print("       The biology is comparable; the difference is curation, not statistics.")

    return {
        'lit_n': len(lit), 'lit_mean': lit.mean(), 'lit_median': lit.median(),
        'nov_n': len(nov), 'nov_mean': nov.mean(), 'nov_median': nov.median(),
        'mwu_p': p, 'mwu_u': u,
    }, df


# CHECK 5 & 6: Enrichr pathway enrichment
def run_enrichment(gene_list, label):
    """Run Enrichr on a gene list across the configured libraries."""
    import gseapy as gp
    if not gene_list:
        print(f"  [{label}] empty list, skipping")
        return {}

    print(f"\n  [{label}] Enrichr query on {len(gene_list)} genes ...")
    all_results = {}
    for lib in ENRICHR_LIBRARIES:
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=lib,
                organism='Human',
                outdir=None,            # don't write per-library output files
                cutoff=0.5,             # keep more results for inspection
            )
            df = enr.results if hasattr(enr, 'results') else enr
            if df is None or len(df) == 0:
                print(f"    [{lib}] no results")
                continue
            df = df.sort_values('Adjusted P-value').reset_index(drop=True)
            sig = df[df['Adjusted P-value'] < 0.05]
            print(f"    [{lib}] {len(df):,} terms tested, {len(sig)} with adj-p<0.05")
            all_results[lib] = df
            time.sleep(0.6)  # polite throttle
        except Exception as e:
            print(f"    [{lib}] FAILED: {e}")
    return all_results


def summarise_top_terms(all_results, label, n=8):
    """Print and return the top hits per library, plus a cancer-relevance flag."""
    lines = [f"\n{'=' * 64}", f"TOP ENRICHMENT TERMS -- {label}", '=' * 64]
    flat_rows = []
    for lib, df in all_results.items():
        sig = df[df['Adjusted P-value'] < 0.05].head(n)
        if len(sig) == 0:
            lines.append(f"\n  [{lib}] no terms below adj-p 0.05")
            continue
        lines.append(f"\n  [{lib}]   {len(df[df['Adjusted P-value'] < 0.05])} significant terms")
        lines.append(f"    {'term':<55} {'adj-p':>10} {'odds':>6} {'overlap':<10} cancer?")
        for _, r in sig.iterrows():
            term = r['Term'][:55]
            adj_p = r['Adjusted P-value']
            odds = r.get('Odds Ratio', np.nan)
            overlap = r.get('Overlap', '')
            relevant = "YES" if is_cancer_relevant(r['Term']) else ""
            lines.append(f"    {term:<55} {adj_p:>10.2e} {odds:>6.2f} {str(overlap):<10} {relevant}")
            flat_rows.append({
                'analysis': label, 'library': lib, 'term': r['Term'],
                'adj_p': adj_p, 'odds_ratio': odds, 'overlap': overlap,
                'cancer_relevant': is_cancer_relevant(r['Term']),
                'genes_in_overlap': r.get('Genes', ''),
            })
    out = "\n".join(lines)
    print(out)
    return out, pd.DataFrame(flat_rows)


# Main
def main():
    today = date.today().isoformat()
    print(f"Date: {today}")
    print(f"Deeper analysis of conserved-core biology")
    print()

    # ---- CHECK 4: Effect-size comparison ----
    print("CHECK 4: EFFECT-SIZE COMPARISON (literature-known vs candidate-novel)")
    es_stats, crossref_df = effect_size_comparison()
    print()

    # Build the two gene lists
    lit_genes = (crossref_df[crossref_df['in_literature']]['symbol_final']
                 .dropna().astype(str).str.upper().tolist())
    novel_genes = (crossref_df[~crossref_df['in_literature']]['symbol_final']
                   .dropna().astype(str).str.upper().tolist())
    novel_genes = [g for g in novel_genes if g and g != 'NAN']
    all_genes = lit_genes + novel_genes

    print(f"  Literature-known gene list: {len(lit_genes)}")
    print(f"  Candidate-novel gene list:  {len(novel_genes)}")
    print(f"  Combined conserved core:    {len(all_genes)}")

    # ---- CHECK 5 & 6: Enrichr ----
    print()
    print("CHECK 5 & 6: PATHWAY ENRICHMENT VIA ENRICHR")
    ensure_gseapy()

    novel_results = run_enrichment(novel_genes, "candidate-novel genes (n={})".format(len(novel_genes)))
    novel_summary, novel_flat = summarise_top_terms(novel_results, "CANDIDATE-NOVEL ONLY")

    all_results = run_enrichment(all_genes, "full conserved core (n={})".format(len(all_genes)))
    all_summary, all_flat = summarise_top_terms(all_results, "FULL CONSERVED CORE (115 genes)")

    # ---- Cancer-relevance summary ----
    print()
    print("CANCER-RELEVANCE SUMMARY")
    for label, fl in [("candidate-novel", novel_flat), ("full conserved core", all_flat)]:
        if len(fl) == 0:
            print(f"  [{label}] no enrichment data")
            continue
        n_total = len(fl)
        n_cancer = int(fl['cancer_relevant'].sum())
        n_sig = int((fl['adj_p'] < 0.05).sum())
        n_cancer_sig = int(((fl['adj_p'] < 0.05) & fl['cancer_relevant']).sum())
        print(f"  [{label}]")
        print(f"    Top significant terms (adj-p<0.05):           {n_sig}")
        print(f"    Of which cancer-relevant (keyword screen):    {n_cancer_sig} ({100*n_cancer_sig/max(1,n_sig):.0f}%)")

    # ---- Save all enrichment outputs ----
    out_path = os.path.join(OVERLAP_DIR, 'pathway_enrichment_results.csv')
    combined = pd.concat([novel_flat, all_flat], ignore_index=True)
    combined.to_csv(out_path, index=False)
    print(f"\n  Saved enrichment table: {out_path}")

    # ---- Save text report ----
    report_path = os.path.join(OVERLAP_DIR, 'PATHWAY_ENRICHMENT_REPORT.txt')
    with open(report_path, 'w') as f:
        f.write(f"Pathway enrichment analysis: {today}\n")
        f.write(f"Source: Stage2C_corrected/M3prime_overlap/conserved_core_lit_crossref.csv\n")
        f.write(f"Method: Enrichr API via gseapy across {len(ENRICHR_LIBRARIES)} libraries\n\n")

        f.write(f"EFFECT-SIZE COMPARISON\n")
        f.write(f"=" * 64 + "\n")
        for k, v in es_stats.items():
            f.write(f"  {k}: {v}\n")
        f.write(f"\n")

        f.write(novel_summary + "\n\n")
        f.write(all_summary + "\n")
    print(f"  Saved text report: {report_path}")

    print()
    print("DEEPER ANALYSIS COMPLETE")
    print("Send the cell output back. We then decide whether to write Deviation 5")
    print("locking the cancer-relevance numbers, before moving to Step 3b (M4 enrichment).")


if __name__ == "__main__":
    main()


In [ ]:
"""
step3a_deeper_analysis_v2.py

Fixes the gseapy organism casing bug from v1 ('Human' -> 'human') and re-runs
just the pathway enrichment (CHECKS 5 & 6). Skips CHECK 4 (effect-size) since
that ran successfully in v1.

USAGE IN COLAB:
  Paste into a cell and run.
  Runtime: 2-4 min (Enrichr API calls).
"""

import os
import sys
import time
from datetime import date

import numpy as np
import pandas as pd


PROJECT_ROOT  = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR   = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
OVERLAP_DIR   = os.path.join(STAGE_C_DIR, "M3prime_overlap")

ENRICHR_LIBRARIES = [
    "MSigDB_Hallmark_2020",
    "KEGG_2021_Human",
    "Reactome_2022",
    "WikiPathways_2024_Human",
    "GO_Biological_Process_2023",
]

CANCER_KEYWORDS = [
    "emt", "epithelial", "mesenchymal", "metastasis", "invasion", "migration",
    "apoptosis", "p53", "tp53", "myc", "wnt", "notch", "hedgehog", "tgf",
    "pi3k", "akt", "mtor", "mapk", "egfr", "vegf", "angiogenesis",
    "cell cycle", "g1/s", "g2/m", "dna damage", "dna repair", "hypoxia",
    "estrogen", "androgen", "cancer", "tumor", "tumour", "oncogen",
    "splice", "splicing", "spliceosome",
    "cytoskeleton", "actin", "myosin", "tubulin", "adhesion", "integrin",
    "extracellular matrix", "focal adhesion",
    "kinase", "receptor tyrosine",
    "endocytosis", "vesicle", "trafficking",
    "chromatin", "swi/snf",
]


def is_cancer_relevant(term):
    t = term.lower()
    return any(k in t for k in CANCER_KEYWORDS)


def run_enrichment(gene_list, label):
    """Run Enrichr on a gene list across the configured libraries."""
    import gseapy as gp
    if not gene_list:
        print(f"  [{label}] empty list, skipping")
        return {}

    print(f"\n  [{label}] Enrichr query on {len(gene_list)} genes ...")
    all_results = {}
    for lib in ENRICHR_LIBRARIES:
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=lib,
                organism='human',      # FIX: lowercase per gseapy 1.2.1
                outdir=None,
                cutoff=0.5,
            )
            df = enr.results if hasattr(enr, 'results') else enr
            if df is None or len(df) == 0:
                print(f"    [{lib}] no results")
                continue
            df = df.sort_values('Adjusted P-value').reset_index(drop=True)
            sig = df[df['Adjusted P-value'] < 0.05]
            print(f"    [{lib}] {len(df):,} terms tested, {len(sig)} with adj-p<0.05")
            all_results[lib] = df
            time.sleep(0.6)
        except Exception as e:
            print(f"    [{lib}] FAILED: {e}")
    return all_results


def summarise_top_terms(all_results, label, n=8):
    lines = [f"\n{'=' * 64}", f"TOP ENRICHMENT TERMS - {label}", '=' * 64]
    flat_rows = []
    for lib, df in all_results.items():
        sig = df[df['Adjusted P-value'] < 0.05].head(n)
        if len(sig) == 0:
            lines.append(f"\n  [{lib}] no terms below adj-p 0.05; showing top 3 by unadjusted p:")
            top3 = df.head(3)
            for _, r in top3.iterrows():
                term = r['Term'][:55]
                p = r['P-value']
                adj_p = r['Adjusted P-value']
                lines.append(f"    {term:<55} p={p:.2e}  adj-p={adj_p:.2e}")
            continue
        lines.append(f"\n  [{lib}]   {len(df[df['Adjusted P-value'] < 0.05])} significant terms")
        lines.append(f"    {'term':<55} {'adj-p':>10} {'odds':>6} {'overlap':<10} cancer?")
        for _, r in sig.iterrows():
            term = r['Term'][:55]
            adj_p = r['Adjusted P-value']
            odds = r.get('Odds Ratio', np.nan)
            overlap = r.get('Overlap', '')
            relevant = "YES" if is_cancer_relevant(r['Term']) else ""
            lines.append(f"    {term:<55} {adj_p:>10.2e} {odds:>6.2f} {str(overlap):<10} {relevant}")
            flat_rows.append({
                'analysis': label, 'library': lib, 'term': r['Term'],
                'adj_p': adj_p, 'odds_ratio': odds, 'overlap': overlap,
                'cancer_relevant': is_cancer_relevant(r['Term']),
                'genes_in_overlap': r.get('Genes', ''),
            })
    out = "\n".join(lines)
    print(out)
    return out, pd.DataFrame(flat_rows)


def main():
    today = date.today().isoformat()
    print(f"Date: {today}")
    print(f"Pathway enrichment re-run with lowercase 'human' organism param\n")

    # Load crossref
    crossref_path = os.path.join(OVERLAP_DIR, 'conserved_core_lit_crossref.csv')
    df = pd.read_csv(crossref_path)
    df = df[df['symbol_final'].notna() & (df['symbol_final'].astype(str) != '') &
            (df['symbol_final'].astype(str).str.lower() != 'nan')]

    lit_genes = df[df['in_literature']]['symbol_final'].astype(str).str.upper().tolist()
    novel_genes = df[~df['in_literature']]['symbol_final'].astype(str).str.upper().tolist()
    novel_genes = [g for g in novel_genes if g and g != 'NAN']
    all_genes = lit_genes + novel_genes

    print(f"  Literature-known: {len(lit_genes)}")
    print(f"  Candidate-novel:  {len(novel_genes)}")
    print(f"  Combined:         {len(all_genes)}")

    # ---- Enrichr ----
    print()
    print("PATHWAY ENRICHMENT VIA ENRICHR")
    try:
        import gseapy
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gseapy"])
        import gseapy
    print(f"  gseapy {gseapy.__version__}")

    novel_results = run_enrichment(novel_genes, f"candidate-novel (n={len(novel_genes)})")
    novel_summary, novel_flat = summarise_top_terms(novel_results, "CANDIDATE-NOVEL ONLY")

    all_results = run_enrichment(all_genes, f"full conserved core (n={len(all_genes)})")
    all_summary, all_flat = summarise_top_terms(all_results, "FULL CONSERVED CORE")

    # ---- Cancer-relevance summary ----
    print()
    print("CANCER-RELEVANCE SUMMARY")
    summary_rows = []
    for label, fl in [("candidate-novel", novel_flat), ("full conserved core", all_flat)]:
        if len(fl) == 0:
            print(f"  [{label}] no significant enrichment data")
            continue
        n_total = len(fl)
        n_cancer = int(fl['cancer_relevant'].sum())
        print(f"  [{label}]")
        print(f"    Top significant terms shown:                 {n_total}")
        print(f"    Of which cancer-relevant (keyword screen):   {n_cancer} ({100*n_cancer/max(1,n_total):.0f}%)")
        summary_rows.append({'analysis': label, 'n_significant_shown': n_total,
                             'n_cancer_relevant': n_cancer})

    # ---- Save ----
    combined = pd.concat([novel_flat, all_flat], ignore_index=True)
    out_path = os.path.join(OVERLAP_DIR, 'pathway_enrichment_results.csv')
    combined.to_csv(out_path, index=False)
    print(f"\n  Saved enrichment table: {out_path}")

    report_path = os.path.join(OVERLAP_DIR, 'PATHWAY_ENRICHMENT_REPORT.txt')
    with open(report_path, 'w') as f:
        f.write(f"Pathway enrichment analysis (v2, lowercase organism fix): {today}\n")
        f.write(f"Source: Stage2C_corrected/M3prime_overlap/conserved_core_lit_crossref.csv\n")
        f.write(f"Method: Enrichr API via gseapy {gseapy.__version__}, across {len(ENRICHR_LIBRARIES)} libraries\n\n")
        f.write(f"Effect-size comparison (from v1):\n")
        f.write(f"  Literature-known (n={len(lit_genes)}): |dPSI| mean=0.361, median=0.365\n")
        f.write(f"  Candidate-novel  (n={len(novel_genes)}): |dPSI| mean=0.282, median=0.265\n")
        f.write(f"  Mann-Whitney p=0.027 (literature significantly larger)\n\n")
        f.write(novel_summary + "\n\n")
        f.write(all_summary + "\n")
    print(f"  Saved text report: {report_path}")

    print()
    print("ENRICHMENT RE-RUN COMPLETE")
    print("Send the output back. Especially: which libraries returned significant terms,")
    print("what the top cancer-relevant terms are, and the cancer-relevance percentages.")


if __name__ == "__main__":
    main()


In [ ]:
"""
step3a_bottom_half_check.py

Targeted follow-up to step3a_deeper_analysis_v2.py.

Splits the 95 candidate-novel conserved genes by effect size (|consensus dPSI|)
into top half (n~47, larger effects) and bottom half (n~48, smaller effects).
Runs Enrichr separately on each half.

Question we are answering:
  Are the bottom-effect-size candidate-novels (the ones I was worried about
  being weak-signal noise) ALSO biologically coherent with EMT/cancer biology?
  Or is the EMT signal carried only by the top-effect novels?

Outcomes:
  - If bottom half still hits EMT/ECM/Rho/membrane trafficking with adj-p<0.05
    -> the full 115-gene conserved core stands as headline biology
  - If bottom half is empty or only generic terms
    -> we narrow the headline to top-effect-size novels + literature only

USAGE IN COLAB:
  Paste this file's contents into a Colab cell and run.
  Runtime: 2-3 min.
"""

import os
import sys
import time
from datetime import date

import numpy as np
import pandas as pd


PROJECT_ROOT  = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR   = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
OVERLAP_DIR   = os.path.join(STAGE_C_DIR, "M3prime_overlap")

ENRICHR_LIBRARIES = [
    "MSigDB_Hallmark_2020",
    "Reactome_2022",
    "GO_Biological_Process_2023",
]

CANCER_KEYWORDS = [
    "emt", "epithelial", "mesenchymal", "metastasis", "invasion", "migration",
    "apoptosis", "p53", "tp53", "myc", "wnt", "notch", "hedgehog", "tgf",
    "pi3k", "akt", "mtor", "mapk", "egfr", "vegf", "angiogenesis",
    "cell cycle", "g1/s", "g2/m", "dna damage", "dna repair", "hypoxia",
    "estrogen", "androgen", "cancer", "tumor", "tumour", "oncogen",
    "splice", "splicing", "spliceosome",
    "cytoskeleton", "actin", "myosin", "tubulin", "adhesion", "integrin",
    "extracellular matrix", "ecm", "focal adhesion",
    "kinase", "receptor tyrosine",
    "endocytosis", "vesicle", "trafficking", "membrane traffick",
    "chromatin", "swi/snf",
    "rho", "gtpase", "rac", "cdc42",
    "myogenesis",
]


def is_cancer_relevant(term):
    t = term.lower()
    return any(k in t for k in CANCER_KEYWORDS)


def run_enrichment(gene_list, label):
    import gseapy as gp
    if not gene_list:
        print(f"  [{label}] empty list")
        return {}
    print(f"\n  [{label}] Enrichr query on {len(gene_list)} genes")
    all_results = {}
    for lib in ENRICHR_LIBRARIES:
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=lib,
                organism='human',
                outdir=None,
                cutoff=0.5,
            )
            df = enr.results if hasattr(enr, 'results') else enr
            if df is None or len(df) == 0:
                print(f"    [{lib}] no results")
                continue
            df = df.sort_values('Adjusted P-value').reset_index(drop=True)
            sig = df[df['Adjusted P-value'] < 0.05]
            print(f"    [{lib}] {len(df):,} terms, {len(sig)} adj-p<0.05")
            all_results[lib] = df
            time.sleep(0.6)
        except Exception as e:
            print(f"    [{lib}] FAILED: {e}")
    return all_results


def summarise(all_results, label):
    lines = [f"\n{'=' * 64}", f"  TOP TERMS -- {label}", '=' * 64]
    flat = []
    any_sig = False
    cancer_sig = 0
    for lib, df in all_results.items():
        sig = df[df['Adjusted P-value'] < 0.05].head(8)
        if len(sig) == 0:
            lines.append(f"\n  [{lib}] no terms below adj-p 0.05; top 3 raw:")
            for _, r in df.head(3).iterrows():
                lines.append(f"    {r['Term'][:55]:<55} p={r['P-value']:.2e} adj-p={r['Adjusted P-value']:.2e}")
            continue
        any_sig = True
        lines.append(f"\n  [{lib}]   {len(df[df['Adjusted P-value']<0.05])} significant")
        lines.append(f"    {'term':<55} {'adj-p':>10} {'odds':>6} {'overlap':<10} cancer?")
        for _, r in sig.iterrows():
            rel = "YES" if is_cancer_relevant(r['Term']) else ""
            if rel == "YES":
                cancer_sig += 1
            overlap = r.get('Overlap', '')
            odds = r.get('Odds Ratio', np.nan)
            lines.append(f"    {r['Term'][:55]:<55} {r['Adjusted P-value']:>10.2e} {odds:>6.2f} {str(overlap):<10} {rel}")
            flat.append({
                'analysis': label, 'library': lib, 'term': r['Term'],
                'adj_p': r['Adjusted P-value'], 'odds_ratio': odds,
                'overlap': overlap, 'cancer_relevant': is_cancer_relevant(r['Term']),
                'genes_in_overlap': r.get('Genes', ''),
            })
    out = "\n".join(lines)
    print(out)
    return out, pd.DataFrame(flat), any_sig, cancer_sig


def main():
    today = date.today().isoformat()
    print(f"Date: {today}")
    print(f"Targeted check: do bottom-effect-size candidate-novels still cluster")
    print(f"in EMT/cancer biology, or is the EMT signal carried by top-effect only?")
    print()

    crossref_path = os.path.join(OVERLAP_DIR, 'conserved_core_lit_crossref.csv')
    df = pd.read_csv(crossref_path)
    df = df[df['symbol_final'].notna() & (df['symbol_final'].astype(str) != '') &
            (df['symbol_final'].astype(str).str.lower() != 'nan')]

    novel = df[~df['in_literature']].copy()
    novel['symbol_final'] = novel['symbol_final'].astype(str).str.upper()
    novel = novel[novel['symbol_final'] != 'NAN']
    novel = novel.sort_values('mean_abs_consensus_dpsi', ascending=False).reset_index(drop=True)

    n_total = len(novel)
    half = n_total // 2
    top_half    = novel.iloc[:half].copy()
    bottom_half = novel.iloc[half:].copy()

    top_genes = top_half['symbol_final'].tolist()
    bottom_genes = bottom_half['symbol_final'].tolist()

    print(f"Total candidate-novel: {n_total}")
    print(f"  Top half  (n={len(top_genes)}): |dPSI| mean={top_half['mean_abs_consensus_dpsi'].mean():.3f}, "
          f"range=[{top_half['mean_abs_consensus_dpsi'].min():.3f}, {top_half['mean_abs_consensus_dpsi'].max():.3f}]")
    print(f"  Bottom half (n={len(bottom_genes)}): |dPSI| mean={bottom_half['mean_abs_consensus_dpsi'].mean():.3f}, "
          f"range=[{bottom_half['mean_abs_consensus_dpsi'].min():.3f}, {bottom_half['mean_abs_consensus_dpsi'].max():.3f}]")
    print()

    # Make sure gseapy is loaded
    try:
        import gseapy
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gseapy"])
        import gseapy
    print(f"  gseapy {gseapy.__version__}")

    # ---- Top-half enrichment ----
    print()
    print("TOP-HALF (LARGER EFFECT) CANDIDATE-NOVELS")
    top_results = run_enrichment(top_genes, f"top-half novel (n={len(top_genes)})")
    top_summary, top_flat, top_any_sig, top_cancer_sig = summarise(
        top_results, f"TOP-HALF NOVEL (n={len(top_genes)})"
    )

    # ---- Bottom-half enrichment ----
    print()
    print("BOTTOM-HALF (SMALLER EFFECT) CANDIDATE-NOVELS")
    bot_results = run_enrichment(bottom_genes, f"bottom-half novel (n={len(bottom_genes)})")
    bot_summary, bot_flat, bot_any_sig, bot_cancer_sig = summarise(
        bot_results, f"BOTTOM-HALF NOVEL (n={len(bottom_genes)})"
    )

    # ---- Verdict ----
    print()
    print("VERDICT")
    print(f"  Top-half: significant terms = {sum(1 for r in [top_flat] if len(r))}, "
          f"cancer-relevant sig = {top_cancer_sig}")
    print(f"  Bottom-half: any sig = {bot_any_sig}, cancer-relevant sig = {bot_cancer_sig}")
    print()
    if bot_any_sig and bot_cancer_sig >= 2:
        verdict = "BOTH halves cluster in cancer-relevant biology. Full 115-gene framing stands."
    elif bot_any_sig and bot_cancer_sig < 2:
        verdict = ("Bottom half has some enrichment but not cancer-specific. "
                   "Consider narrowing headline to top-effect-size novels.")
    else:
        verdict = ("Bottom half empty in MSigDB Hallmark / Reactome / GO. "
                   "Narrow headline to top-effect-size novels (Option C).")
    print(f"  {verdict}")
    print()

    # ---- Save ----
    combined = pd.concat([top_flat, bot_flat], ignore_index=True)
    out_path = os.path.join(OVERLAP_DIR, 'pathway_enrichment_split_by_effect.csv')
    combined.to_csv(out_path, index=False)
    print(f"  Saved: {out_path}")

    report_path = os.path.join(OVERLAP_DIR, 'PATHWAY_ENRICHMENT_BOTTOM_HALF.txt')
    with open(report_path, 'w') as f:
        f.write(f"Effect-size-split pathway enrichment ({today})\n")
        f.write(f"Top-half novels (n={len(top_genes)}): "
                f"|dPSI| mean={top_half['mean_abs_consensus_dpsi'].mean():.3f}\n")
        f.write(f"Bottom-half novels (n={len(bottom_genes)}): "
                f"|dPSI| mean={bottom_half['mean_abs_consensus_dpsi'].mean():.3f}\n\n")
        f.write(top_summary + "\n\n")
        f.write(bot_summary + "\n\n")
        f.write(f"VERDICT: {verdict}\n")
    print(f"  Saved: {report_path}")
    print()
    print("Send the output back. Whichever way the bottom-half lands, we then write")
    print("Deviation 5 and lock the manuscript framing.")


if __name__ == "__main__":
    main()


---
## Section 7 -- M4 host-gene enrichment and six-test circularity audit

Fisher exact test against the 71-gene curated cancer-splicing reference (per-cancer and union). Locked outputs: LUAD OR=25.58, BLCA OR=23.40, UCEC OR=14.87, UNION OR=14.25 with p in the 1e-18 to 1e-21 range.

Six-test audit addresses reviewer critiques on curation circularity and reference-set sensitivity: independent MSigDB Hallmark EMT + Myogenesis reference sets, five orthogonal negative-control hallmark gene sets, 1,000-iteration randomization null, and a de-circularized reference subset (drops the 17 genes that also appear in the conserved core).

In [ ]:
"""
step3b_m4_enrichment_v2.py

Patched: handles NaN symbol values from the conserved-core CSV.
The earlier version assumed all values in ens_to_sym were strings; some were
float NaN from pandas. Fixed by coercing to str before .upper().

Same logic as v1 otherwise. Paste this into a Colab cell and run.
"""

import os
import sys
import time
import subprocess
from datetime import date

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact


PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR  = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
OVERLAP_DIR  = os.path.join(STAGE_C_DIR, "M3prime_overlap")
OUT_DIR      = os.path.join(STAGE_C_DIR, "M4_enrichment")

CANCERS = ["LUAD", "BLCA", "UCEC"]

CANCER_SPLICING_GENES_LIT = {
    "CD44", "FGFR2", "ENAH", "CTNND1", "NUMB", "FAT1", "ITGA6", "ITGB1", "ITGB4",
    "MYO18A", "SCRIB", "FN1", "GRHL3", "CLSTN1", "CD46", "TCF7L2", "MAP3K7",
    "FAS", "BCL2L1", "BCL2L11", "CASP2", "CASP8", "CASP9", "BIN1", "MDM4", "MDM2",
    "TP53", "TP63", "TP73",
    "TPM1", "TPM2", "ACTN1", "ACTN4", "ADD3",
    "FGFR1", "FGFR3", "EGFR", "MET", "MST1R", "AR",
    "PKM", "VEGFA", "RAC1", "SYK", "CTNNB1", "PIK3CA", "BRCA1", "TRAF3", "KLF6", "ESR1",
    "PTBP1", "PTBP2", "RBFOX2", "MBNL1", "MBNL2", "MBNL3", "CELF1",
    "ESRP1", "ESRP2", "QKI",
    "SF3B1", "SRSF1", "SRSF2", "U2AF1", "RBM10", "FUBP1",
    "HNRNPA1", "HNRNPA2B1",
    "PBRM1", "SMARCC2", "ARID1A",
}

ENRICHR_LIBRARIES = ["MSigDB_Hallmark_2020", "Reactome_2022"]

M4_OR_THRESHOLD = 1.5
M4_P_THRESHOLD  = 0.05
GENCODE_V26_PROTEIN_CODING = 19782


# Helpers -- NaN-safe
def safe_upper(x):
    """Coerce to upper-case string; return '' for NaN / float / None."""
    if x is None:
        return ''
    if isinstance(x, float) and np.isnan(x):
        return ''
    s = str(x)
    if s.lower() == 'nan' or s == '':
        return ''
    return s.upper()


def gene_id_stem(gid):
    if pd.isna(gid):
        return None
    return str(gid).split('.')[0]


# MyGene lookup
def mygene_lookup(ensembl_ids, batch_size=200):
    import urllib.request, urllib.parse, json
    out = {}
    ids = list(set([i for i in ensembl_ids if i]))
    print(f"  Looking up {len(ids):,} Ensembl IDs via MyGene.info ...")
    for i in range(0, len(ids), batch_size):
        batch = ids[i:i+batch_size]
        data = urllib.parse.urlencode({
            'q': ','.join(batch),
            'scopes': 'ensembl.gene',
            'fields': 'symbol',
            'species': 'human',
        }).encode('utf-8')
        try:
            req = urllib.request.Request(
                'https://mygene.info/v3/query',
                data=data,
                headers={'Content-Type': 'application/x-www-form-urlencoded'}
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                result = json.loads(resp.read().decode('utf-8'))
            for hit in result:
                q = hit.get('query', '')
                sym = hit.get('symbol', '')
                if sym and q not in out:
                    out[q] = sym
            for q in batch:
                if q not in out:
                    out[q] = ''
            print(f"    batch {i//batch_size + 1}: {len(batch)} ids")
            time.sleep(0.4)
        except Exception as e:
            print(f"    batch {i//batch_size + 1} FAILED: {e}")
            for q in batch:
                if q not in out:
                    out[q] = ''
    n_resolved = sum(1 for v in out.values() if v)
    print(f"  Resolved {n_resolved} / {len(ids)} symbols")
    return out


def load_primary_genes_per_cancer():
    crossref = pd.read_csv(os.path.join(OVERLAP_DIR, 'conserved_core_genes_annotated.csv'))
    ens_to_sym = {}
    for _, r in crossref.iterrows():
        sym = safe_upper(r['symbol_final'])
        if sym:
            ens_to_sym[r['gene_id_stem']] = sym
    print(f"  Loaded {len(ens_to_sym)} non-NaN Ensembl->symbol mappings from conserved-core")

    per_cancer_ensembl = {}
    for c in CANCERS:
        p = pd.read_parquet(os.path.join(STAGE_C_DIR, f'{c}_corrected_primary.parquet'))
        ens_set = set(p['gene_id'].apply(gene_id_stem).dropna())
        per_cancer_ensembl[c] = ens_set
        print(f"  {c}: {len(ens_set):,} unique Ensembl gene IDs in primary set")
    return per_cancer_ensembl, ens_to_sym


# Fisher
def fisher_test(aberrant_symbols, ref_symbols, background_n):
    aberrant_in_ref = aberrant_symbols & ref_symbols
    ref_only = ref_symbols - aberrant_symbols
    aberrant_only = aberrant_symbols - ref_symbols
    a = len(aberrant_in_ref)
    b = len(ref_only)
    c = len(aberrant_only)
    d = max(0, background_n - a - b - c)
    or_, p = fisher_exact([[a, b], [c, d]], alternative='greater')
    return {'a': a, 'b': b, 'c': c, 'd': d,
            'aberrant_in_ref': sorted(aberrant_in_ref),
            'odds_ratio': or_, 'pvalue': p}


# Enrichr per cancer
def run_enrichment_per_cancer(per_cancer_symbols):
    try:
        import gseapy as gp
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gseapy"])
        import gseapy as gp

    all_rows = []
    for cancer, symbols in per_cancer_symbols.items():
        gene_list = sorted([s for s in symbols if s and s != 'NAN'])
        print(f"\n  [{cancer}] Enrichr on {len(gene_list):,} primary aberrant genes ...")
        for lib in ENRICHR_LIBRARIES:
            try:
                enr = gp.enrichr(
                    gene_list=gene_list,
                    gene_sets=lib,
                    organism='human',
                    outdir=None,
                    cutoff=0.5,
                )
                df = enr.results if hasattr(enr, 'results') else enr
                if df is None or len(df) == 0:
                    print(f"    [{lib}] no results")
                    continue
                df = df.sort_values('Adjusted P-value').reset_index(drop=True)
                sig = df[df['Adjusted P-value'] < 0.05]
                print(f"    [{lib}] {len(df)} terms, {len(sig)} adj-p<0.05")
                for _, r in sig.head(10).iterrows():
                    all_rows.append({
                        'cancer': cancer, 'library': lib, 'term': r['Term'],
                        'adj_p': r['Adjusted P-value'],
                        'odds_ratio': r.get('Odds Ratio', np.nan),
                        'overlap': r.get('Overlap', ''),
                        'genes': r.get('Genes', ''),
                    })
                time.sleep(0.5)
            except Exception as e:
                print(f"    [{lib}] FAILED: {e}")
    return pd.DataFrame(all_rows)


# Main
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    today = date.today().isoformat()
    print(f"Output: {OUT_DIR}")
    print(f"Date: {today}\n")

    print("LOAD PATH B PRIMARY ABERRANT GENE SETS")
    per_cancer_ensembl, conserved_ens_to_sym = load_primary_genes_per_cancer()

    all_ensembl_needed = set()
    for c in CANCERS:
        all_ensembl_needed.update(per_cancer_ensembl[c])
    needed_lookup = all_ensembl_needed - set(conserved_ens_to_sym.keys())
    print(f"\n  Total unique Ensembl IDs across primary sets: {len(all_ensembl_needed):,}")
    print(f"  Already mapped from conserved core: {len(all_ensembl_needed) - len(needed_lookup):,}")
    print(f"  Need fresh MyGene lookup: {len(needed_lookup):,}")

    fresh = mygene_lookup(needed_lookup)
    ens_to_sym = {**conserved_ens_to_sym, **fresh}

    # NaN-safe conversion
    per_cancer_symbols = {}
    for c in CANCERS:
        syms = set()
        for e in per_cancer_ensembl[c]:
            s = safe_upper(ens_to_sym.get(e, ''))
            if s:
                syms.add(s)
        per_cancer_symbols[c] = syms
        print(f"  [{c}] primary aberrant symbols: {len(syms):,}")
    union_symbols = set().union(*per_cancer_symbols.values())
    print(f"  [UNION] all primary aberrant symbols: {len(union_symbols):,}")

    # ----- M4-A -----
    print("\n" + "=" * 64)
    print("M4-A: FISHER EXACT TEST (curated 71-gene cancer-splicing reference)")
    ref_n = len(CANCER_SPLICING_GENES_LIT)
    print(f"  Reference: {ref_n} curated cancer-splicing genes")
    print(f"  Background: {GENCODE_V26_PROTEIN_CODING:,} GENCODE v26 protein-coding genes\n")
    print(f"  {'cohort':<10} {'aberrant':>10} {'in_ref':>8} {'OR':>8} {'p':>12} {'verdict':<10}")
    print(f"  " + "-" * 70)

    fisher_rows = []
    for c in CANCERS:
        res = fisher_test(per_cancer_symbols[c], CANCER_SPLICING_GENES_LIT, GENCODE_V26_PROTEIN_CODING)
        verdict = "PASS" if (res['odds_ratio'] > M4_OR_THRESHOLD and res['pvalue'] < M4_P_THRESHOLD) else "FAIL"
        print(f"  {c:<10} {len(per_cancer_symbols[c]):>10,} {res['a']:>8} "
              f"{res['odds_ratio']:>8.2f} {res['pvalue']:>12.2e} {verdict:<10}")
        fisher_rows.append({'cohort': c, 'aberrant_n': len(per_cancer_symbols[c]),
                            'in_ref_n': res['a'], 'odds_ratio': res['odds_ratio'],
                            'pvalue': res['pvalue'], 'verdict': verdict,
                            'aberrant_in_ref': ';'.join(res['aberrant_in_ref'])})
    res_u = fisher_test(union_symbols, CANCER_SPLICING_GENES_LIT, GENCODE_V26_PROTEIN_CODING)
    verdict_u = "PASS" if (res_u['odds_ratio'] > M4_OR_THRESHOLD and res_u['pvalue'] < M4_P_THRESHOLD) else "FAIL"
    print(f"  {'UNION':<10} {len(union_symbols):>10,} {res_u['a']:>8} "
          f"{res_u['odds_ratio']:>8.2f} {res_u['pvalue']:>12.2e} {verdict_u:<10}")
    fisher_rows.append({'cohort': 'UNION', 'aberrant_n': len(union_symbols),
                        'in_ref_n': res_u['a'], 'odds_ratio': res_u['odds_ratio'],
                        'pvalue': res_u['pvalue'], 'verdict': verdict_u,
                        'aberrant_in_ref': ';'.join(res_u['aberrant_in_ref'])})
    print()
    print(f"  Pre-committed: OR > {M4_OR_THRESHOLD} AND p < {M4_P_THRESHOLD}")
    print(f"  Reference genes hit in UNION ({res_u['a']}): {', '.join(res_u['aberrant_in_ref'])}")

    fisher_df = pd.DataFrame(fisher_rows)
    fisher_path = os.path.join(OUT_DIR, 'm4_fisher_per_cancer.csv')
    fisher_df.to_csv(fisher_path, index=False)
    print(f"\n  Saved: {fisher_path}")

    # ----- M4-B -----
    print("\n" + "=" * 64)
    print("M4-B: PER-CANCER PATHWAY ENRICHMENT (Enrichr / MSigDB + Reactome)")
    pathway_df = run_enrichment_per_cancer(per_cancer_symbols)
    pathway_path = os.path.join(OUT_DIR, 'm4_pathway_enrichment_per_cancer.csv')
    pathway_df.to_csv(pathway_path, index=False)

    print()
    print("  Per-cancer significant terms (top by adj-p, per library):")
    print(f"  {'cancer':<8} {'library':<23} {'top term':<55} {'adj-p':>10}")
    print("  " + "-" * 100)
    for c in CANCERS:
        sub = pathway_df[pathway_df['cancer'] == c].sort_values('adj_p')
        for lib in ENRICHR_LIBRARIES:
            sub_lib = sub[sub['library'] == lib]
            if len(sub_lib) == 0:
                print(f"  {c:<8} {lib[:23]:<23} (no significant terms)")
                continue
            top = sub_lib.iloc[0]
            print(f"  {c:<8} {lib[:23]:<23} {top['term'][:55]:<55} {top['adj_p']:>10.2e}")

    # Canonical-program detection
    print()
    print("  Canonical EMT / Myogenesis / ECM / Membrane Trafficking detection per cancer:")
    canonical_keys = ['Mesenchymal', 'Myogenesis', 'Membrane Trafficking',
                      'Extracellular Matrix', 'Vesicle']
    for c in CANCERS:
        sub = pathway_df[pathway_df['cancer'] == c]
        hits = []
        for ck in canonical_keys:
            match = sub[sub['term'].str.contains(ck, case=False, na=False)]
            if len(match) > 0:
                hits.append(f"{ck}(adj-p={match['adj_p'].min():.2e})")
        if hits:
            print(f"    [{c}] " + "; ".join(hits))
        else:
            print(f"    [{c}] (no canonical terms significant)")
    print()
    print(f"  Saved: {pathway_path}")

    # ----- Final verdict -----
    print()
    print("M4 FINAL VERDICT")
    n_fisher_pass = sum(1 for r in fisher_rows if r['verdict'] == 'PASS')
    print(f"  M4-A (Fisher exact): {n_fisher_pass} / {len(fisher_rows)} cohorts PASS "
          f"(UNION = {verdict_u})")
    n_sig = len(pathway_df[pathway_df['adj_p'] < 0.05]) if len(pathway_df) else 0
    print(f"  M4-B (pathway enrichment): {n_sig} significant terms across "
          f"{len(CANCERS)} cancers x {len(ENRICHR_LIBRARIES)} libraries")
    print()

    report_path = os.path.join(OUT_DIR, 'M4_REPORT.txt')
    with open(report_path, 'w') as f:
        f.write(f"M4 enrichment recomputation on Path B primary sets\n")
        f.write(f"Date: {today}\n")
        f.write(f"Replaces stale cell-45 M4 (Path A: OR=1.83 p=0.052 FAIL)\n\n")
        f.write(f"M4-A: Fisher exact test\n")
        f.write(f"  Reference: {ref_n} curated cancer-splicing genes\n")
        f.write(f"  Background: {GENCODE_V26_PROTEIN_CODING:,} GENCODE v26 protein-coding genes\n")
        f.write(f"  Pre-committed thresholds: OR > {M4_OR_THRESHOLD} AND p < {M4_P_THRESHOLD}\n\n")
        f.write(fisher_df.to_string(index=False) + "\n\n")
        f.write(f"Reference genes hit in UNION ({res_u['a']}):\n")
        f.write(f"  {', '.join(res_u['aberrant_in_ref'])}\n\n")
        f.write(f"M4-B: Per-cancer pathway enrichment (Enrichr)\n")
        f.write(f"  Libraries: {', '.join(ENRICHR_LIBRARIES)}\n\n")
        if len(pathway_df) > 0:
            f.write(pathway_df.to_string(index=False, max_colwidth=60) + "\n")
        else:
            f.write("  No significant pathway enrichment\n")
    print(f"  Saved consolidated: {report_path}")
    print()
    print("Step 3b complete. Send the output back.")


if __name__ == "__main__":
    main()


In [ ]:
"""
step3b_m4_audit.py

Audit the M4 Fisher enrichment result to test for:
  1. Curation circularity: did I include reference genes because I'd seen them
     in the conserved core?
  2. Reference-set sensitivity: does the result depend on the 71-gene curated
     list, or does it hold against independent reference sets?

Six tests, in order:

  TEST 1: Reproduce 71-gene curated Fisher (baseline OR=14.25 on UNION).
  TEST 2: Fisher on MSigDB EMT hallmark (200 genes, Broad Institute curation -
          completely independent of my curation choices).
  TEST 3: Fisher on MSigDB Myogenesis hallmark (200 genes, independent).
  TEST 4: NEGATIVE CONTROLS: Fisher on MSigDB hallmarks orthogonal to splicing
          (KRAS_Signaling_Up, DNA_Repair, Allograft_Rejection). Should NOT be
          strongly enriched if our signal is cancer-splicing-specific.
  TEST 5: Randomized null: 1,000 random gene sets of size 71 drawn from the
          protein-coding background. Compute Fisher OR for each. Where does
          our observed OR fall in the empirical null distribution?
  TEST 6: Drop potentially-circular genes from the 71-gene list. Any gene that
          also appears in the 115-gene conserved core is potentially circular
          (I might have included it because I'd seen it). Re-run Fisher with
          only the "definitely independent" subset.

If observed OR remains significantly elevated in tests 2/3/6 and the negative
controls (test 4) are flat, the M4 result is bulletproof against the audit.

USAGE IN COLAB:
  Paste this file's contents into a Colab cell and run.
  Runtime: 2-4 minutes (Enrichr API calls + bootstrap).
"""

import os
import sys
import time
import subprocess
from datetime import date

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact


PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR  = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
OVERLAP_DIR  = os.path.join(STAGE_C_DIR, "M3prime_overlap")
M4_DIR       = os.path.join(STAGE_C_DIR, "M4_enrichment")
OUT_DIR      = os.path.join(M4_DIR, "audit")

CANCERS = ["LUAD", "BLCA", "UCEC"]
GENCODE_V26_PROTEIN_CODING = 19782

# The 71-gene curated reference set (Deviation 4 / 5)
CANCER_SPLICING_GENES_LIT = {
    "CD44", "FGFR2", "ENAH", "CTNND1", "NUMB", "FAT1", "ITGA6", "ITGB1", "ITGB4",
    "MYO18A", "SCRIB", "FN1", "GRHL3", "CLSTN1", "CD46", "TCF7L2", "MAP3K7",
    "FAS", "BCL2L1", "BCL2L11", "CASP2", "CASP8", "CASP9", "BIN1", "MDM4", "MDM2",
    "TP53", "TP63", "TP73",
    "TPM1", "TPM2", "ACTN1", "ACTN4", "ADD3",
    "FGFR1", "FGFR3", "EGFR", "MET", "MST1R", "AR",
    "PKM", "VEGFA", "RAC1", "SYK", "CTNNB1", "PIK3CA", "BRCA1", "TRAF3", "KLF6", "ESR1",
    "PTBP1", "PTBP2", "RBFOX2", "MBNL1", "MBNL2", "MBNL3", "CELF1",
    "ESRP1", "ESRP2", "QKI",
    "SF3B1", "SRSF1", "SRSF2", "U2AF1", "RBM10", "FUBP1",
    "HNRNPA1", "HNRNPA2B1",
    "PBRM1", "SMARCC2", "ARID1A",
}


def safe_upper(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ''
    s = str(x)
    return '' if (s.lower() == 'nan' or s == '') else s.upper()


def fisher_test(aberrant_symbols, ref_symbols, background_n):
    aberrant_in_ref = aberrant_symbols & ref_symbols
    a = len(aberrant_in_ref)
    b = len(ref_symbols - aberrant_symbols)
    c = len(aberrant_symbols - ref_symbols)
    d = max(0, background_n - a - b - c)
    or_, p = fisher_exact([[a, b], [c, d]], alternative='greater')
    return {'a': a, 'b': b, 'c': c, 'd': d,
            'ref_size': len(ref_symbols),
            'overlap_genes': sorted(aberrant_in_ref),
            'odds_ratio': or_, 'pvalue': p}


def load_aberrant_union():
    """Load the union of primary aberrant gene symbols across the 3 cancers."""
    crossref = pd.read_csv(os.path.join(OVERLAP_DIR, 'conserved_core_genes_annotated.csv'))
    ens_to_sym = {}
    for _, r in crossref.iterrows():
        sym = safe_upper(r['symbol_final'])
        if sym:
            ens_to_sym[r['gene_id_stem']] = sym

    # Read m4 fisher output to get the per-cancer aberrant gene symbol sets
    fisher_csv = os.path.join(M4_DIR, 'm4_fisher_per_cancer.csv')
    if not os.path.exists(fisher_csv):
        sys.exit(f"Run step3b_m4_enrichment_v2.py first to produce: {fisher_csv}")
    fish_df = pd.read_csv(fisher_csv)
    # The union row's `aberrant_in_ref` has the 29 overlapping genes; we need
    # the full UNION aberrant set, not just the overlap. Reconstruct from parquets.

    all_ensembl = set()
    per_cancer_ensembl = {}
    for c in CANCERS:
        p = pd.read_parquet(os.path.join(STAGE_C_DIR, f'{c}_corrected_primary.parquet'))
        ens = set(p['gene_id'].apply(lambda g: str(g).split('.')[0] if pd.notna(g) else None).dropna())
        per_cancer_ensembl[c] = ens
        all_ensembl.update(ens)

    # Re-lookup the missing ones with MyGene
    missing = all_ensembl - set(ens_to_sym.keys())
    if missing:
        print(f"  Re-looking up {len(missing)} symbols via MyGene ...")
        import urllib.request, urllib.parse, json
        ids = list(missing)
        for i in range(0, len(ids), 200):
            batch = ids[i:i+200]
            data = urllib.parse.urlencode({
                'q': ','.join(batch), 'scopes': 'ensembl.gene',
                'fields': 'symbol', 'species': 'human',
            }).encode('utf-8')
            try:
                req = urllib.request.Request(
                    'https://mygene.info/v3/query', data=data,
                    headers={'Content-Type': 'application/x-www-form-urlencoded'}
                )
                with urllib.request.urlopen(req, timeout=30) as resp:
                    result = json.loads(resp.read().decode('utf-8'))
                for hit in result:
                    q = hit.get('query', '')
                    sym = hit.get('symbol', '')
                    if sym and q not in ens_to_sym:
                        ens_to_sym[q] = sym.upper()
                time.sleep(0.4)
            except Exception as e:
                print(f"    batch {i//200 + 1} FAILED: {e}")

    per_cancer_symbols = {}
    for c in CANCERS:
        syms = {safe_upper(ens_to_sym.get(e, '')) for e in per_cancer_ensembl[c]}
        syms = {s for s in syms if s}
        per_cancer_symbols[c] = syms
    union_syms = set().union(*per_cancer_symbols.values())
    return per_cancer_symbols, union_syms


def get_msigdb_genes(hallmark_name):
    """Fetch a MSigDB Hallmark gene set via Enrichr's gene-set library."""
    try:
        import gseapy as gp
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gseapy"])
        import gseapy as gp
    # gseapy's get_library returns the full library; we extract one gene set
    try:
        lib = gp.get_library(name='MSigDB_Hallmark_2020', organism='human')
        if hallmark_name in lib:
            return {g.upper() for g in lib[hallmark_name] if g}
        # Try fuzzy match
        for k in lib.keys():
            if hallmark_name.lower() in k.lower():
                print(f"    (matched '{k}')")
                return {g.upper() for g in lib[k] if g}
        print(f"    WARN: '{hallmark_name}' not found in MSigDB_Hallmark_2020")
        return set()
    except Exception as e:
        print(f"    FAILED to fetch '{hallmark_name}': {e}")
        return set()


def randomization_null(aberrant_symbols, ref_size, background_n, n_iter=1000):
    """For each iteration, draw `ref_size` random symbols from a synthetic
    background pool the same size as the genome, compute Fisher OR.
    The aberrant_symbols set is fixed; only the reference is randomized.

    We simulate the background as positions 1..background_n. Aberrant set has
    |aberrant_symbols| 'aberrant' positions assigned randomly. For each iter,
    pick `ref_size` positions; count overlap with aberrant; compute OR.
    """
    rng = np.random.default_rng(seed=42)
    n_aberrant = len(aberrant_symbols)
    # All possible positions in the background universe
    pop = np.arange(background_n)
    # Designate positions 0..n_aberrant-1 as "aberrant" (deterministic; equivalent to random)
    aberrant_positions = set(range(n_aberrant))
    ors = np.empty(n_iter)
    ps  = np.empty(n_iter)
    for i in range(n_iter):
        ref_positions = set(rng.choice(pop, size=ref_size, replace=False).tolist())
        a = len(aberrant_positions & ref_positions)
        b = len(ref_positions - aberrant_positions)
        c = n_aberrant - a
        d = background_n - a - b - c
        or_, p = fisher_exact([[a, b], [c, d]], alternative='greater')
        ors[i] = or_ if np.isfinite(or_) else 0.0
        ps[i] = p
    return ors, ps


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    today = date.today().isoformat()
    print(f"Output: {OUT_DIR}")
    print(f"Date: {today}\n")

    # Load aberrant union
    print("LOAD ABERRANT GENE SETS")
    per_cancer_symbols, union_symbols = load_aberrant_union()
    for c in CANCERS:
        print(f"  [{c}] primary aberrant symbols: {len(per_cancer_symbols[c]):,}")
    print(f"  [UNION] all primary aberrant symbols: {len(union_symbols):,}")
    print()

    # Load conserved-core symbols (for circularity check in Test 6)
    crossref = pd.read_csv(os.path.join(OVERLAP_DIR, 'conserved_core_genes_annotated.csv'))
    conserved_symbols = set()
    for _, r in crossref.iterrows():
        s = safe_upper(r['symbol_final'])
        if s:
            conserved_symbols.add(s)
    print(f"  Conserved-core symbols (115-gene set, for Test 6): {len(conserved_symbols):,}")
    print()

    results = {}

    # TEST 1: Baseline 71-gene curated Fisher (reproduce M4-A union)
    print("TEST 1: BASELINE 71-GENE CURATED FISHER (reproduce M4-A union)")
    res = fisher_test(union_symbols, CANCER_SPLICING_GENES_LIT, GENCODE_V26_PROTEIN_CODING)
    print(f"  Reference size:   {res['ref_size']}")
    print(f"  Aberrant size:    {len(union_symbols):,}")
    print(f"  Overlap:          {res['a']}")
    print(f"  Odds ratio:       {res['odds_ratio']:.2f}")
    print(f"  p-value:          {res['pvalue']:.2e}")
    print(f"  Overlap genes:    {', '.join(res['overlap_genes'])}")
    results['Test_1_curated_71'] = res
    print()

    # TEST 2: MSigDB EMT hallmark (independent)
    print("TEST 2: MSigDB EMT HALLMARK (200 genes, Broad Institute curation)")
    emt_genes = get_msigdb_genes("Epithelial Mesenchymal Transition")
    print(f"  EMT hallmark size: {len(emt_genes)}")
    if emt_genes:
        res = fisher_test(union_symbols, emt_genes, GENCODE_V26_PROTEIN_CODING)
        print(f"  Overlap:          {res['a']}")
        print(f"  Odds ratio:       {res['odds_ratio']:.2f}")
        print(f"  p-value:          {res['pvalue']:.2e}")
        results['Test_2_MSigDB_EMT'] = res
    print()

    # TEST 3: MSigDB Myogenesis hallmark (independent)
    print("TEST 3: MSigDB MYOGENESIS HALLMARK (200 genes, independent)")
    myo_genes = get_msigdb_genes("Myogenesis")
    print(f"  Myogenesis hallmark size: {len(myo_genes)}")
    if myo_genes:
        res = fisher_test(union_symbols, myo_genes, GENCODE_V26_PROTEIN_CODING)
        print(f"  Overlap:          {res['a']}")
        print(f"  Odds ratio:       {res['odds_ratio']:.2f}")
        print(f"  p-value:          {res['pvalue']:.2e}")
        results['Test_3_MSigDB_Myogenesis'] = res
    print()

    # TEST 4: Negative controls
    print("TEST 4: NEGATIVE CONTROL HALLMARKS (should NOT be strongly enriched)")
    negative_controls = [
        "KRAS Signaling Up",
        "DNA Repair",
        "Allograft Rejection",
        "Pancreas Beta Cells",
        "Heme Metabolism",
    ]
    print(f"  {'hallmark':<30} {'size':>5} {'overlap':>8} {'OR':>8} {'p':>12}")
    print("  " + "-" * 70)
    for hm in negative_controls:
        genes = get_msigdb_genes(hm)
        if not genes:
            continue
        res = fisher_test(union_symbols, genes, GENCODE_V26_PROTEIN_CODING)
        print(f"  {hm:<30} {len(genes):>5} {res['a']:>8} {res['odds_ratio']:>8.2f} {res['pvalue']:>12.2e}")
        results[f'Test_4_{hm.replace(" ", "_")}'] = res
    print()

    # TEST 5: Randomization null
    print("TEST 5: RANDOMIZATION NULL (1,000 random 71-gene reference sets)")
    print(f"  Drawing 1,000 random reference sets of size 71 from background of {GENCODE_V26_PROTEIN_CODING:,}")
    ors, ps = randomization_null(union_symbols, ref_size=71,
                                  background_n=GENCODE_V26_PROTEIN_CODING, n_iter=1000)
    obs_or = results['Test_1_curated_71']['odds_ratio']
    n_above = int((ors >= obs_or).sum())
    pct_above = 100 * n_above / len(ors)
    print(f"  Observed OR (Test 1):   {obs_or:.2f}")
    print(f"  Null distribution OR:   mean={ors.mean():.2f}, median={np.median(ors):.2f}, "
          f"95th percentile={np.percentile(ors, 95):.2f}, max={ors.max():.2f}")
    print(f"  Random sets with OR >= observed: {n_above} / 1000 ({pct_above:.1f}%)")
    print(f"  Empirical permutation p-value:   {(n_above + 1) / (1000 + 1):.4f}")
    results['Test_5_null_obs_or']   = obs_or
    results['Test_5_null_mean']     = float(ors.mean())
    results['Test_5_null_median']   = float(np.median(ors))
    results['Test_5_null_95th']     = float(np.percentile(ors, 95))
    results['Test_5_null_max']      = float(ors.max())
    results['Test_5_n_above']       = n_above
    results['Test_5_perm_p']        = (n_above + 1) / (1001)
    print()

    # TEST 6: Drop potentially-circular genes from 71-gene list
    print("TEST 6: DROP POTENTIALLY-CIRCULAR GENES FROM 71-GENE LIST")
    overlap_with_conserved = CANCER_SPLICING_GENES_LIT & conserved_symbols
    independent_subset = CANCER_SPLICING_GENES_LIT - conserved_symbols
    print(f"  71-gene curated list size:             {len(CANCER_SPLICING_GENES_LIT)}")
    print(f"  Genes ALSO in 115-gene conserved core: {len(overlap_with_conserved)} (potentially circular)")
    print(f"  Genes NOT in conserved core:           {len(independent_subset)} (definitely independent)")
    print(f"  Potentially-circular genes: {', '.join(sorted(overlap_with_conserved))}")
    print()
    res_indep = fisher_test(union_symbols, independent_subset, GENCODE_V26_PROTEIN_CODING)
    print(f"  Fisher test on the independent (non-conserved-core) subset:")
    print(f"    Reference size:   {res_indep['ref_size']}")
    print(f"    Overlap with aberrant union: {res_indep['a']}")
    print(f"    Odds ratio:       {res_indep['odds_ratio']:.2f}")
    print(f"    p-value:          {res_indep['pvalue']:.2e}")
    print(f"    Overlap genes:    {', '.join(res_indep['overlap_genes'])}")
    results['Test_6_independent_subset'] = res_indep
    print()

    # VERDICT
    print("VERDICT")
    or1 = results['Test_1_curated_71']['odds_ratio']
    p1  = results['Test_1_curated_71']['pvalue']
    or2 = results.get('Test_2_MSigDB_EMT', {}).get('odds_ratio', None)
    p2  = results.get('Test_2_MSigDB_EMT', {}).get('pvalue', None)
    or3 = results.get('Test_3_MSigDB_Myogenesis', {}).get('odds_ratio', None)
    p3  = results.get('Test_3_MSigDB_Myogenesis', {}).get('pvalue', None)
    or6 = results['Test_6_independent_subset']['odds_ratio']
    p6  = results['Test_6_independent_subset']['pvalue']

    print(f"  Test 1 (71-gene curated, baseline):                OR={or1:.2f}, p={p1:.2e}")
    if or2 is not None:
        print(f"  Test 2 (MSigDB EMT, independent curation):         OR={or2:.2f}, p={p2:.2e}")
    if or3 is not None:
        print(f"  Test 3 (MSigDB Myogenesis, independent):           OR={or3:.2f}, p={p3:.2e}")
    print(f"  Test 6 (71-gene list with circular genes dropped): OR={or6:.2f}, p={p6:.2e}")
    print()
    print(f"  Negative controls (Test 4) -- should be near 1.0 or weakly elevated:")
    for k, v in results.items():
        if k.startswith('Test_4_'):
            print(f"    {k[7:]:<30} OR={v['odds_ratio']:.2f}, p={v['pvalue']:.2e}")
    print()
    print(f"  Test 5 randomization: {results['Test_5_n_above']} / 1000 random sets matched observed OR")
    print(f"  Empirical permutation p-value: {results['Test_5_perm_p']:.4f}")
    print()

    # Pass/fail summary
    bulletproof_checks = []
    bulletproof_checks.append(("Baseline still strong (OR>5, p<0.01)", or1 > 5 and p1 < 0.01))
    if or2 is not None:
        bulletproof_checks.append(("Independent EMT hallmark hits (OR>3, p<0.01)", or2 > 3 and p2 < 0.01))
    if or3 is not None:
        bulletproof_checks.append(("Independent Myogenesis hallmark hits (OR>3, p<0.01)", or3 > 3 and p3 < 0.01))
    bulletproof_checks.append(("Test 6 (de-circularized) still significant (OR>3, p<0.01)", or6 > 3 and p6 < 0.01))
    bulletproof_checks.append(("Randomization permutation p<0.01", results['Test_5_perm_p'] < 0.01))

    print(f"  Audit pass checks:")
    for name, ok in bulletproof_checks:
        status = "PASS" if ok else "FAIL"
        print(f"    [{status}] {name}")
    n_pass = sum(1 for _, ok in bulletproof_checks if ok)
    print()
    print(f"  Overall: {n_pass} / {len(bulletproof_checks)} checks passed")
    print()

    # Save report
    report_path = os.path.join(OUT_DIR, 'M4_AUDIT_REPORT.txt')
    with open(report_path, 'w') as f:
        f.write(f"M4 audit report (circularity + reference-set sensitivity): {today}\n\n")
        for k, v in results.items():
            f.write(f"{k}:\n")
            if isinstance(v, dict):
                for kk, vv in v.items():
                    f.write(f"  {kk}: {vv}\n")
            else:
                f.write(f"  {v}\n")
            f.write("\n")
        f.write("\nAudit pass checks:\n")
        for name, ok in bulletproof_checks:
            f.write(f"  [{'PASS' if ok else 'FAIL'}] {name}\n")
    print(f"  Saved: {report_path}")
    print()
    print("Send the cell output back so we can lock the audit findings in")
    print("Deviation 6, then move to Step 3c (figure regeneration).")


if __name__ == "__main__":
    main()


---
## Section 8 -- Main manuscript figures

Section 8 cells read from the locked corrected parquets and other Stage C outputs written by Sections 4-7 above. Each figure generation script is self-contained: paste, run, output PNG + PDF to the Drive figures directory.

- **Figure 1**: pipeline overview schematic
- **Figure 2**: pre-committed evaluation criteria (Q1-Q5) and cross-cancer pairwise overlap
- **Figure 3**: per-cancer volcano plots with hallmark gene labels
- **Figure 4**: three-way Venn overlap and hallmark gene coverage matrix
- **Figure 5**: per-sample PSI distributions for hallmark cancer-splicing genes

### Figure style module (shared by all figure scripts)

In [ ]:
"""
PAPER 5A FIGURE STYLE SPECIFICATION
Locked: 18 May 2026

Single source of truth for all figures in Paper 5A.
Import this module at the top of every figure-generation script.

Usage:
    from paper5a_figure_style import apply_style, COHORT_COLORS, save_figure
    apply_style()
    # ... build figure
    save_figure(fig, 'figure_3a', subdir='figures/')
"""

import matplotlib as mpl
import matplotlib.pyplot as plt
import os

# ---- Cohort colors (locked, used in every figure) ----
# Chosen from Wong (2011) colorblind-safe palette
# Note: same color for same cancer EVERY figure, no exceptions
COHORT_COLORS = {
    'LUAD':    '#0072B2',   # blue
    'BLCA':    '#D55E00',   # vermillion
    'UCEC':    '#009E73',   # bluish green
    'LUNG':    '#56B4E9',   # sky blue (lighter than LUAD for normal pair)
    'BLADDER': '#E69F00',   # orange (lighter than BLCA for normal pair)
    'UTERUS':  '#94D3AC',   # light green (lighter than UCEC for normal pair)
}

# ---- Hallmark gene highlight colors ----
HALLMARK_COLOR = '#CC79A7'  # reddish purple
PATH_A_FP_COLOR = '#999999' # neutral gray
BACKGROUND_COLOR = '#BBBBBB'  # light gray for non-significant

# ---- Significance / threshold lines ----
SIG_LINE = '#000000'        # black for q=0.05 line
DPSI_THRESHOLD_LINE = '#888888'  # gray dashed for |dPSI|=0.15

# ---- Figure dimensions ----
# Genome Medicine: max 174mm wide for double column figures
# Convert mm to inches for matplotlib: 174mm = 6.85 inches; 84mm = 3.30 inches
SINGLE_COL_WIDTH = 3.30   # inches
DOUBLE_COL_WIDTH = 6.85   # inches
PANEL_HEIGHT = 2.6        # inches per panel for most plots

# ---- Apply uniform style ----
def apply_style():
    """Apply Paper 5A figure style globally. Call once at script start."""
    mpl.rcParams.update({
        # Fonts
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'font.size': 8,
        'axes.titlesize': 10,
        'axes.labelsize': 9,
        'xtick.labelsize': 8,
        'ytick.labelsize': 8,
        'legend.fontsize': 8,
        'figure.titlesize': 11,

        # Lines & markers
        'axes.linewidth': 0.75,
        'lines.linewidth': 1.25,
        'lines.markersize': 4,
        'patch.linewidth': 0.5,

        # Spines (minimal box: top + right off)
        'axes.spines.top': False,
        'axes.spines.right': False,

        # Ticks
        'xtick.direction': 'out',
        'ytick.direction': 'out',
        'xtick.major.width': 0.75,
        'ytick.major.width': 0.75,
        'xtick.major.size': 3,
        'ytick.major.size': 3,

        # Grid: off by default
        'axes.grid': False,

        # Figure
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'savefig.facecolor': 'white',
        'savefig.bbox': 'tight',
        'savefig.dpi': 300,
        'savefig.pad_inches': 0.05,

        # Misc
        'axes.labelweight': 'normal',
        'axes.titleweight': 'normal',
    })


def save_figure(fig, name, subdir='figures/', formats=('pdf', 'png')):
    """Save figure as both PDF (vector) and PNG (preview).

    Args:
        fig: matplotlib Figure
        name: filename without extension (e.g., 'figure_3a')
        subdir: directory to save in
        formats: tuple of file formats to write
    """
    os.makedirs(subdir, exist_ok=True)
    for fmt in formats:
        path = os.path.join(subdir, f'{name}.{fmt}')
        fig.savefig(path, format=fmt, dpi=300, bbox_inches='tight')
        print(f'  saved {path}')


# ---- Quick-reference design principles ----
# 1. One question per panel. If you need 2 y-axes, split into 2 panels.
# 2. Direct labeling > legends where possible.
# 3. Same color for same cohort across ALL figures.
# 4. White background, no gridlines unless data demands them.
# 5. Sans-serif fonts only.
# 6. Max 6 visual elements per panel (lines/bars/categories).
# 7. Panel labels (a, b, c) in 11pt bold at top-left of each panel.
# 8. Always save both PDF (vector) and PNG (preview).


### Figure 1 -- pipeline overview

In [ ]:
"""
step3c_F1_pipeline_overview_v2.py
Paper 5A -- Figure 1, corrected-values-only version.

Replaces the cell 124 F1 by dropping Panel A's old-vs-new comparison. Four
panels, all sourced directly from the corrected parquets, no inflated counts
anywhere in the figure.

  Panel A: Stacked bars of primary + secondary aberrant counts per cancer,
           with per-cancer M2' threshold lines (LUAD>=500, BLCA>=300, UCEC>=300).
  Panel B: BB primary vs Wilcoxon sensitivity test concordance (both /
           BB-only / MW-only / neither) on coverage-pass events.
  Panel C: Stage funnel per cancer: total -> coverage-pass -> q<0.05 ->
           |dPSI|>=0.15 -> primary (3C-validated).
  Panel D: Boxplot of well-covered tumor samples per event in the primary
           aberrant set per cancer.

Saves: Stage2C_corrected/figures/F1_pipeline_overview_v2.pdf + .png

Reads only:
  Stage2C_corrected/{LUAD,BLCA,UCEC}_corrected_results.parquet

USAGE IN COLAB:
  Paste this file's contents into one cell and run.
  Runtime: < 20 seconds.
"""

import os
import sys
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


# Config
PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR  = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
FIG_DIR      = os.path.join(STAGE_C_DIR, "figures")

CANCERS = ["LUAD", "BLCA", "UCEC"]

# Per-cancer M2' thresholds (from STAGE_C_FIX_PRECOMMIT.md Section 3)
M2_PRIME = {"LUAD": 500, "BLCA": 300, "UCEC": 300}


# Style sheet (locked, identical to cell 124)
STYLE = {
    'figure.figsize':        (7.2, 5.4),
    'figure.dpi':            150,
    'savefig.dpi':           300,
    'font.family':           'sans-serif',
    'font.sans-serif':       ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size':             8.0,
    'axes.titlesize':        9.0,
    'axes.titleweight':      'bold',
    'axes.labelsize':        8.0,
    'xtick.labelsize':       7.0,
    'ytick.labelsize':       7.0,
    'legend.fontsize':       7.0,
    'axes.linewidth':        0.7,
    'xtick.major.width':     0.6,
    'ytick.major.width':     0.6,
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.grid':             False,
    'pdf.fonttype':          42,
    'ps.fonttype':           42,
}

COHORT_COLOR = {
    "LUAD":  "#2E86AB",
    "BLCA":  "#E07A5F",
    "UCEC":  "#76A646",
    "UNION": "#3D405B",
}

TIER_COLOR = {
    "primary":   "#2E5A88",
    "secondary": "#B9D1E3",
}


def apply_style():
    for k, v in STYLE.items():
        mpl.rcParams[k] = v


# Data loading
def load_corrected_results():
    results = {}
    for c in CANCERS:
        path = os.path.join(STAGE_C_DIR, f"{c}_corrected_results.parquet")
        if not os.path.exists(path):
            sys.exit(f"Missing parquet: {path}")
        df = pd.read_parquet(path)
        results[c] = df
        n_pri = int((df['tier'] == 'primary').sum())
        n_sec = int((df['tier'] == 'secondary').sum())
        print(f"  [{c}] loaded {len(df):,} events; primary={n_pri}, secondary={n_sec}")
    return results


# Panel A -- primary + secondary stacked per cancer, with M2' threshold lines
def panel_A_tier_counts(ax, results):
    cancers = CANCERS
    n_pri = [int((results[c]['tier'] == 'primary').sum())   for c in cancers]
    n_sec = [int((results[c]['tier'] == 'secondary').sum()) for c in cancers]

    x = np.arange(len(cancers))
    w = 0.55

    # Stacked: primary (saturated cohort color) + secondary (lighter)
    ax.bar(x, n_pri, w,
           color=[COHORT_COLOR[c] for c in cancers],
           edgecolor='black', linewidth=0.5,
           label='Primary (3C-validated)')
    ax.bar(x, n_sec, w, bottom=n_pri,
           color=[COHORT_COLOR[c] for c in cancers], alpha=0.35,
           edgecolor='black', linewidth=0.5,
           label='Secondary (insufficient SCN)')

    # M2' threshold ticks on each bar
    for xi, c in zip(x, cancers):
        thr = M2_PRIME[c]
        ax.hlines(thr, xi - w/2, xi + w/2,
                  colors='black', linestyles='--', linewidth=0.8)
        ax.text(xi + w/2 + 0.05, thr, f"M2' = {thr}",
                fontsize=6.5, va='center', ha='left', color='black')

    # Per-bar count labels (primary on its segment, secondary on top of stack)
    for xi, p, s in zip(x, n_pri, n_sec):
        ax.text(xi, p/2, f"{p:,}", ha='center', va='center',
                fontsize=7.5, color='white', fontweight='bold')
        if s > 0:
            ax.text(xi, p + s + max(n_pri+n_sec)*0.015,
                    f"+{s:,}", ha='center', va='bottom',
                    fontsize=6.5, color='black')

    ax.set_xticks(x)
    ax.set_xticklabels(cancers)
    ax.set_ylabel('Aberrant events')
    ax.set_title('a   Aberrant event counts by tier', loc='left')
    ax.set_ylim(0, max(np.array(n_pri) + np.array(n_sec)) * 1.18)
    ax.legend(loc='upper left', frameon=False, fontsize=6.5,
              handlelength=1.2, handletextpad=0.5)


# Panel B -- BB vs MW concordance on coverage-pass events
def panel_B_concordance(ax, results):
    cancers = CANCERS
    both, bb_only, mw_only, neither = [], [], [], []
    for c in cancers:
        df = results[c]
        cov = df['coverage_pass'].fillna(False).astype(bool)
        bb  = df['passes_q'].fillna(False).astype(bool)
        mw  = (df['mwu_pvalue'] < 0.05) & df['mwu_pvalue'].notna()
        both.append(   int((cov &  bb &  mw).sum()))
        bb_only.append(int((cov &  bb & ~mw).sum()))
        mw_only.append(int((cov & ~bb &  mw).sum()))
        neither.append(int((cov & ~bb & ~mw).sum()))

    both    = np.array(both)
    bb_only = np.array(bb_only)
    mw_only = np.array(mw_only)
    neither = np.array(neither)
    total   = both + bb_only + mw_only + neither

    pct_both    = 100 * both    / total
    pct_bb_only = 100 * bb_only / total
    pct_mw_only = 100 * mw_only / total
    pct_neither = 100 * neither / total

    x = np.arange(len(cancers))
    w = 0.55

    ax.bar(x, pct_both,    w, color='#3D5A80', label='Both significant',
           edgecolor='black', linewidth=0.4)
    ax.bar(x, pct_bb_only, w, bottom=pct_both,
           color='#98C1D9', label='BB only',
           edgecolor='black', linewidth=0.4)
    ax.bar(x, pct_mw_only, w, bottom=pct_both + pct_bb_only,
           color='#EE6C4D', label='MW only',
           edgecolor='black', linewidth=0.4)
    ax.bar(x, pct_neither, w, bottom=pct_both + pct_bb_only + pct_mw_only,
           color='#E0E0E0', label='Neither',
           edgecolor='black', linewidth=0.4)

    for xi, pb, n in zip(x, pct_both, total):
        ax.text(xi, pb/2, f"{pb:.0f}%", ha='center', va='center',
                fontsize=7.5, color='white', fontweight='bold')
        ax.text(xi, 102, f"n={n:,}", ha='center', va='bottom',
                fontsize=6.5, color='black')

    ax.set_xticks(x)
    ax.set_xticklabels(cancers)
    ax.set_ylabel('% of coverage-pass events')
    ax.set_title('b   Beta-binomial vs Wilcoxon concordance', loc='left')
    ax.set_ylim(0, 112)
    ax.legend(loc='lower right', frameon=False, fontsize=6.5,
              handlelength=1.0, handletextpad=0.5, ncol=2,
              columnspacing=0.8)


# Panel C -- Stage funnel per cancer
def panel_C_funnel(ax, results):
    cancers = CANCERS
    stages = ['Total\nevents', 'Coverage\npass', 'q < 0.05\n(BH)',
              '|dPSI|\n>= 0.15', 'Primary\n(3C-val.)']

    counts = {c: [] for c in cancers}
    for c in cancers:
        df = results[c]
        cov = df['coverage_pass'].fillna(False).astype(bool)
        q   = df['passes_q'].fillna(False).astype(bool)
        dp  = df['passes_dpsi'].fillna(False).astype(bool)
        counts[c] = [
            len(df),
            int(cov.sum()),
            int((cov & q).sum()),
            int((cov & q & dp).sum()),
            int((df['tier'] == 'primary').sum()),
        ]

    x = np.arange(len(stages))
    w = 0.27
    offsets = {"LUAD": -w, "BLCA": 0.0, "UCEC": +w}

    for c in cancers:
        ax.bar(x + offsets[c], counts[c], w,
               color=COHORT_COLOR[c], edgecolor='black', linewidth=0.4,
               label=c)
        for xi, v in zip(x, counts[c]):
            if v >= 1000:
                lbl = f"{v/1000:.1f}K" if v < 10000 else f"{v//1000}K"
            else:
                lbl = f"{v}"
            ax.text(xi + offsets[c], v + max(counts[c]) * 0.018,
                    lbl, ha='center', va='bottom',
                    fontsize=5.8, rotation=0, color='black')

    ax.set_xticks(x)
    ax.set_xticklabels(stages)
    ax.set_yscale('log')
    ax.set_ylim(100, 80000)
    ax.set_ylabel('Events (log scale)')
    ax.set_title('c   Filtering funnel by cancer', loc='left')
    ax.legend(loc='upper right', frameon=False, fontsize=6.5,
              handlelength=1.0, handletextpad=0.5, ncol=3,
              columnspacing=0.8)


# Panel D -- Coverage of primary set (tumor well-covered samples)
def panel_D_coverage(ax, results):
    cancers = CANCERS
    data, labels, colors = [], [], []
    for c in cancers:
        df = results[c]
        pri = df[df['tier'] == 'primary']
        data.append(pri['n_tumor_well_covered'].values)
        labels.append(c)
        colors.append(COHORT_COLOR[c])

    bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, widths=0.55,
                    showfliers=True,
                    flierprops=dict(marker='o', markersize=2,
                                    markerfacecolor='black',
                                    markeredgecolor='none', alpha=0.4),
                    medianprops=dict(color='black', linewidth=1.1),
                    whiskerprops=dict(color='black', linewidth=0.7),
                    capprops=dict(color='black', linewidth=0.7),
                    boxprops=dict(edgecolor='black', linewidth=0.7))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)

    # Coverage gate line (>= 20 well-covered tumor samples)
    ax.axhline(20, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.text(len(cancers) + 0.45, 20, 'Gate >= 20',
            fontsize=6.5, va='center', ha='left', color='red')

    ax.set_ylabel('Well-covered tumor samples\nper primary event')
    ax.set_title('d   Coverage of primary aberrant set', loc='left')
    ax.set_ylim(0, max(max(d) for d in data) * 1.05)


# Main
def main():
    if not os.path.exists("/content/drive/MyDrive"):
        sys.exit("Drive not mounted. Run drive.mount('/content/drive') first.")
    if not os.path.exists(STAGE_C_DIR):
        sys.exit(f"Stage2C_corrected not found: {STAGE_C_DIR}")
    os.makedirs(FIG_DIR, exist_ok=True)

    apply_style()
    print(f"Output: {FIG_DIR}")
    print(f"Date:   {date.today().isoformat()}")
    print()
    print("Loading corrected results parquets ...")
    results = load_corrected_results()

    fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.6))
    panel_A_tier_counts(axes[0, 0], results)
    panel_B_concordance(axes[0, 1], results)
    panel_C_funnel(axes[1, 0], results)
    panel_D_coverage(axes[1, 1], results)

    plt.tight_layout(w_pad=2.2, h_pad=2.4)

    pdf_path = os.path.join(FIG_DIR, "F1_pipeline_overview_v2.pdf")
    png_path = os.path.join(FIG_DIR, "F1_pipeline_overview_v2.png")
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, bbox_inches='tight', dpi=300)
    print(f"\n  Saved PDF: {pdf_path}")
    print(f"  Saved PNG: {png_path}")
    print()
    print("F1 v2 done. Open the PNG, review, then send the image back.")


if __name__ == "__main__":
    main()


### Figure 2 -- pre-committed evaluation criteria (Q1-Q5) and cross-cancer overlap

This is v4, which replaces v3 by using descriptive row labels ("PSI-testable events", "Primary aberrant events", etc.) in place of the internal M1'-M5' notation that was not defined in the manuscript text.

In [ ]:
"""
step3c_F2_meta_thresholds_v4.py
Paper 5A -- Figure 2, v4.

Same data and verdict logic as v3. Only change: the five pre-commitment
criteria are now labeled Q1-Q5 with plain descriptive text instead of the
internal M1'/M2'/M3'/M4/M5' notation, since that notation is never defined
in the manuscript or supplementary text and reads as an unresolved internal
artifact to a reviewer. Panel b title and footnote also de-jargoned to match.

No data, thresholds, or verdicts change. Reads same inputs, writes:
  Stage2C_corrected/figures/F2_meta_thresholds_v4.pdf + .png

USAGE: paste this file into one Colab cell and run.
"""

import os
import sys
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR  = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
OVERLAP_DIR  = os.path.join(STAGE_C_DIR, "M3prime_overlap")
M4_DIR       = os.path.join(STAGE_C_DIR, "M4_enrichment")
FIG_DIR      = os.path.join(STAGE_C_DIR, "figures")

CANCERS = ["LUAD", "BLCA", "UCEC"]

STYLE = {
    'figure.figsize':        (7.2, 4.6),
    'figure.dpi':            150,
    'savefig.dpi':           300,
    'font.family':           'sans-serif',
    'font.sans-serif':       ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size':             8.0,
    'axes.titlesize':        9.0,
    'axes.titleweight':      'bold',
    'axes.labelsize':        8.0,
    'xtick.labelsize':       7.0,
    'ytick.labelsize':       7.0,
    'legend.fontsize':       7.0,
    'axes.linewidth':        0.7,
    'xtick.major.width':     0.6,
    'ytick.major.width':     0.6,
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.grid':             False,
    'pdf.fonttype':          42,
    'ps.fonttype':           42,
}

COHORT_COLOR = {"LUAD": "#2E86AB", "BLCA": "#E07A5F", "UCEC": "#76A646"}
VERDICT_COLOR = {
    "PASS":       "#A9D8A0",
    "PARTIAL":    "#F2D17C",
    "BORDERLINE": "#F2D17C",
    "REFRAMED":   "#C8B6E2",
    "FAIL":       "#E89A8A",
}


def apply_style():
    for k, v in STYLE.items():
        mpl.rcParams[k] = v


def load_data():
    q1 = {"LUAD": 25314, "BLCA": 17448, "UCEC": 20890}

    q2 = {}
    for c in CANCERS:
        path = os.path.join(STAGE_C_DIR, f"{c}_corrected_results.parquet")
        if not os.path.exists(path):
            sys.exit(f"Missing parquet: {path}")
        df = pd.read_parquet(path)
        q2[c] = int((df['tier'] == 'primary').sum())

    pw_path = os.path.join(OVERLAP_DIR, "pairwise_event_overlap.csv")
    if not os.path.exists(pw_path):
        sys.exit(f"Missing CSV: {pw_path}")
    pw = pd.read_csv(pw_path)

    cc_path = os.path.join(OVERLAP_DIR, "three_way_shared_events.parquet")
    n_conserved_events = None
    if os.path.exists(cc_path):
        n_conserved_events = len(pd.read_parquet(cc_path))

    m4_path = os.path.join(M4_DIR, "m4_fisher_per_cancer.csv")
    q4_or, q4_p = {}, {}
    if os.path.exists(m4_path):
        m4 = pd.read_csv(m4_path)
        for _, row in m4.iterrows():
            label = row.get('cohort', row.get('cancer', None))
            if label is None:
                continue
            label = str(label).strip().upper()
            if label in CANCERS:
                q4_or[label] = float(row['OR']) if 'OR' in row else float(row.get('odds_ratio', np.nan))
                q4_p[label]  = float(row['p_value']) if 'p_value' in row else float(row.get('p', row.get('pvalue', np.nan)))

    return q1, q2, pw, n_conserved_events, q4_or, q4_p


def panel_a_grid(ax, q1, q2, pw, q4_or, q4_p):
    rows = ["Q1", "Q2", "Q3", "Q4", "Q5"]
    row_labels = [
        "PSI-testable events\n[20K, 200K]",
        "Primary aberrant events\nLUAD>=500; BLCA, UCEC>=300",
        "Cross-cancer pairwise overlap\n[5%, 30%]",
        "Host-gene enrichment\nOR>1.5, p<0.05",
        "Anti-MUC16 negative control\n0 Path-A false positives",
    ]
    cols = CANCERS

    def q1_verdict(val):
        if 20000 <= val <= 200000:
            return "PASS"
        if 10000 <= val < 20000:
            return "PARTIAL"
        return "FAIL"

    def q2_verdict(c, val):
        thr = 500 if c == "LUAD" else 300
        if c == "LUAD" and val < thr:
            return "BORDERLINE"
        return "PASS" if val >= thr else "FAIL"

    def cancer_q3_range(c):
        rel = pw[(pw['cancer_A'] == c) | (pw['cancer_B'] == c)]
        vals = []
        for _, row in rel.iterrows():
            if row['cancer_A'] == c:
                vals.append(row['pct_of_A'])
            else:
                vals.append(row['pct_of_B'])
        if not vals:
            return None, None
        return min(vals), max(vals)

    n_rows, n_cols = len(rows), len(cols)
    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(-0.5, n_rows - 0.5)
    ax.invert_yaxis()

    for j, c in enumerate(cols):
        for i, r in enumerate(rows):
            verdict = "PASS"; val_text = ""; footnote = ""
            if r == "Q1":
                v = q1[c]; verdict = q1_verdict(v); val_text = f"{v:,}"
            elif r == "Q2":
                v = q2[c]; verdict = q2_verdict(c, v); val_text = f"{v:,}"
                if verdict == "BORDERLINE":
                    footnote = "*"
            elif r == "Q3":
                lo, hi = cancer_q3_range(c)
                if lo is None:
                    val_text = "n/a"; verdict = "FAIL"
                else:
                    val_text = f"{lo:.1f}-{hi:.1f}%"; verdict = "REFRAMED"
            elif r == "Q4":
                if c in q4_or:
                    val_text = f"OR={q4_or[c]:.1f}"
                    p = q4_p.get(c, 1.0)
                    verdict = "PASS" if (q4_or[c] > 1.5 and p < 0.05) else "FAIL"
                else:
                    val_text = "n/a"; verdict = "FAIL"
            elif r == "Q5":
                val_text = "0 FPs"; verdict = "PASS"

            rect = mpl.patches.Rectangle((j - 0.45, i - 0.40), 0.90, 0.80,
                                         facecolor=VERDICT_COLOR[verdict],
                                         edgecolor='black', linewidth=0.7)
            ax.add_patch(rect)
            ax.text(j, i - 0.12, verdict + footnote,
                    ha='center', va='center', fontsize=7.0,
                    fontstyle='italic', color='black')
            ax.text(j, i + 0.16, val_text,
                    ha='center', va='center', fontsize=7.5,
                    fontweight='bold', color='black')

    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(cols, fontweight='bold', fontsize=8.5)
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(row_labels, fontsize=7.5)
    ax.tick_params(axis='both', which='both', length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title('a   Pre-committed evaluation criteria', loc='left')

    legend_items = [("PASS", "PASS"),
                    ("BORDERLINE", "BORDERLINE*"),
                    ("REFRAMED", "REFRAMED")]
    for k, (key, lbl) in enumerate(legend_items):
        ax.add_patch(mpl.patches.Rectangle(
            (-0.40 + k * 1.10, n_rows - 0.15), 0.18, 0.18,
            facecolor=VERDICT_COLOR[key], edgecolor='black', linewidth=0.5,
            transform=ax.transData, clip_on=False))
        ax.text(-0.18 + k * 1.10, n_rows - 0.06, lbl,
                ha='left', va='center', fontsize=6.5,
                transform=ax.transData, clip_on=False)

    ax.text(n_cols - 0.5, n_rows + 0.25,
            "* LUAD primary aberrant count = 464 vs 500 threshold; see Methods Deviation 2.",
            ha='right', va='center', fontsize=6.0, fontstyle='italic',
            transform=ax.transData, clip_on=False)


def panel_b_overlap(ax, pw, n_conserved_events):
    pairs, pct_a, pct_b = [], [], []
    for _, row in pw.iterrows():
        pairs.append(f"{row['cancer_A']}\nvs {row['cancer_B']}")
        pct_a.append(row['pct_of_A'])
        pct_b.append(row['pct_of_B'])

    x = np.arange(len(pairs))
    w = 0.36

    bars_a = ax.bar(x - w/2, pct_a, w, color='#5C8A9E',
                    edgecolor='black', linewidth=0.4, label='% of cancer A')
    bars_b = ax.bar(x + w/2, pct_b, w, color='#C97B5E',
                    edgecolor='black', linewidth=0.4, label='% of cancer B')

    for bar in list(bars_a) + list(bars_b):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1.2,
                f"{h:.1f}%", ha='center', va='bottom', fontsize=6.8)

    ax.axhline(30, color='black', linestyle='--', linewidth=0.8)
    x_mid = (x[0] + x[-1]) / 2
    ax.text(x_mid, 31.2, 'Precommit ceiling 30%',
            fontsize=6.5, va='bottom', ha='center', color='black',
            bbox=dict(boxstyle='round,pad=0.15',
                      facecolor='white', edgecolor='none', alpha=0.85))

    ax.axhline(5, color='black', linestyle=':', linewidth=0.7, alpha=0.6)
    ax.text(x[0] - w, 5.8, 'Floor 5%', fontsize=6.0,
            va='bottom', ha='left', color='black', alpha=0.7)

    ax.set_xticks(x)
    ax.set_xticklabels(pairs)
    ax.set_ylabel('Event-level pairwise overlap (%)')
    ax.set_ylim(0, max(max(pct_a), max(pct_b)) * 1.18)
    ax.set_title('b   Cross-cancer pairwise overlap', loc='left')

    leg = ax.legend(loc='upper left', frameon=False, fontsize=6,
                    handlelength=1.0, handletextpad=0.5,
                    bbox_to_anchor=(0.01, 0.99))
    ax.add_artist(leg)

    if n_conserved_events is not None:
        ax.text(0.01, 0.82,
                f"3-way conserved: {n_conserved_events} events\n"
                f"99.2% direction-consistent\n"
                f"100% bootstrap stable",
                transform=ax.transAxes, ha='left', va='top',
                fontsize=5.8, color='black',
                bbox=dict(boxstyle='round,pad=0.25',
                          facecolor='white',
                          edgecolor='black', linewidth=0.4))


def main():
    if not os.path.exists("/content/drive/MyDrive"):
        sys.exit("Drive not mounted. Run drive.mount('/content/drive') first.")
    if not os.path.exists(STAGE_C_DIR):
        sys.exit(f"Stage2C_corrected not found: {STAGE_C_DIR}")
    os.makedirs(FIG_DIR, exist_ok=True)

    apply_style()
    print(f"Output: {FIG_DIR}")
    print(f"Date:   {date.today().isoformat()}")
    print()
    print("Loading inputs ...")
    q1, q2, pw, n_cons, q4_or, q4_p = load_data()
    print(f"  Q1 (PSI-testable events): {q1}")
    print(f"  Q2 (primary aberrant events, from corrected parquets): {q2}")
    print(f"  Q3 pairwise rows: {len(pw)}")
    print(f"  Q3 three-way conserved core events: {n_cons}")
    print(f"  Q4 Fisher OR per cohort: {q4_or}")
    print()

    fig, axes = plt.subplots(1, 2, figsize=(7.6, 4.4),
                             gridspec_kw={'width_ratios': [1.05, 1.0]})
    panel_a_grid(axes[0], q1, q2, pw, q4_or, q4_p)
    panel_b_overlap(axes[1], pw, n_cons)

    plt.tight_layout(w_pad=2.2)

    pdf_path = os.path.join(FIG_DIR, "F2_meta_thresholds_v4.pdf")
    png_path = os.path.join(FIG_DIR, "F2_meta_thresholds_v4.png")
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, bbox_inches='tight', dpi=300)
    print(f"  Saved PDF: {pdf_path}")
    print(f"  Saved PNG: {png_path}")
    print()
    print("F2 v4 done (M1'-M5' notation removed from labels). Open the PNG, review, then send it back.")


if __name__ == "__main__":
    main()


### Figure 3 -- per-cancer volcano plots

In [ ]:
"""
step3c_F3_volcanoes_v1.py
Paper 5A -- Figure 3, corrected-values-only.

Three panels: LUAD, BLCA, UCEC volcano plots from the corrected beta-binomial
LRT. Signed dPSI (mean-based) on x; -log10(q_BB) on y. Coverage-pass events
that fail gates are grey low-alpha; events passing q<0.05 AND |dPSI|>=0.15
are cancer-colored. Primary-tier (3C-validated) events get a black edge ring.
Hallmark genes that surface in primary tier are labeled.

Saves: Stage2C_corrected/figures/F3_volcanoes_v1.pdf + .png

Reads only:
  Stage2C_corrected/{LUAD,BLCA,UCEC}_corrected_results.parquet

USAGE IN COLAB:
  Paste this file's contents into one cell and run. Run the Drive-mount
  block first if Drive is not yet mounted.
  Runtime: ~30-60s depending on adjustText availability.
"""

import os
import sys
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    from adjustText import adjust_text
    HAS_ADJUST = True
except ImportError:
    HAS_ADJUST = False


# Config
PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR  = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
FIG_DIR      = os.path.join(STAGE_C_DIR, "figures")

CANCERS = ["LUAD", "BLCA", "UCEC"]

# Per-cancer sample sizes (T vs N) for subtitles
SAMPLE_SIZES = {
    "LUAD": (542, 655),
    "BLCA": (414, 21),
    "UCEC": (554, 159),
}

# Hallmark genes locked in cell 107 (28 genes)
HALLMARK_GENES = {
    "FAS", "CD44", "FGFR1", "FGFR2", "FGFR3", "BCL2L1", "MDM4", "MDM2", "VEGFA",
    "KLF6", "PKM", "MET", "EGFR", "AR", "TP53", "RAC1", "RON", "SYK", "TIA1",
    "NUMB", "CTNNB1", "BIN1", "CASP9", "CASP8", "TRAF3", "ESR1", "BRCA1", "PIK3CA",
}

# Gate thresholds (locked)
Q_THR     = 0.05
DPSI_THR  = 0.15


# Style
STYLE = {
    'figure.dpi':            150,
    'savefig.dpi':           300,
    'font.family':           'sans-serif',
    'font.sans-serif':       ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size':             8.0,
    'axes.titlesize':        9.0,
    'axes.titleweight':      'bold',
    'axes.labelsize':        8.0,
    'xtick.labelsize':       7.0,
    'ytick.labelsize':       7.0,
    'legend.fontsize':       6.8,
    'axes.linewidth':        0.7,
    'xtick.major.width':     0.6,
    'ytick.major.width':     0.6,
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.grid':             False,
    'pdf.fonttype':          42,
    'ps.fonttype':           42,
}

COHORT_COLOR = {
    "LUAD":  "#2E86AB",
    "BLCA":  "#E07A5F",
    "UCEC":  "#76A646",
}


def apply_style():
    for k, v in STYLE.items():
        mpl.rcParams[k] = v


# Data loading + prep
def load_one(c):
    path = os.path.join(STAGE_C_DIR, f"{c}_corrected_results.parquet")
    if not os.path.exists(path):
        sys.exit(f"Missing parquet: {path}")
    df = pd.read_parquet(path)
    # Keep only coverage-pass events for the volcano (others have no test
    # result we can plot). This matches what the betabin actually tested.
    cov = df['coverage_pass'].fillna(False).astype(bool)
    df = df[cov].copy()

    # Required columns: BB q-value (Stage 2C corrected pipeline writes 'q_value'
    # from the BB LRT + BH-FDR), dpsi (mean-based, signed), tier, gene_symbol.
    # Fall back across naming variants defensively.
    qcol = None
    for cand in ['q_value', 'bb_qvalue', 'qvalue_bb', 'q_bb', 'qvalue']:
        if cand in df.columns:
            qcol = cand; break
    if qcol is None:
        sys.exit(f"[{c}] cannot find BB q-value column. Got: {list(df.columns)}")

    dpcol = None
    for cand in ['delta_psi', 'dpsi', 'mean_dpsi']:
        if cand in df.columns:
            dpcol = cand; break
    if dpcol is None:
        sys.exit(f"[{c}] cannot find signed dPSI column. Got: {list(df.columns)}")

    gsym = None
    for cand in ['gene_symbol', 'gene_name', 'symbol', 'hgnc_symbol']:
        if cand in df.columns:
            gsym = cand; break

    df = df.rename(columns={qcol: 'q', dpcol: 'dpsi'})
    if gsym and gsym != 'gene_symbol':
        df = df.rename(columns={gsym: 'gene_symbol'})
    elif gsym is None:
        df['gene_symbol'] = ''

    # Clean
    df['q'] = df['q'].astype(float).clip(lower=1e-300)  # avoid log(0)
    df['neglog10q'] = -np.log10(df['q'])
    df['dpsi'] = df['dpsi'].astype(float)
    df['gene_symbol'] = df['gene_symbol'].fillna('').astype(str)

    return df


def label_hallmark_in_primary(df_primary):
    """Pick one row per hallmark gene to label: the row with max |dpsi|."""
    if df_primary.empty:
        return df_primary.iloc[0:0]
    hits = df_primary[df_primary['gene_symbol'].isin(HALLMARK_GENES)].copy()
    if hits.empty:
        return hits
    hits['abs_dpsi'] = hits['dpsi'].abs()
    # one row per gene = the strongest by |dpsi|
    hits = (hits.sort_values('abs_dpsi', ascending=False)
                .drop_duplicates('gene_symbol')
                .reset_index(drop=True))
    return hits


# Per-panel plotter
def plot_volcano(ax, c, df, y_cap):
    color = COHORT_COLOR[c]

    # Slice events into visual categories
    cov_total   = len(df)
    passes_q    = df['q'] < Q_THR
    passes_dp   = df['dpsi'].abs() >= DPSI_THR
    is_primary  = df['tier'] == 'primary'
    is_signif   = passes_q & passes_dp

    # Capped y for display only
    y = df['neglog10q'].clip(upper=y_cap)
    n_at_cap = int((df['neglog10q'] > y_cap).sum())

    # Background grey (everything not passing both gates)
    grey_idx = ~is_signif
    ax.scatter(df.loc[grey_idx, 'dpsi'], y[grey_idx],
               s=4, c='#CCCCCC', alpha=0.45,
               edgecolors='none', rasterized=True)

    # Significant non-primary (secondary or scn-discordant)
    sig_nonpri = is_signif & ~is_primary
    ax.scatter(df.loc[sig_nonpri, 'dpsi'], y[sig_nonpri],
               s=6, c=color, alpha=0.55,
               edgecolors='none', rasterized=True)

    # Primary tier on top with black ring
    pri = is_primary
    ax.scatter(df.loc[pri, 'dpsi'], y[pri],
               s=10, c=color, alpha=0.95,
               edgecolors='black', linewidths=0.25,
               rasterized=True)

    # Gate lines
    ax.axhline(-np.log10(Q_THR), color='black', linestyle=':',
               linewidth=0.5, alpha=0.5)
    ax.axvline( DPSI_THR, color='black', linestyle=':',
               linewidth=0.5, alpha=0.5)
    ax.axvline(-DPSI_THR, color='black', linestyle=':',
               linewidth=0.5, alpha=0.5)
    ax.axvline(0, color='black', linewidth=0.4, alpha=0.4)

    # Hallmark labels (drawn from primary tier only) -- vertical orientation
    # to avoid overlap when many hallmarks pile up at the y-cap line.
    df_pri = df[pri].copy()
    df_pri['neglog10q_capped'] = df_pri['neglog10q'].clip(upper=y_cap)
    hits = label_hallmark_in_primary(df_pri)

    # Sort by x so labels read in dPSI order along the cap; helps visual scan
    hits = hits.sort_values('dpsi').reset_index(drop=True)

    # Vertical offset above each marker so the text starts above the point
    # and reads upward. Tweak label_offset_frac if labels still touch markers.
    label_offset_frac = 0.025  # fraction of y_cap above each marker
    label_offset = label_offset_frac * y_cap

    for _, row in hits.iterrows():
        # Purple ring marker
        ax.scatter([row['dpsi']], [row['neglog10q_capped']],
                   s=22, facecolor='#C5A3D8', edgecolor='black',
                   linewidth=0.5, zorder=5)
        # Vertical label, anchored bottom so it grows upward from the marker
        ax.text(row['dpsi'],
                row['neglog10q_capped'] + label_offset,
                row['gene_symbol'],
                fontsize=6.5, fontstyle='italic', zorder=6,
                color='black',
                rotation=90,
                rotation_mode='anchor',
                ha='center', va='bottom')

    # Title with sample sizes
    nT, nN = SAMPLE_SIZES[c]
    ax.set_title(f"{c}  (n={nT} vs {nN})",
                 loc='center', color=color, fontweight='bold')

    # Axis labels
    ax.set_xlabel(r'$\Delta$PSI (tumor - normal)')
    ax.set_ylabel(r'$-\log_{10}(q)$')
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(0, y_cap * 1.04)

    # Counts inset
    n_primary   = int(pri.sum())
    n_secondary = int(((df['tier'] == 'secondary')).sum())
    inset = f"primary: {n_primary:,}\nsecondary: {n_secondary:,}"
    if n_at_cap > 0:
        inset += f"\n({n_at_cap:,} at y-cap)"
    ax.text(0.95, 0.90, inset, transform=ax.transAxes,
            ha='right', va='top', fontsize=6.5,
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='white', edgecolor='black',
                      linewidth=0.4))


# Main
def main():
    if not os.path.exists("/content/drive/MyDrive"):
        sys.exit("Drive not mounted. Run drive.mount('/content/drive') first.")
    if not os.path.exists(STAGE_C_DIR):
        sys.exit(f"Stage2C_corrected not found: {STAGE_C_DIR}")
    os.makedirs(FIG_DIR, exist_ok=True)

    apply_style()
    print(f"Output: {FIG_DIR}")
    print(f"Date:   {date.today().isoformat()}")
    print(f"adjustText available: {HAS_ADJUST}")
    print()
    print("Loading corrected results ...")
    data = {c: load_one(c) for c in CANCERS}
    for c, df in data.items():
        n_pri = int((df['tier'] == 'primary').sum())
        n_sec = int((df['tier'] == 'secondary').sum())
        max_y = df['neglog10q'].replace(np.inf, np.nan).max()
        p99   = df['neglog10q'].replace(np.inf, np.nan).quantile(0.99)
        print(f"  [{c}] coverage-pass={len(df):,}  primary={n_pri}  "
              f"secondary={n_sec}  max(-log10q)={max_y:.1f}  "
              f"p99(-log10q)={p99:.1f}")

    # Shared y-cap across panels: max of per-cancer p99, capped at 50
    p99s = [data[c]['neglog10q'].replace(np.inf, np.nan).quantile(0.99)
            for c in CANCERS]
    y_cap = float(min(50.0, max(p99s) * 1.05))
    y_cap = max(y_cap, 10.0)  # never less than 10 for readability
    print(f"\n  Shared y-cap (display only): {y_cap:.1f}")

    fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.6), sharey=True)
    for ax, c in zip(axes, CANCERS):
        plot_volcano(ax, c, data[c], y_cap)

    plt.tight_layout(w_pad=1.6)

    pdf_path = os.path.join(FIG_DIR, "F3_volcanoes_v1.pdf")
    png_path = os.path.join(FIG_DIR, "F3_volcanoes_v1.png")
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, bbox_inches='tight', dpi=300)
    print(f"\n  Saved PDF: {pdf_path}")
    print(f"  Saved PNG: {png_path}")
    print()
    if not HAS_ADJUST:
        print("  NOTE: adjustText not installed; labels may overlap. "
              "Install with: !pip install adjustText -q")
    print("F3 v1 done. Open the PNG, review, then send the image back.")


if __name__ == "__main__":
    main()


### Figure 4 -- three-way Venn and hallmark gene coverage

In [ ]:
"""
step3c_F4_overlap_hallmarks_v1.py
Paper 5A - Figure 4, corrected-values-only.

Two panels:

  Panel a: 3-circle Venn diagram of primary-tier aberrant cassette events
           across LUAD, BLCA, UCEC. Region counts computed from the
           corrected parquets at event_id level. Annotation below the Venn
           reports union size, 3-way conserved count, and directional
           consistency.

  Panel b: Hallmark gene presence grid. Rows are hallmark cancer-splicing
           genes that surface in primary tier in at least one cancer.
           Columns are cancers. Filled cells = primary-tier hit, with the
           strongest signed dPSI for that gene in that cancer printed in
           the cell. Empty cells = not in primary tier. Rows ordered by
           total cancers hit (3 -> 2 -> 1), then alphabetically.

Saves: Stage2C_corrected/figures/F4_overlap_hallmarks_v1.pdf + .png

Reads only:
  Stage2C_corrected/{LUAD,BLCA,UCEC}_corrected_results.parquet
  Stage2C_corrected/M3prime_overlap/three_way_shared_events.parquet

USAGE IN COLAB:
  Mount Drive (and optionally !pip install matplotlib-venn -q), then paste
  this file into one cell and run.
  Runtime: <15s.
"""

import os
import sys
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    from matplotlib_venn import venn3, venn3_circles
    HAS_VENN = True
except ImportError:
    HAS_VENN = False


# Config
PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR  = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
OVERLAP_DIR  = os.path.join(STAGE_C_DIR, "M3prime_overlap")
FIG_DIR      = os.path.join(STAGE_C_DIR, "figures")

CANCERS = ["LUAD", "BLCA", "UCEC"]

# Hallmark cancer-splicing genes (28-gene set locked in cell 107)
HALLMARK_GENES = [
    "FAS", "CD44", "FGFR1", "FGFR2", "FGFR3", "BCL2L1", "MDM4", "MDM2", "VEGFA",
    "KLF6", "PKM", "MET", "EGFR", "AR", "TP53", "RAC1", "RON", "SYK", "TIA1",
    "NUMB", "CTNNB1", "BIN1", "CASP9", "CASP8", "TRAF3", "ESR1", "BRCA1", "PIK3CA",
]
HALLMARK_SET = set(HALLMARK_GENES)


# Style
STYLE = {
    'figure.dpi':            150,
    'savefig.dpi':           300,
    'font.family':           'sans-serif',
    'font.sans-serif':       ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size':             8.0,
    'axes.titlesize':        9.0,
    'axes.titleweight':      'bold',
    'axes.labelsize':        8.0,
    'xtick.labelsize':       7.0,
    'ytick.labelsize':       7.0,
    'legend.fontsize':       6.8,
    'axes.linewidth':        0.7,
    'xtick.major.width':     0.6,
    'ytick.major.width':     0.6,
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.grid':             False,
    'pdf.fonttype':          42,
    'ps.fonttype':           42,
}

COHORT_COLOR = {
    "LUAD": "#2E86AB",
    "BLCA": "#E07A5F",
    "UCEC": "#76A646",
}


def apply_style():
    for k, v in STYLE.items():
        mpl.rcParams[k] = v


# Data loading
def load_one(c):
    path = os.path.join(STAGE_C_DIR, f"{c}_corrected_results.parquet")
    if not os.path.exists(path):
        sys.exit(f"Missing parquet: {path}")
    df = pd.read_parquet(path)
    # Defensive renames to match the actual column schema seen in F3:
    if 'gene_name' in df.columns and 'gene_symbol' not in df.columns:
        df = df.rename(columns={'gene_name': 'gene_symbol'})
    pri = df[df['tier'] == 'primary'].copy()
    return pri


def load_threeway():
    path = os.path.join(OVERLAP_DIR, "three_way_shared_events.parquet")
    if not os.path.exists(path):
        return None
    return pd.read_parquet(path)


# Panel a - Venn
def panel_a_venn(ax, primary_sets, three_way_df):
    """Venn at event_id level across the three primary sets."""
    if not HAS_VENN:
        ax.text(0.5, 0.5,
                "matplotlib_venn not installed.\n"
                "Run:  !pip install matplotlib-venn -q\n"
                "then rerun this script.",
                ha='center', va='center', fontsize=8,
                transform=ax.transAxes,
                bbox=dict(boxstyle='round,pad=0.4',
                          facecolor='#FFF2CC', edgecolor='black'))
        ax.set_axis_off()
        ax.set_title('a   Cross-cancer overlap of primary aberrant events',
                     loc='left')
        return

    luad_ev = set(primary_sets["LUAD"]['event_id'].astype(str))
    blca_ev = set(primary_sets["BLCA"]['event_id'].astype(str))
    ucec_ev = set(primary_sets["UCEC"]['event_id'].astype(str))

    # Region counts
    only_L  = len(luad_ev - blca_ev - ucec_ev)
    only_B  = len(blca_ev - luad_ev - ucec_ev)
    only_U  = len(ucec_ev - luad_ev - blca_ev)
    LB      = len((luad_ev & blca_ev) - ucec_ev)
    LU      = len((luad_ev & ucec_ev) - blca_ev)
    BU      = len((blca_ev & ucec_ev) - luad_ev)
    LBU     = len(luad_ev & blca_ev & ucec_ev)
    union   = len(luad_ev | blca_ev | ucec_ev)

    print(f"  Venn regions (events):")
    print(f"    LUAD only={only_L}, BLCA only={only_B}, UCEC only={only_U}")
    print(f"    LUAD&BLCA only={LB}, LUAD&UCEC only={LU}, BLCA&UCEC only={BU}")
    print(f"    LUAD&BLCA&UCEC={LBU}")
    print(f"    union={union}")

    v = venn3(subsets=(only_L, only_B, LB, only_U, LU, BU, LBU),
              set_labels=('LUAD', 'BLCA', 'UCEC'),
              ax=ax,
              set_colors=(COHORT_COLOR['LUAD'],
                          COHORT_COLOR['BLCA'],
                          COHORT_COLOR['UCEC']),
              alpha=0.55)

    # Style adjustments
    for pid in ('100', '010', '001', '110', '101', '011', '111'):
        patch = v.get_patch_by_id(pid)
        if patch:
            patch.set_edgecolor('black')
            patch.set_linewidth(0.6)
    for label_id, color in zip(('A', 'B', 'C'),
                               (COHORT_COLOR['LUAD'],
                                COHORT_COLOR['BLCA'],
                                COHORT_COLOR['UCEC'])):
        lbl = v.get_label_by_id(label_id)
        if lbl:
            lbl.set_color(color)
            lbl.set_fontweight('bold')
            lbl.set_fontsize(9)
    for pid in ('100', '010', '001', '110', '101', '011', '111'):
        lbl = v.get_label_by_id(pid)
        if lbl:
            lbl.set_fontsize(7.5)

    ax.set_title('a   Cross-cancer overlap of primary aberrant events',
                 loc='left')

    # Annotation under the Venn
    n_genes_conserved = None
    if three_way_df is not None and len(three_way_df) > 0:
        gcol = None
        for cand in ['gene_symbol', 'gene_name']:
            if cand in three_way_df.columns:
                gcol = cand; break
        if gcol is not None:
            n_genes_conserved = three_way_df[gcol].dropna().nunique()

    annot = f"Union: {union:,} events  |  3-way conserved: {LBU} events"
    if n_genes_conserved is not None:
        annot += f" / {n_genes_conserved} genes"
    annot += "\n99.2% directionally consistent  |  100% bootstrap sign-stable"

    ax.text(0.5, -0.08, annot,
            transform=ax.transAxes, ha='center', va='top',
            fontsize=7.0, fontstyle='italic')


# Panel b - Hallmark presence grid
def panel_b_hallmark_grid(ax, primary_sets):
    """For each hallmark gene, mark which cancers have it in primary tier and
    print the strongest signed dPSI per cell."""

    # Build per-gene per-cancer max-|dpsi| (signed) lookup
    presence = {}
    for c, df in primary_sets.items():
        sub = df[df['gene_symbol'].isin(HALLMARK_SET)].copy()
        if sub.empty:
            continue
        sub['abs_dpsi'] = sub['delta_psi'].abs()
        best = (sub.sort_values('abs_dpsi', ascending=False)
                   .drop_duplicates('gene_symbol')
                   [['gene_symbol', 'delta_psi']])
        for _, row in best.iterrows():
            g = row['gene_symbol']
            presence.setdefault(g, {})[c] = float(row['delta_psi'])

    if not presence:
        ax.text(0.5, 0.5, "No hallmark genes in primary tier.",
                ha='center', va='center', fontsize=9,
                transform=ax.transAxes)
        ax.set_axis_off()
        ax.set_title('b   Hallmark cancer-splicing genes per cancer', loc='left')
        return

    # Order rows: 3-cancer hits first, then 2, then 1, alphabetical within
    def hit_count(g):
        return len(presence[g])
    genes_sorted = sorted(presence.keys(),
                          key=lambda g: (-hit_count(g), g))

    n_rows = len(genes_sorted)
    n_cols = len(CANCERS)

    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(-0.5, n_rows - 0.5)
    ax.invert_yaxis()

    for i, gene in enumerate(genes_sorted):
        for j, c in enumerate(CANCERS):
            dpsi = presence[gene].get(c)
            if dpsi is None:
                rect = mpl.patches.Rectangle(
                    (j - 0.42, i - 0.38), 0.84, 0.76,
                    facecolor='#EEEEEE', edgecolor='black', linewidth=0.4)
                ax.add_patch(rect)
            else:
                rect = mpl.patches.Rectangle(
                    (j - 0.42, i - 0.38), 0.84, 0.76,
                    facecolor=COHORT_COLOR[c], alpha=0.85,
                    edgecolor='black', linewidth=0.5)
                ax.add_patch(rect)
                sign = '+' if dpsi >= 0 else ''
                ax.text(j, i, f"{sign}{dpsi:.2f}",
                        ha='center', va='center',
                        fontsize=7.0, color='white',
                        fontweight='bold')

    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(CANCERS, fontweight='bold', fontsize=8.5)
    for tick, c in zip(ax.get_xticklabels(), CANCERS):
        tick.set_color(COHORT_COLOR[c])
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(genes_sorted, fontsize=7.5, fontstyle='italic')
    ax.tick_params(axis='both', which='both', length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title('b   Hallmark cancer-splicing genes per cancer', loc='left')

    # Footer summary
    n3 = sum(1 for g in presence if len(presence[g]) == 3)
    n2 = sum(1 for g in presence if len(presence[g]) == 2)
    n1 = sum(1 for g in presence if len(presence[g]) == 1)
    ntot = len(presence)
    nset = len(HALLMARK_GENES)
    footer = (f"{ntot} of {nset} hallmark genes hit  |  "
              f"3 cancers: {n3}  |  2 cancers: {n2}  |  1 cancer: {n1}")
    ax.text(0.5, -0.05, footer,
            transform=ax.transAxes, ha='center', va='top',
            fontsize=7.0, fontstyle='italic')

    # Numeric cell convention note
    ax.text(0.5, -0.10,
            "Cells show strongest signed dPSI per gene (tumor - normal).",
            transform=ax.transAxes, ha='center', va='top',
            fontsize=6.2, color='#555555')


# Main
def main():
    if not os.path.exists("/content/drive/MyDrive"):
        sys.exit("Drive not mounted. Run drive.mount('/content/drive') first.")
    if not os.path.exists(STAGE_C_DIR):
        sys.exit(f"Stage2C_corrected not found: {STAGE_C_DIR}")
    os.makedirs(FIG_DIR, exist_ok=True)

    apply_style()
    print(f"Output: {FIG_DIR}")
    print(f"Date:   {date.today().isoformat()}")
    print(f"matplotlib_venn available: {HAS_VENN}")
    print()
    print("Loading corrected primary-tier sets ...")
    primary_sets = {}
    for c in CANCERS:
        pri = load_one(c)
        primary_sets[c] = pri
        print(f"  [{c}] primary={len(pri):,}  "
              f"unique genes={pri['gene_symbol'].nunique():,}")

    three_way_df = load_threeway()
    if three_way_df is not None:
        print(f"  3-way conserved events on disk: {len(three_way_df)}")
    else:
        print("  Three-way parquet not found (Venn will still compute on the fly).")
    print()

    # Height scales with hallmark grid row count; we don't know it until after
    # the panel runs, so pick a reasonable two-panel canvas.
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 5.4),
                             gridspec_kw={'width_ratios': [1.15, 1.0]})
    panel_a_venn(axes[0], primary_sets, three_way_df)
    panel_b_hallmark_grid(axes[1], primary_sets)

    plt.tight_layout(w_pad=2.6)

    pdf_path = os.path.join(FIG_DIR, "F4_overlap_hallmarks_v1.pdf")
    png_path = os.path.join(FIG_DIR, "F4_overlap_hallmarks_v1.png")
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, bbox_inches='tight', dpi=300)
    print(f"  Saved PDF: {pdf_path}")
    print(f"  Saved PNG: {png_path}")
    print()
    if not HAS_VENN:
        print("  NOTE: install matplotlib-venn for Panel a to render:")
        print("    !pip install matplotlib-venn -q")
    print("F4 v1 done. Open the PNG, review, then send the image back.")


if __name__ == "__main__":
    main()


### Figure 5 -- hallmark gene per-sample PSI case studies

In [ ]:
"""
step3c_F5_hallmark_case_studies_v1.py
Paper 5A - Figure 5, corrected-values-only.

Six-panel grid of hallmark cancer-splicing genes showing per-sample PSI
distributions in tumor vs normal, for each cancer where the gene surfaces
in primary tier.

Panels are picked from the corrected primary-tier sets across the three
cancers. For each candidate gene, the script picks the event_id with the
strongest |dPSI| per cancer and pulls the per-sample PSI vector from the
cell-60 npz files. Mean PSI lines are drawn per group (matching the
corrected pipeline convention of mean, not median).

Saves: Stage2C_corrected/figures/F5_hallmark_case_studies_v1.pdf + .png

Reads:
  Stage2C_corrected/{LUAD,BLCA,UCEC}_corrected_results.parquet
  data/processed/suppa/suppa_events_parsed.parquet      (event ordering)
  data/processed/psi/{LUAD,BLCA,UCEC}_tumor_psi.npz     (per-sample PSI)
  data/processed/psi/{LUNG,BLADDER,UTERUS}_normal_psi.npz

USAGE IN COLAB:
  Paste this file into one cell and run after mounting Drive.
  Runtime: ~30-40s.

NOTE on input paths:
  npz filenames vary across pipeline iterations. The script tries several
  plausible filename patterns and reports which it loaded. If none match,
  it prints the candidate filenames and exits cleanly so we can patch.
"""

import os
import sys
import glob
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


# Config
PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR  = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
PSI_DIR      = os.path.join(PROJECT_ROOT, "data/processed/psi")
SUPPA_DIR    = PSI_DIR  # suppa_events_parsed.parquet lives in data/processed/psi/
FIG_DIR      = os.path.join(STAGE_C_DIR, "figures")

CANCERS = ["LUAD", "BLCA", "UCEC"]
NORMALS = {"LUAD": "LUNG", "BLCA": "BLADDER", "UCEC": "UTERUS"}

# Candidate hallmarks to consider for the 6 panels. The actual 6 picked
# is determined from the data: prefer genes primary in >=2 cancers, then by
# total |dPSI|. This keeps the selection data-driven and reproducible.
HALLMARK_CANDIDATES = [
    "CD44", "NUMB", "FAS", "FGFR2", "FGFR3", "VEGFA", "MDM4", "SRSF2",
    "FGFR1", "SYK", "BIN1", "RON", "CASP8", "BCL2L1",
]
N_PANELS = 6


# Style
STYLE = {
    'figure.dpi':            150,
    'savefig.dpi':           300,
    'font.family':           'sans-serif',
    'font.sans-serif':       ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size':             8.0,
    'axes.titlesize':        9.0,
    'axes.titleweight':      'bold',
    'axes.labelsize':        8.0,
    'xtick.labelsize':       7.0,
    'ytick.labelsize':       7.0,
    'legend.fontsize':       6.8,
    'axes.linewidth':        0.7,
    'xtick.major.width':     0.6,
    'ytick.major.width':     0.6,
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'axes.grid':             False,
    'pdf.fonttype':          42,
    'ps.fonttype':           42,
}

COHORT_COLOR = {
    "LUAD": "#2E86AB",
    "BLCA": "#E07A5F",
    "UCEC": "#76A646",
}


def apply_style():
    for k, v in STYLE.items():
        mpl.rcParams[k] = v


# Data loading
def find_npz(label, kind):
    """kind in {'tumor', 'normal'}. Tries common filename patterns."""
    patterns = [
        f"{label}_{kind}_psi.npz",
        f"{label}_psi.npz",
        f"{label.lower()}_{kind}_psi.npz",
        f"{label.lower()}_psi.npz",
        f"{label}_{kind}.npz",
    ]
    for p in patterns:
        cand = os.path.join(PSI_DIR, p)
        if os.path.exists(cand):
            return cand
    # Fallback: glob for any file starting with this label
    matches = glob.glob(os.path.join(PSI_DIR, f"{label}*psi*.npz"))
    if matches:
        return matches[0]
    return None


def load_psi_matrix(label, kind):
    """Returns a dict with at least 'psi' (n_events x n_samples)."""
    path = find_npz(label, kind)
    if path is None:
        print(f"  [{label}/{kind}] npz NOT FOUND in {PSI_DIR}")
        return None
    data = np.load(path, allow_pickle=True)
    keys = list(data.keys())
    print(f"  [{label}/{kind}] loaded {os.path.basename(path)}  keys={keys}")
    if 'psi' not in keys:
        print(f"    No 'psi' key. Available: {keys}")
        return None
    psi = data['psi']  # shape (n_events, n_samples)
    return psi


def load_suppa_events():
    path = os.path.join(SUPPA_DIR, "suppa_events_parsed.parquet")
    if not os.path.exists(path):
        sys.exit(f"Missing SUPPA events parquet: {path}")
    df = pd.read_parquet(path)
    print(f"  SUPPA events: {len(df):,} rows; columns={list(df.columns)[:8]} ...")
    return df


def load_corrected_primary(c):
    path = os.path.join(STAGE_C_DIR, f"{c}_corrected_results.parquet")
    df = pd.read_parquet(path)
    if 'gene_name' in df.columns and 'gene_symbol' not in df.columns:
        df = df.rename(columns={'gene_name': 'gene_symbol'})
    return df[df['tier'] == 'primary'].copy()


# Gene + event selection
def pick_genes_and_events(primary_sets):
    """For each candidate hallmark gene, find which cancers it's primary in
    and the event_id with strongest |dPSI| in each. Then pick top N_PANELS
    by total |dPSI| across cancers, breaking ties by number of cancers hit."""
    info = {}  # gene -> {cancer: {event_id, delta_psi, abs_dpsi}}
    for g in HALLMARK_CANDIDATES:
        for c in CANCERS:
            df = primary_sets[c]
            sub = df[df['gene_symbol'] == g].copy()
            if sub.empty:
                continue
            sub['abs_dpsi'] = sub['delta_psi'].abs()
            best = sub.sort_values('abs_dpsi', ascending=False).iloc[0]
            info.setdefault(g, {})[c] = {
                'event_id':  str(best['event_id']),
                'delta_psi': float(best['delta_psi']),
                'abs_dpsi':  float(best['abs_dpsi']),
            }

    if not info:
        sys.exit("No hallmark candidates found in any primary set.")

    # Rank genes: more cancers hit > total |dPSI|
    def score(g):
        n_cancers = len(info[g])
        total_abs = sum(d['abs_dpsi'] for d in info[g].values())
        return (n_cancers, total_abs)
    genes_ranked = sorted(info.keys(), key=score, reverse=True)
    picked = genes_ranked[:N_PANELS]

    print(f"\n  Hallmark gene selection (top {N_PANELS}):")
    for g in picked:
        per_c = info[g]
        cc = ', '.join(f"{c}:{d['delta_psi']:+.2f}"
                       for c, d in per_c.items())
        print(f"    {g}  [{len(per_c)} cancer(s)]  {cc}")
    print()

    return {g: info[g] for g in picked}


# PSI vector retrieval
def build_event_index(suppa_df):
    """event_id -> row position in SUPPA events. Try multiple id columns."""
    for col in ['event_id', 'event', 'event_name', 'id']:
        if col in suppa_df.columns:
            id_col = col; break
    else:
        sys.exit(f"No event id column in SUPPA parquet. Got: {list(suppa_df.columns)}")
    print(f"  Event id column in SUPPA: {id_col}")
    return {str(eid): i for i, eid in enumerate(suppa_df[id_col])}, id_col


def get_psi_vector(psi_matrix, event_idx):
    if psi_matrix is None or event_idx is None:
        return None
    if event_idx >= psi_matrix.shape[0]:
        return None
    v = psi_matrix[event_idx, :].astype(float)
    # Drop NaN samples for strip plotting
    return v[~np.isnan(v)]


# Per-panel plotter
def plot_gene_panel(ax, gene, gene_info, event_id_to_idx, psi_tumor, psi_normal):
    """Strip plot of per-sample PSI for tumor and normal in each cancer where
    the gene is primary. mean lines drawn per group, light line connecting
    T-mean to N-mean within each cancer."""
    cancers_present = [c for c in CANCERS if c in gene_info]

    # Columns: [T_LUAD, N_LUAD, T_BLCA, N_BLCA, ...] in cancers_present order
    x_positions = []
    x_labels    = []
    pair_centers = []
    deltas = {}

    col_x = 0
    for c in cancers_present:
        eid = gene_info[c]['event_id']
        idx = event_id_to_idx.get(eid)
        if idx is None:
            print(f"    [{gene}/{c}] event_id {eid} not found in SUPPA index")
            continue

        t_vec = get_psi_vector(psi_tumor[c], idx)
        n_vec = get_psi_vector(psi_normal[c], idx)
        if t_vec is None or n_vec is None or len(t_vec) == 0 or len(n_vec) == 0:
            print(f"    [{gene}/{c}] empty PSI vectors at event_id {eid}")
            continue

        color = COHORT_COLOR[c]

        # Tumor strip
        xt = col_x + np.random.uniform(-0.18, 0.18, size=len(t_vec))
        ax.scatter(xt, t_vec, s=4, c=color, alpha=0.30,
                   edgecolors='none', rasterized=True)
        # Normal strip
        xn = (col_x + 1) + np.random.uniform(-0.18, 0.18, size=len(n_vec))
        ax.scatter(xn, n_vec, s=4, c=color, alpha=0.30,
                   edgecolors='none', rasterized=True)

        # Mean lines
        mt = float(np.mean(t_vec))
        mn = float(np.mean(n_vec))
        ax.hlines(mt, col_x - 0.30, col_x + 0.30,
                  colors='black', linewidth=1.6, zorder=5)
        ax.hlines(mn, col_x + 1 - 0.30, col_x + 1 + 0.30,
                  colors='black', linewidth=1.6, zorder=5)

        # Connector
        ax.plot([col_x, col_x + 1], [mt, mn],
                color='grey', linewidth=0.6, alpha=0.7, zorder=4)

        x_positions.extend([col_x, col_x + 1])
        x_labels.extend(['T', 'N'])
        pair_centers.append((col_x + 0.5, c, mt - mn))
        deltas[c] = mt - mn

        col_x += 2.6  # gap between cancer groups

    # Strip-level styling
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels, fontsize=7.0)
    ax.set_ylim(-0.04, 1.04)
    ax.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
    ax.set_ylabel('PSI', fontsize=7.5)

    # Cancer labels under each pair
    for center, c, _ in pair_centers:
        ax.text(center, -0.18, c,
                ha='center', va='top', fontsize=8.0,
                color=COHORT_COLOR[c], fontweight='bold',
                transform=ax.get_xaxis_transform())

    # Per-cancer dPSI text above each pair
    for center, c, dpsi in pair_centers:
        sign = '+' if dpsi >= 0 else ''
        ax.text(center, 1.06, f"Δ={sign}{dpsi:.2f}",
                ha='center', va='bottom', fontsize=6.8,
                color='black')

    # Title: gene name only, italic, centered
    ax.set_title(gene, loc='center', fontstyle='italic', fontweight='bold',
                 pad=18)

    # Tidy: shrink x range
    if x_positions:
        ax.set_xlim(min(x_positions) - 0.6, max(x_positions) + 0.6)


# Main
def main():
    if not os.path.exists("/content/drive/MyDrive"):
        sys.exit("Drive not mounted. Run drive.mount('/content/drive') first.")
    if not os.path.exists(STAGE_C_DIR):
        sys.exit(f"Stage2C_corrected not found: {STAGE_C_DIR}")
    os.makedirs(FIG_DIR, exist_ok=True)

    np.random.seed(42)  # jitter reproducibility
    apply_style()
    print(f"Output: {FIG_DIR}")
    print(f"Date:   {date.today().isoformat()}")
    print()

    # Primary tier per cancer
    print("Loading corrected primary-tier sets ...")
    primary_sets = {c: load_corrected_primary(c) for c in CANCERS}
    for c, df in primary_sets.items():
        print(f"  [{c}] primary={len(df):,}")
    print()

    # Pick six hallmark genes from the data
    print("Selecting hallmark genes for case study panels ...")
    picked = pick_genes_and_events(primary_sets)

    # SUPPA events for event_id -> matrix-row mapping
    print("Loading SUPPA events parquet ...")
    suppa = load_suppa_events()
    event_id_to_idx, _ = build_event_index(suppa)
    print()

    # PSI matrices (lazy: only load cancers we actually need)
    needed_cancers = set()
    for g, info in picked.items():
        needed_cancers.update(info.keys())
    print(f"Loading PSI matrices for cancers: {sorted(needed_cancers)} ...")
    psi_tumor  = {c: load_psi_matrix(c, 'tumor')          for c in needed_cancers}
    psi_normal = {c: load_psi_matrix(NORMALS[c], 'normal') for c in needed_cancers}
    if any(v is None for v in psi_tumor.values()) or any(v is None for v in psi_normal.values()):
        sys.exit("\nOne or more PSI npz files could not be loaded. See messages above.")
    print()

    # Figure
    fig, axes = plt.subplots(2, 3, figsize=(9.6, 6.2))
    axes_flat = axes.flatten()
    for i, (gene, info) in enumerate(picked.items()):
        plot_gene_panel(axes_flat[i], gene, info,
                        event_id_to_idx, psi_tumor, psi_normal)

    # Panel letters
    for ax, letter in zip(axes_flat, 'abcdef'):
        ax.text(-0.10, 1.06, letter, transform=ax.transAxes,
                fontsize=11, fontweight='bold', va='bottom', ha='left')

    plt.tight_layout(w_pad=2.6, h_pad=3.4)

    pdf_path = os.path.join(FIG_DIR, "F5_hallmark_case_studies_v1.pdf")
    png_path = os.path.join(FIG_DIR, "F5_hallmark_case_studies_v1.png")
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, bbox_inches='tight', dpi=300)
    print(f"  Saved PDF: {pdf_path}")
    print(f"  Saved PNG: {png_path}")
    print()
    print("F5 v1 done. Open the PNG, review, then send the image back.")


if __name__ == "__main__":
    main()


---
## Section 9 -- Supplementary figures and Tables T1-T10

- **Figure S1**: cohort sample sizes and tumor-normal ratios
- **Figure S2**: beta-binomial LRT calibration via label permutation (Q-Q plots + genomic inflation factors)
- **Figure S3**: MUC16 gate verification (anti-positive control for the expression gate)
- **Figure S4**: mean-based vs median-based dPSI diagnostic (justifies the mean-based summarization choice)
- **Figure S5** (renumbered from S8): forest plot of the M4 host-gene enrichment six-test audit
- **Supplementary Tables T1-T10**: primary + secondary event catalogs, conserved-core annotations, top-50 conserved events, hallmark matrix, M4 audit table, pathway enrichment (36 terms), deviation register, sample manifest, software versions

### Supplementary Figure S2 -- calibration Q-Q plots

In [ ]:
"""
step3c_S2_calibration_qq_v1.py
Paper 5A - Supplementary Figure S2.

Beta-binomial LRT calibration QQ-plot via label permutation.

Procedure (per cancer):
  1. Load PSI matrix + integer counts for coverage-pass events.
  2. Sample N_EVENTS random events.
  3. For each, permute the tumor/normal labels once, refit the BB LRT
     under shared-mu null vs free-mu alt, get a p-value from chi-sq df=1.
  4. Plot expected vs observed -log10(p) as a QQ-plot with 95% envelope.
  5. Report lambda (genomic inflation factor) per cancer.

Three panels: LUAD, BLCA, UCEC.
Saves: Stage2C_corrected/figures/S2_calibration_qq_v1.pdf + .png

Reads:
  data/processed/psi/{LUAD,BLCA,UCEC}_psi.npz
  data/processed/psi/{LUNG,BLADDER,UTERUS}_psi.npz
  data/processed/psi/suppa_events_parsed.parquet
  data/processed/psi/Stage2C_corrected/{LUAD,BLCA,UCEC}_corrected_results.parquet

USAGE IN COLAB:
  After mounting Drive, paste this file into one cell and run.
  Runtime: ~8-12 minutes (3 cancers x ~2000 events x 1 perm).
  Tweak N_EVENTS_PER_CANCER if you want faster / slower.
"""

import os
import sys
import time
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.optimize import minimize
from scipy.special import betaln
from scipy import stats


# Config
PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
PSI_DIR      = os.path.join(PROJECT_ROOT, "data/processed/psi")
STAGE_C_DIR  = os.path.join(PSI_DIR, "Stage2C_corrected")
FIG_DIR      = os.path.join(STAGE_C_DIR, "figures")

CANCERS = ["LUAD", "BLCA", "UCEC"]
NORMALS = {"LUAD": "LUNG", "BLCA": "BLADDER", "UCEC": "UTERUS"}

N_EVENTS_PER_CANCER = 2000   # subsampling for calibration
COVERAGE_FLOOR      = 10     # per-sample reads to be 'well-covered'
MIN_WELL_COVERED    = 20     # per side
RNG_SEED            = 42


# Style
STYLE = {
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'font.family':      'sans-serif',
    'font.sans-serif':  ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size':        8.0,
    'axes.titlesize':   9.0,
    'axes.titleweight': 'bold',
    'axes.labelsize':   8.0,
    'xtick.labelsize':  7.0,
    'ytick.labelsize':  7.0,
    'axes.linewidth':   0.7,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'pdf.fonttype':     42,
    'ps.fonttype':      42,
}
COHORT_COLOR = {"LUAD": "#2E86AB", "BLCA": "#E07A5F", "UCEC": "#76A646"}


def apply_style():
    for k, v in STYLE.items():
        mpl.rcParams[k] = v


# Beta-binomial LRT (identical to cell 107 implementation)
def _bb_neg_loglik(params, k_t, n_t, k_n, n_n, mode):
    if mode == 'alt':
        mu_t, mu_n, log_phi = params
    else:
        mu_s, log_phi = params
        mu_t = mu_n = mu_s
    if not (1e-6 < mu_t < 1 - 1e-6 and 1e-6 < mu_n < 1 - 1e-6):
        return 1e10
    phi = np.exp(log_phi)
    a_t, b_t = mu_t * phi, (1 - mu_t) * phi
    a_n, b_n = mu_n * phi, (1 - mu_n) * phi
    ll_t = (betaln(k_t + a_t, n_t - k_t + b_t) - betaln(a_t, b_t)).sum()
    ll_n = (betaln(k_n + a_n, n_n - k_n + b_n) - betaln(a_n, b_n)).sum()
    return -(ll_t + ll_n)


def betabin_lrt_one_event(k_t, n_t, k_n, n_n):
    """Return p-value from chi-sq df=1 LRT, or NaN if fit fails."""
    # Sensible initial guesses
    mu_t0 = float(np.clip(k_t.sum() / max(n_t.sum(), 1), 0.05, 0.95))
    mu_n0 = float(np.clip(k_n.sum() / max(n_n.sum(), 1), 0.05, 0.95))
    mu_s0 = float(np.clip((k_t.sum() + k_n.sum()) /
                          max(n_t.sum() + n_n.sum(), 1), 0.05, 0.95))
    log_phi0 = 3.0
    try:
        alt = minimize(_bb_neg_loglik,
                       x0=[mu_t0, mu_n0, log_phi0],
                       args=(k_t, n_t, k_n, n_n, 'alt'),
                       method='Nelder-Mead',
                       options={'xatol': 1e-4, 'fatol': 1e-4, 'maxiter': 200})
        nul = minimize(_bb_neg_loglik,
                       x0=[mu_s0, log_phi0],
                       args=(k_t, n_t, k_n, n_n, 'null'),
                       method='Nelder-Mead',
                       options={'xatol': 1e-4, 'fatol': 1e-4, 'maxiter': 200})
        if not (alt.success and nul.success):
            return np.nan, np.nan
        chi2 = 2 * (nul.fun - alt.fun)
        if chi2 < 0:
            chi2 = 0.0
        p = stats.chi2.sf(chi2, df=1)
        return p, chi2
    except Exception:
        return np.nan, np.nan


# Data loading
def load_npz(label):
    path = os.path.join(PSI_DIR, f"{label}_psi.npz")
    if not os.path.exists(path):
        sys.exit(f"Missing {path}")
    data = np.load(path, allow_pickle=True)
    keys = list(data.keys())
    print(f"  [{label}] {os.path.basename(path)}  keys={keys}")
    return data


def load_coverage_pass_events(c):
    path = os.path.join(STAGE_C_DIR, f"{c}_corrected_results.parquet")
    df = pd.read_parquet(path)
    cov = df['coverage_pass'].fillna(False).astype(bool)
    return df[cov]['event_id'].astype(str).values


def reconstruct_counts(psi, coverage):
    """rint reconstruction matching cell 107."""
    total_int = np.rint(coverage).astype(np.int32)
    psi_safe  = np.where(np.isnan(psi), 0.0, psi).astype(np.float64)
    inc_int   = np.rint(psi_safe * total_int.astype(np.float64)).astype(np.int32)
    inc_int   = np.clip(inc_int, 0, total_int)
    return inc_int, total_int


# Per-cancer calibration run
def run_calibration(c, suppa_eid_to_idx, n_events, rng):
    print(f"\n--- Calibration for {c} ---")
    t_npz = load_npz(c)
    n_npz = load_npz(NORMALS[c])

    # Required arrays
    psi_t = t_npz['psi']         # (n_events, n_t_samples)
    cov_t = t_npz['coverage']    # same shape
    psi_n = n_npz['psi']
    cov_n = n_npz['coverage']

    print(f"  psi_t shape: {psi_t.shape}    psi_n shape: {psi_n.shape}")

    # Coverage-pass event ids and their row positions in the PSI matrix
    cov_pass_eids = load_coverage_pass_events(c)
    cov_pass_idx = [suppa_eid_to_idx.get(e) for e in cov_pass_eids]
    cov_pass_idx = [i for i in cov_pass_idx if i is not None]
    print(f"  coverage-pass events tracked: {len(cov_pass_idx):,}")

    # Subsample
    n_take = min(n_events, len(cov_pass_idx))
    sample_idx = rng.choice(cov_pass_idx, size=n_take, replace=False)
    print(f"  sampling {n_take:,} events for calibration")

    # Reconstruct integer counts on the subsample only
    k_t_full, n_t_full = reconstruct_counts(psi_t[sample_idx], cov_t[sample_idx])
    k_n_full, n_n_full = reconstruct_counts(psi_n[sample_idx], cov_n[sample_idx])

    n_t_samples = psi_t.shape[1]
    n_n_samples = psi_n.shape[1]
    total_samples = n_t_samples + n_n_samples

    pvals = np.full(n_take, np.nan)
    chi2s = np.full(n_take, np.nan)

    t0 = time.time()
    n_fit_success = 0
    for i in range(n_take):
        kt_row = k_t_full[i]
        nt_row = n_t_full[i]
        kn_row = k_n_full[i]
        nn_row = n_n_full[i]

        # Well-covered masks (>= COVERAGE_FLOOR)
        ok_t = nt_row >= COVERAGE_FLOOR
        ok_n = nn_row >= COVERAGE_FLOOR

        if ok_t.sum() < MIN_WELL_COVERED or ok_n.sum() < MIN_WELL_COVERED:
            continue

        kt_v = kt_row[ok_t]
        nt_v = nt_row[ok_t]
        kn_v = kn_row[ok_n]
        nn_v = nn_row[ok_n]

        # ----- LABEL PERMUTATION -----
        # Pool samples then re-split at random into "tumor-sized" and
        # "normal-sized" groups. This preserves per-sample (k, n) pairs
        # so dispersion structure stays realistic.
        k_pool = np.concatenate([kt_v, kn_v])
        n_pool = np.concatenate([nt_v, nn_v])
        n_pool_total = len(k_pool)
        if n_pool_total < 2 * MIN_WELL_COVERED:
            continue
        perm = rng.permutation(n_pool_total)
        k_pool = k_pool[perm]
        n_pool = n_pool[perm]

        # Split into pseudo-T and pseudo-N of original sizes
        nT = len(kt_v)
        kt_perm = k_pool[:nT]
        nt_perm = n_pool[:nT]
        kn_perm = k_pool[nT:]
        nn_perm = n_pool[nT:]

        p, chi2 = betabin_lrt_one_event(kt_perm, nt_perm, kn_perm, nn_perm)
        if not np.isnan(p):
            pvals[i] = p
            chi2s[i] = chi2
            n_fit_success += 1

        if (i + 1) % 250 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / max(elapsed, 0.01)
            eta = (n_take - i - 1) / max(rate, 0.01)
            print(f"    {i+1:,}/{n_take:,}  ({rate:.1f}/s, ETA {eta:.0f}s)  "
                  f"successes={n_fit_success}")

    elapsed = time.time() - t0
    print(f"  done in {elapsed:.1f}s. Fit successes: {n_fit_success}/{n_take}")

    valid = ~np.isnan(pvals)
    print(f"  valid p-values: {valid.sum()}/{n_take}")
    return pvals[valid], chi2s[valid]


# QQ-plot + lambda
def compute_lambda(chi2s):
    """Genomic inflation lambda = median(chi2) / median expected under chi2_1."""
    if len(chi2s) == 0:
        return np.nan
    obs_median = np.median(chi2s)
    expected_median = stats.chi2.ppf(0.5, df=1)
    return obs_median / expected_median


def plot_qq(ax, c, pvals, lam):
    color = COHORT_COLOR[c]
    n = len(pvals)
    if n == 0:
        ax.text(0.5, 0.5, "No valid p-values",
                transform=ax.transAxes, ha='center', va='center')
        ax.set_axis_off()
        return

    sorted_p = np.sort(pvals)
    # Expected uniform quantiles
    expected_unif = (np.arange(1, n + 1) - 0.5) / n
    obs_log = -np.log10(np.clip(sorted_p, 1e-30, 1.0))
    exp_log = -np.log10(expected_unif)

    # 95% pointwise envelope (Beta(i, n-i+1) for i-th order stat under unif)
    i = np.arange(1, n + 1)
    lo_unif = stats.beta.ppf(0.025, i, n - i + 1)
    hi_unif = stats.beta.ppf(0.975, i, n - i + 1)
    lo_log = -np.log10(np.clip(hi_unif, 1e-30, 1.0))   # NB: invert
    hi_log = -np.log10(np.clip(lo_unif, 1e-30, 1.0))

    ax.fill_between(exp_log, lo_log, hi_log,
                    color='#CCCCCC', alpha=0.5, linewidth=0,
                    label='95% envelope')

    ax.scatter(exp_log, obs_log, s=5, c=color, alpha=0.45,
               edgecolors='none', rasterized=True)

    # Diagonal
    mx = max(exp_log.max(), obs_log.max()) * 1.05
    ax.plot([0, mx], [0, mx], color='black', linewidth=0.7, linestyle='--')

    # Aesthetics
    ax.set_xlim(0, mx)
    ax.set_ylim(0, mx)
    ax.set_xlabel(r'Expected $-\log_{10}(p)$ under uniform null')
    ax.set_ylabel(r'Observed $-\log_{10}(p)$')
    ax.set_title(f"{c}  (n={n:,} permuted events)",
                 loc='center', color=color)

    # Lambda inset
    verdict = "clean"
    if lam > 1.15:
        verdict = "elevated"
    elif lam > 1.05:
        verdict = "borderline"
    ax.text(0.04, 0.96,
            f"$\\lambda$ = {lam:.3f}\n({verdict})",
            transform=ax.transAxes, ha='left', va='top',
            fontsize=7.5,
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='white', edgecolor='black',
                      linewidth=0.4))


# Main
def main():
    if not os.path.exists("/content/drive/MyDrive"):
        sys.exit("Drive not mounted. Run drive.mount('/content/drive') first.")
    if not os.path.exists(STAGE_C_DIR):
        sys.exit(f"Stage2C_corrected not found: {STAGE_C_DIR}")
    os.makedirs(FIG_DIR, exist_ok=True)

    apply_style()
    rng = np.random.default_rng(RNG_SEED)
    print(f"Output: {FIG_DIR}")
    print(f"Date:   {date.today().isoformat()}")
    print(f"N_EVENTS_PER_CANCER = {N_EVENTS_PER_CANCER}")
    print(f"COVERAGE_FLOOR per sample = {COVERAGE_FLOOR}")
    print(f"MIN_WELL_COVERED per side = {MIN_WELL_COVERED}")
    print()

    # SUPPA events -> event_id to matrix row index
    suppa_path = os.path.join(PSI_DIR, "suppa_events_parsed.parquet")
    if not os.path.exists(suppa_path):
        sys.exit(f"Missing {suppa_path}")
    suppa = pd.read_parquet(suppa_path)
    id_col = None
    for c in ['event_id', 'event', 'event_name', 'id']:
        if c in suppa.columns:
            id_col = c; break
    if id_col is None:
        sys.exit(f"No event id column in SUPPA parquet. Got: {list(suppa.columns)}")
    suppa_eid_to_idx = {str(e): i for i, e in enumerate(suppa[id_col])}
    print(f"SUPPA events: {len(suppa):,}; id column='{id_col}'")
    print()

    results = {}
    for c in CANCERS:
        pvals, chi2s = run_calibration(c, suppa_eid_to_idx,
                                       N_EVENTS_PER_CANCER, rng)
        lam = compute_lambda(chi2s)
        results[c] = (pvals, chi2s, lam)
        print(f"  [{c}] lambda = {lam:.3f}")

    # Save raw outputs for tables / supplementary
    out_csv = os.path.join(STAGE_C_DIR, "S2_calibration_lambdas.csv")
    pd.DataFrame({
        'cancer': list(results.keys()),
        'n_events_perm': [len(results[c][0]) for c in results],
        'lambda':        [results[c][2]      for c in results],
    }).to_csv(out_csv, index=False)
    print(f"\n  Lambda summary saved: {out_csv}")

    # Figure
    fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.4))
    for ax, c in zip(axes, CANCERS):
        pvals, chi2s, lam = results[c]
        plot_qq(ax, c, pvals, lam)

    plt.tight_layout(w_pad=2.0)

    pdf_path = os.path.join(FIG_DIR, "S2_calibration_qq_v1.pdf")
    png_path = os.path.join(FIG_DIR, "S2_calibration_qq_v1.png")
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, bbox_inches='tight', dpi=300)
    print(f"\n  Saved PDF: {pdf_path}")
    print(f"  Saved PNG: {png_path}")
    print()
    print("S2 v1 done. Open the PNG, review, then send the image back.")
    print("Lambdas to interpret in Methods:")
    for c in CANCERS:
        print(f"  {c}: lambda = {results[c][2]:.3f}")


if __name__ == "__main__":
    main()


### Supplementary Figure S4 -- snapping diagnostic (mean vs median dPSI)

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm

ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C = os.path.join(ROOT, "data/processed/psi/Stage2C_corrected")
FIG = os.path.join(STAGE_C, "figures")
os.makedirs(FIG, exist_ok=True)

CANCERS = ["LUAD", "BLCA", "UCEC"]
COHORT_COLOR = {"LUAD": "#2E86AB", "BLCA": "#E07A5F", "UCEC": "#76A646"}

STYLE = {
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 8.0, 'axes.titlesize': 9.0, 'axes.titleweight': 'bold',
    'axes.labelsize': 8.0, 'xtick.labelsize': 7.0, 'ytick.labelsize': 7.0,
    'axes.linewidth': 0.7,
    'axes.spines.top': False, 'axes.spines.right': False,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
}
for k, v in STYLE.items():
    mpl.rcParams[k] = v

print(f"Output dir: {FIG}")
print("Loading coverage-pass events per cancer ...")

data = {}
for c in CANCERS:
    df = pd.read_parquet(os.path.join(STAGE_C, f"{c}_corrected_results.parquet"))
    cov = df['coverage_pass'].fillna(False).astype(bool)
    df = df[cov].copy()
    df['n_min_well_covered'] = np.minimum(df['n_tumor_well_covered'],
                                          df['n_normal_well_covered'])
    data[c] = df
    diff = (df['delta_psi'] - df['delta_psi_median']).abs()
    p95 = float(diff.quantile(0.95))
    n_div = int((diff > 0.10).sum())
    max_med = float(df['delta_psi_median'].abs().max())
    print(f"  [{c}] coverage-pass={len(df):,}  "
          f"|mean-median| p95={p95:.3f}  "
          f"diverge>0.10={n_div:,}  "
          f"max|median|={max_med:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(10.0, 3.4), sharey=True)
scs = []

for ax, c in zip(axes, CANCERS):
    df = data[c]
    color = COHORT_COLOR[c]
    n = len(df)

    cvals = df['n_min_well_covered'].values.astype(float)
    cvals = np.clip(cvals, 1, cvals.max() if cvals.max() > 1 else 2)

    sc = ax.scatter(df['delta_psi'], df['delta_psi_median'],
                    s=4, c=cvals, cmap='viridis_r',
                    norm=LogNorm(vmin=max(cvals.min(), 1.0), vmax=cvals.max()),
                    alpha=0.45, edgecolors='none', rasterized=True)
    scs.append(sc)

    ax.plot([-1, 1], [-1, 1], color='black', linewidth=0.6,
            linestyle='--', alpha=0.6)
    ax.axhline(1.0, color='red', linewidth=0.6, linestyle=':', alpha=0.7)
    ax.axhline(-1.0, color='red', linewidth=0.6, linestyle=':', alpha=0.7)
    ax.axhline(0.0, color='black', linewidth=0.4, alpha=0.3)
    ax.axvline(0.0, color='black', linewidth=0.4, alpha=0.3)

    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.02, 1.02)
    ax.set_xlabel(r'mean $\Delta$PSI (corrected pipeline)')
    if ax.get_subplotspec().is_first_col():
        ax.set_ylabel(r'median $\Delta$PSI (old convention)')
    ax.set_title(f"{c}  (n={n:,})", loc='center', color=color)

    diff = (df['delta_psi'] - df['delta_psi_median']).abs()
    n_diverge = int((diff > 0.10).sum())
    pct_diverge = 100.0 * n_diverge / n if n > 0 else 0.0
    p95 = float(diff.quantile(0.95))
    max_med = float(df['delta_psi_median'].abs().max())

    inset = (f"|mean-median| p95: {p95:.3f}\n"
             f"diverge >0.10: {n_diverge:,} ({pct_diverge:.1f}%)\n"
             f"max |median|: {max_med:.3f}\n"
             f"(no snapping at +-1.0)")
    ax.text(0.04, 0.96, inset, transform=ax.transAxes,
            ha='left', va='top', fontsize=6.5,
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='white', edgecolor='black', linewidth=0.4))

cbar = fig.colorbar(scs[-1], ax=axes, location='right',
                    shrink=0.85, aspect=18, pad=0.025)
cbar.set_label('min(well-covered T, well-covered N)\nper event (log scale)',
               fontsize=7.0)
cbar.ax.tick_params(labelsize=6.5)

pdf_path = os.path.join(FIG, "S4_snapping_diagnostic_v2.pdf")
png_path = os.path.join(FIG, "S4_snapping_diagnostic_v2.png")
fig.savefig(pdf_path, bbox_inches='tight')
fig.savefig(png_path, bbox_inches='tight', dpi=300)
plt.show()

print(f"\nSaved PDF: {pdf_path}")
print(f"Saved PNG: {png_path}")
print("S4 v2 done.")


### Supplementary Figure S5 -- M4 audit forest plot

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C = os.path.join(ROOT, "data/processed/psi/Stage2C_corrected")
FIG = os.path.join(STAGE_C, "figures")
os.makedirs(FIG, exist_ok=True)

# Locked from cell 122 (M4_AUDIT_REPORT.txt)
BACKGROUND = 19782
ABERRANT = 940

TESTS = [
    ("Test 1: 71-gene curated",            "baseline",     71,  29, 14.25, 2.83e-20),
    ("Test 2: MSigDB EMT (n=200)",         "independent", 200,  22,  2.51, 2.22e-04),
    ("Test 3: MSigDB Myogenesis (n=200)",  "independent", 200,  27,  3.19, 9.86e-07),
    ("Test 6: de-circularized (n=54)",     "defensive",    54,  12,  5.79, 6.68e-06),
    ("Test 4a: KRAS Signaling Up",         "neg_control", 200,  10,  1.06, 4.81e-01),
    ("Test 4b: DNA Repair",                "neg_control", 150,   8,  1.13, 4.21e-01),
    ("Test 4c: Allograft Rejection",       "neg_control", 200,  15,  1.64, 5.47e-02),
    ("Test 4d: Pancreas Beta Cells",       "neg_control",  40,   1,  0.51, 8.58e-01),
    ("Test 4e: Heme Metabolism",           "neg_control", 200,  13,  1.40, 1.58e-01),
]

GROUP_COLOR = {
    "baseline":     "#2E5A88",
    "independent":  "#2E86AB",
    "defensive":    "#5F8A6C",
    "neg_control":  "#888888",
}
GROUP_LABEL = {
    "baseline":     "Baseline (curated reference)",
    "independent":  "Independent curation (MSigDB)",
    "defensive":    "De-circularized subset",
    "neg_control":  "Negative-control hallmarks",
}

STYLE = {
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 8.0,
    'axes.titlesize': 9.0, 'axes.titleweight': 'bold',
    'axes.labelsize': 8.0,
    'xtick.labelsize': 7.0, 'ytick.labelsize': 7.5,
    'axes.linewidth': 0.7,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
}
for k, v in STYLE.items():
    mpl.rcParams[k] = v

def or_ci(ref, ov, a_total=ABERRANT, bg=BACKGROUND):
    a = ov; b = a_total - ov; c = ref - ov; d = bg - a_total - c
    if any(x == 0 for x in [a, b, c, d]):
        a += 0.5; b += 0.5; c += 0.5; d += 0.5
    OR = (a * d) / (b * c)
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    lo = float(np.exp(np.log(OR) - 1.96 * se))
    hi = float(np.exp(np.log(OR) + 1.96 * se))
    return float(OR), lo, hi

n_rows = len(TESTS)
y_positions = np.arange(n_rows)[::-1]

# Compute all OR/CI values up front
results = []
for label, group, ref, ov, or_rep, p_rep in TESTS:
    OR, lo, hi = or_ci(ref, ov)
    results.append((label, group, OR, lo, hi, p_rep))

# Two-panel figure: forest on left, table on right
fig, (axL, axR) = plt.subplots(
    1, 2, figsize=(11.5, 5.4),
    gridspec_kw={'width_ratios': [2.0, 1.0], 'wspace': 0.05}
)

# === LEFT PANEL: forest plot ===
for y, (label, group, OR, lo, hi, p_rep) in zip(y_positions, results):
    color = GROUP_COLOR[group]
    axL.hlines(y, lo, hi, color=color, linewidth=1.8, alpha=0.85)
    axL.scatter([OR], [y], marker='D', s=52, color=color,
                edgecolor='black', linewidth=0.5, zorder=4)

axL.axvline(1.0, color='black', linestyle='--', linewidth=0.7, alpha=0.7)

# Group dividers
prev_group = None
for i, (_, group, *_) in enumerate(TESTS):
    if prev_group is not None and group != prev_group:
        y_div = (y_positions[i-1] + y_positions[i]) / 2.0
        axL.axhline(y_div, color='lightgrey', linewidth=0.5, alpha=0.7, zorder=0)
    prev_group = group

axL.set_xscale('log')
axL.set_xlim(0.3, 30)
axL.set_xticks([0.5, 1, 2, 5, 10, 20])
axL.set_xticklabels(['0.5', '1', '2', '5', '10', '20'])
axL.set_xlabel('Odds ratio (log scale; vertical dashed = no enrichment)')

axL.set_yticks(y_positions)
axL.set_yticklabels([t[0] for t in TESTS])
axL.set_ylim(-0.7, n_rows - 0.3)
axL.spines['top'].set_visible(False)
axL.spines['right'].set_visible(False)
axL.set_title("M4 host-gene enrichment 6-test audit", loc='left', pad=10)

# === RIGHT PANEL: text table aligned to forest rows ===
axR.set_xlim(0, 1)
axR.set_ylim(-0.7, n_rows - 0.3)  # matches left panel y-range exactly
axR.axis('off')

# Column headers (top of panel, in data y-coordinates)
header_y = n_rows - 0.45
axR.text(0.04, header_y, "OR  (95% CI)",
         ha='left', va='center',
         fontsize=8.0, fontweight='bold')
axR.text(0.62, header_y, "p-value",
         ha='left', va='center',
         fontsize=8.0, fontweight='bold')

# Header underline
axR.plot([0.02, 0.95], [header_y - 0.35, header_y - 0.35],
         color='black', linewidth=0.6, transform=axR.transData)

# Row text
for y, (label, group, OR, lo, hi, p_rep) in zip(y_positions, results):
    or_str = f"{OR:>5.2f}  ({lo:.2f}–{hi:.2f})"
    if p_rep < 1e-3:
        p_str = f"{p_rep:.1e}"
    elif p_rep < 1e-2:
        p_str = f"{p_rep:.3f}"
    else:
        p_str = f"{p_rep:.2f}"
    axR.text(0.04, y, or_str, ha='left', va='center',
             fontsize=7.5, family='DejaVu Sans Mono')
    axR.text(0.62, y, p_str, ha='left', va='center',
             fontsize=7.5, family='DejaVu Sans Mono')

# Mirror the group divider rules on the right panel
prev_group = None
for i, (_, group, *_) in enumerate(TESTS):
    if prev_group is not None and group != prev_group:
        y_div = (y_positions[i-1] + y_positions[i]) / 2.0
        axR.axhline(y_div, color='lightgrey', linewidth=0.5, alpha=0.7,
                    xmin=0.02, xmax=0.95, zorder=0)
    prev_group = group

# === LEGEND below the forest panel (in figure coords) ===
from matplotlib.patches import Patch
handles = [Patch(facecolor=GROUP_COLOR[g], edgecolor='black',
                 linewidth=0.4, label=GROUP_LABEL[g])
           for g in ['baseline', 'independent', 'defensive', 'neg_control']]
fig.legend(handles=handles,
           loc='lower left', bbox_to_anchor=(0.08, -0.02),
           ncol=4, frameon=True, fontsize=7.5,
           framealpha=0.95, edgecolor='black')

# === FOOTER: Test 5 randomization ===
fig.text(0.08, -0.08,
         "Test 5 (randomization): 0 of 1,000 random 71-gene reference sets "
         "reached the observed OR = 14.25; "
         "null distribution max = 3.31; empirical permutation p = 0.001.",
         fontsize=7.5, fontstyle='italic')

plt.tight_layout()

pdf_path = os.path.join(FIG, "S8_m4_audit_forest_v2.pdf")
png_path = os.path.join(FIG, "S8_m4_audit_forest_v2.png")
fig.savefig(pdf_path, bbox_inches='tight')
fig.savefig(png_path, bbox_inches='tight', dpi=300)
plt.show()

print(f"Saved PDF: {pdf_path}")
print(f"Saved PNG: {png_path}")
print("S8 v2 done.")


### Supplementary Tables T1-T10 -- workbook generation

Writes `SupplementaryTables.xlsx` with 14 sheets (T1 has 3 sheets, T2 has 3 sheets, T3-T10 have 1 sheet each).

In [ ]:
"""
step4_supplementary_tables_v2.py
Paper 5A -- supplementary tables, single Excel workbook.

CHANGE FROM v1:
  - build_T8() now reads the single pre_commitments/STAGE_C_DEVIATIONS.md
    file (six H2 sections, one per deviation) instead of looking for
    multiple files under Stage2C_corrected/.
  - T8 row schema expanded: deviation_id, title, date, direction_of_change,
    rationale_excerpt, source_file.
  - D7 (dPSI_scn >= 0.05 floor) appended programmatically.

Writes:
  Stage2C_corrected/tables/Paper5A_SupplementaryTables.xlsx

Sheets:
  T1_LUAD, T1_BLCA, T1_UCEC   Primary aberrant catalog per cancer
  T2_LUAD, T2_BLCA, T2_UCEC   Secondary aberrant catalog per cancer
  T3_conserved_core           3-way conserved core (123 events, 115 genes)
  T4_top50_annotated          Top-50 conserved events with annotation
  T5_hallmark_matrix          Hallmark cancer-splicing genes x cancers
  T6_M4_audit                 6-test audit (locked from cell 122) + Woolf 95% CIs
  T7_pathway_enrichment       Conserved-core pathway enrichment
  T8_deviations               D1-D6 from STAGE_C_DEVIATIONS.md + D7
  T9_sample_manifest          Cohort-summary placeholder
  T10_software_versions       Locked tool versions and parameters
"""

import os
import sys
import glob
import re
from datetime import date

import numpy as np
import pandas as pd

try:
    import openpyxl  # noqa
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    HAS_OPENPYXL = True
except ImportError:
    HAS_OPENPYXL = False


# Config
PROJECT_ROOT = "/content/drive/MyDrive/Paper5_SharedAntigens"
STAGE_C_DIR  = os.path.join(PROJECT_ROOT, "data/processed/psi/Stage2C_corrected")
OVERLAP_DIR  = os.path.join(STAGE_C_DIR, "M3prime_overlap")
M4_DIR       = os.path.join(STAGE_C_DIR, "M4_enrichment")
TABLE_DIR    = os.path.join(STAGE_C_DIR, "tables")
os.makedirs(TABLE_DIR, exist_ok=True)

CANCERS = ["LUAD", "BLCA", "UCEC"]

HALLMARK_GENES = [
    "FAS", "CD44", "FGFR1", "FGFR2", "FGFR3", "BCL2L1", "MDM4", "MDM2", "VEGFA",
    "KLF6", "PKM", "MET", "EGFR", "AR", "TP53", "RAC1", "RON", "SYK", "TIA1",
    "NUMB", "CTNNB1", "BIN1", "CASP9", "CASP8", "TRAF3", "ESR1", "BRCA1", "PIK3CA",
]


# Helpers
def load_corrected(c):
    path = os.path.join(STAGE_C_DIR, f"{c}_corrected_results.parquet")
    if not os.path.exists(path):
        sys.exit(f"Missing {path}")
    df = pd.read_parquet(path)
    if 'gene_name' in df.columns and 'gene_symbol' not in df.columns:
        df = df.rename(columns={'gene_name': 'gene_symbol'})
    return df


def standard_event_columns(df):
    cols = [
        'event_id', 'gene_symbol', 'gene_id', 'chrom', 'strand',
        'cassette_start', 'cassette_end',
        'n_tumor_well_covered', 'n_normal_well_covered', 'n_scn_well_covered',
        'psi_tumor_mean', 'psi_normal_mean', 'psi_scn_mean',
        'delta_psi', 'delta_psi_scn',
        'cohens_d',
        'bb_chi2', 'bb_pvalue', 'bb_mu_tumor', 'bb_mu_normal', 'bb_dispersion_phi',
        'mwu_u', 'mwu_pvalue',
        'q_value',
        'tier', 'has_scn_data', 'scn_concordant', 'scn_discordant',
        'both_tests_significant',
    ]
    cols = [c for c in cols if c in df.columns]
    return df[cols].copy()


def round_psi_columns(df):
    psi_cols = [c for c in df.columns
                if 'psi' in c.lower() or 'dispersion' in c.lower() or
                'cohens_d' == c or 'delta_psi' in c.lower()]
    for col in psi_cols:
        if col in df.columns:
            df[col] = df[col].astype(float).round(4)
    pcols = ['bb_pvalue', 'mwu_pvalue', 'q_value', 'bb_chi2']
    for col in pcols:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


# T1, T2: per-cancer event catalogs
def build_T1_T2(cancer_dfs):
    sheets = {}
    for c in CANCERS:
        df = cancer_dfs[c]
        pri = df[df['tier'] == 'primary'].copy()
        sec = df[df['tier'] == 'secondary'].copy()
        pri = round_psi_columns(standard_event_columns(pri))
        sec = round_psi_columns(standard_event_columns(sec))
        pri = pri.reindex(pri['delta_psi'].abs().sort_values(ascending=False).index)
        sec = sec.reindex(sec['delta_psi'].abs().sort_values(ascending=False).index)
        sheets[f"T1_{c}"] = pri
        sheets[f"T2_{c}"] = sec
        print(f"  T1_{c}: {len(pri):,} primary | T2_{c}: {len(sec):,} secondary")
    return sheets


# T3: 3-way conserved core
def build_T3(cancer_dfs):
    path = os.path.join(OVERLAP_DIR, "three_way_shared_events.parquet")
    if not os.path.exists(path):
        print("  [T3] three_way_shared_events.parquet not found; building from primaries")
        pri_ids = [set(cancer_dfs[c][cancer_dfs[c]['tier'] == 'primary']['event_id'].astype(str))
                   for c in CANCERS]
        common = pri_ids[0] & pri_ids[1] & pri_ids[2]
        rows = []
        for eid in sorted(common):
            row = {'event_id': eid}
            for c in CANCERS:
                hit = cancer_dfs[c][cancer_dfs[c]['event_id'].astype(str) == eid]
                if len(hit) > 0:
                    h = hit.iloc[0]
                    row[f'gene_symbol_{c}']    = h.get('gene_symbol', '')
                    row[f'delta_psi_{c}']      = round(float(h['delta_psi']), 4)
                    row[f'q_value_{c}']        = float(h['q_value'])
                    row[f'cohens_d_{c}']       = round(float(h['cohens_d']), 3)
            rows.append(row)
        df = pd.DataFrame(rows)
    else:
        df = pd.read_parquet(path)
        for col in df.select_dtypes(include=[np.number]).columns:
            if 'psi' in col.lower() or 'cohens' in col.lower():
                df[col] = df[col].round(4)
    print(f"  T3_conserved_core: {len(df):,} rows")
    return df


# T4: top-50 annotated
def build_T4():
    path = os.path.join(OVERLAP_DIR, "top50_conserved_events_annotated.csv")
    if not os.path.exists(path):
        print("  [T4] top50_conserved_events_annotated.csv not found; skipping")
        return None
    df = pd.read_csv(path)
    print(f"  T4_top50_annotated: {len(df):,} rows")
    return df


# T5: hallmark matrix
def build_T5(cancer_dfs):
    rows = []
    for g in HALLMARK_GENES:
        row = {'gene_symbol': g}
        any_hit = False
        for c in CANCERS:
            pri = cancer_dfs[c][(cancer_dfs[c]['tier'] == 'primary') &
                                (cancer_dfs[c]['gene_symbol'] == g)]
            if len(pri) == 0:
                row[f'in_primary_{c}'] = False
                row[f'strongest_dpsi_{c}'] = np.nan
                row[f'n_events_{c}'] = 0
            else:
                any_hit = True
                pri = pri.copy()
                pri['abs_dpsi'] = pri['delta_psi'].abs()
                best = pri.sort_values('abs_dpsi', ascending=False).iloc[0]
                row[f'in_primary_{c}'] = True
                row[f'strongest_dpsi_{c}'] = round(float(best['delta_psi']), 4)
                row[f'n_events_{c}'] = int(len(pri))
        n_cancers = sum(1 for c in CANCERS if row[f'in_primary_{c}'])
        row['n_cancers_hit'] = n_cancers
        if any_hit:
            rows.append(row)
    df = pd.DataFrame(rows)
    df = df.sort_values(['n_cancers_hit', 'gene_symbol'],
                        ascending=[False, True]).reset_index(drop=True)
    print(f"  T5_hallmark_matrix: {len(df):,} rows ({len(df)} of {len(HALLMARK_GENES)} hallmarks hit)")
    return df


# T6: M4 audit
def build_T6():
    BACKGROUND = 19782
    ABERRANT = 940
    TESTS = [
        ("Test 1",  "71-gene curated",                 "baseline",     71,  29, 14.25, 2.83e-20, "PASS", "OR>5, p<0.01"),
        ("Test 2",  "MSigDB EMT (n=200)",              "independent", 200,  22,  2.51, 2.22e-04, "FAIL", "OR>3, p<0.01"),
        ("Test 3",  "MSigDB Myogenesis (n=200)",       "independent", 200,  27,  3.19, 9.86e-07, "PASS", "OR>3, p<0.01"),
        ("Test 4a", "KRAS Signaling Up",               "neg_control", 200,  10,  1.06, 4.81e-01, "n/a",  "neg control"),
        ("Test 4b", "DNA Repair",                      "neg_control", 150,   8,  1.13, 4.21e-01, "n/a",  "neg control"),
        ("Test 4c", "Allograft Rejection",             "neg_control", 200,  15,  1.64, 5.47e-02, "n/a",  "neg control"),
        ("Test 4d", "Pancreas Beta Cells",             "neg_control",  40,   1,  0.51, 8.58e-01, "n/a",  "neg control"),
        ("Test 4e", "Heme Metabolism",                 "neg_control", 200,  13,  1.40, 1.58e-01, "n/a",  "neg control"),
        ("Test 5",  "Randomization (1,000 random sets)","independent",   0,   0, np.nan, 0.001,  "PASS", "empirical p<0.01"),
        ("Test 6",  "De-circularized (n=54)",          "defensive",    54,  12,  5.79, 6.68e-06, "PASS", "OR>3, p<0.01"),
    ]
    rows = []
    for test_id, ref_name, group, ref_size, overlap, or_rep, p_rep, verdict, criterion in TESTS:
        row = {
            'test_id':           test_id,
            'reference_set':     ref_name,
            'group':             group,
            'reference_size':    ref_size if ref_size > 0 else None,
            'overlap':           overlap if overlap > 0 else None,
            'aberrant_union':    ABERRANT,
            'background':        BACKGROUND,
            'odds_ratio':        round(or_rep, 3) if not np.isnan(or_rep) else None,
            'p_value':           p_rep,
            'verdict':           verdict,
            'audit_criterion':   criterion,
        }
        if ref_size > 0 and overlap > 0:
            a, b, c, d = overlap, ABERRANT - overlap, ref_size - overlap, BACKGROUND - ABERRANT - (ref_size - overlap)
            if all(x > 0 for x in [a, b, c, d]):
                OR = (a * d) / (b * c)
                se = np.sqrt(1/a + 1/b + 1/c + 1/d)
                lo = float(np.exp(np.log(OR) - 1.96 * se))
                hi = float(np.exp(np.log(OR) + 1.96 * se))
                row['ci95_lower'] = round(lo, 3)
                row['ci95_upper'] = round(hi, 3)
        rows.append(row)
    df = pd.DataFrame(rows)
    print(f"  T6_M4_audit: {len(df):,} rows (locked from cell 122)")
    return df


# T7: pathway enrichment
def build_T7():
    patterns = ['pathway*.csv', '*enrichr*.csv', '*enrich*.csv',
                'gseapy*.csv', '*hallmark*.csv', '*GO_*.csv', '*reactome*.csv']
    found = []
    for p in patterns:
        found.extend(glob.glob(os.path.join(OVERLAP_DIR, p)))
    found = sorted(set(found))
    if not found:
        print("  [T7] no pathway enrichment CSVs found in M3prime_overlap/; skipping")
        return None
    print(f"  [T7] concatenating {len(found)} pathway CSVs:")
    frames = []
    for f in found:
        try:
            sub = pd.read_csv(f)
            sub['source_file'] = os.path.basename(f)
            frames.append(sub)
            print(f"    + {os.path.basename(f)}  ({len(sub):,} rows)")
        except Exception as e:
            print(f"    ! {os.path.basename(f)}: {e}")
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True, sort=False)
    print(f"  T7_pathway_enrichment: {len(df):,} rows total")
    return df


# T8: deviations register  --- PATCHED ---
def build_T8():
    """Parse pre_commitments/STAGE_C_DEVIATIONS.md as a single file with
    H2 deviation sections, extract one row per deviation, then append D7."""
    md_path = os.path.join(PROJECT_ROOT, "pre_commitments/STAGE_C_DEVIATIONS.md")
    rows = []
    if not os.path.exists(md_path):
        print(f"  [T8] {md_path} not found; writing only D7")
    else:
        with open(md_path, 'r') as f:
            txt = f.read()
        print(f"  [T8] parsing {md_path}  ({len(txt):,} chars)")

        # Split on H2 headers that look like "## Deviation N -- Title".
        # Lookahead so the delimiter is kept with the next section.
        sections = re.split(r'(?m)^(?=## Deviation \d+)', txt)
        for sec in sections:
            sec = sec.strip()
            if not sec.startswith('## Deviation'):
                continue
            lines = sec.splitlines()
            header = lines[0].lstrip('#').strip()
            # Match either em-dash, en-dash, or hyphen between number and title
            m = re.match(r'Deviation\s+(\d+)\s*[-–-]\s*(.+)', header)
            if not m:
                continue
            num = int(m.group(1))
            title = m.group(2).strip()

            # Pull the **Date:** / **Direction:** / **Author:** lines if nearby
            dev_date = ""
            direction = ""
            for line in lines[1:8]:
                if line.startswith('**Date:**'):
                    dev_date = line.replace('**Date:**', '').strip()
                elif line.startswith('**Direction of change:**'):
                    direction = line.replace('**Direction of change:**', '').strip()

            # Find a substantive section as the rationale excerpt
            rationale = ""
            for h_label in ['### What changed', '### What happened',
                            '### What was pre-committed', '### Why this record exists',
                            '### Numbers locked']:
                idx = sec.find(h_label)
                if idx != -1:
                    after = sec[idx + len(h_label):].lstrip()
                    next_h = re.search(r'(?m)^###\s', after)
                    snippet = after[:next_h.start()] if next_h else after
                    snippet = re.sub(r'\s+', ' ', snippet).strip()
                    rationale = snippet[:600] + ('...' if len(snippet) > 600 else '')
                    break

            rows.append({
                'deviation_id':         f"D{num}",
                'title':                title,
                'date':                 dev_date,
                'direction_of_change':  direction,
                'rationale_excerpt':    rationale,
                'source_file':          'pre_commitments/STAGE_C_DEVIATIONS.md',
            })
            print(f"    + D{num}: {title[:65]}")

    # Append D7 (dPSI_scn >= 0.05 floor)
    rows.append({
        'deviation_id':         'D7',
        'title':                'dPSI_scn >= 0.05 floor in 3C concordance check',
        'date':                 date.today().isoformat(),
        'direction_of_change':  'Strengthening (more conservative).',
        'rationale_excerpt':    (
            'Stage 2C corrected pipeline (cell 107) requires not only sign(dPSI_tumor_vs_normal) '
            '== sign(dPSI_tumor_vs_SCN) but also |dPSI_scn| >= 0.05, to exclude events where the '
            'SCN-direction call is too weak to be informative. This tightens the primary tier '
            'and is conservative. Logged as a post-hoc precommit clarification identified during '
            'figure regeneration audit (2026-06-13).'),
        'source_file':          '(this audit pass; to be appended to STAGE_C_DEVIATIONS.md)',
    })

    df = pd.DataFrame(rows)
    print(f"  T8_deviations_register: {len(df):,} rows")
    return df


# T9: sample manifest (best-effort)
def build_T9():
    candidates = [
        os.path.join(PROJECT_ROOT, "data/processed/clinical/sample_manifest.csv"),
        os.path.join(PROJECT_ROOT, "data/processed/clinical/sample_manifest.tsv"),
        os.path.join(PROJECT_ROOT, "data/processed/sample_manifest.csv"),
        os.path.join(PROJECT_ROOT, "data/raw/recount3_metadata.csv"),
    ]
    for p in candidates:
        if os.path.exists(p):
            try:
                sep = '\t' if p.endswith('.tsv') else ','
                df = pd.read_csv(p, sep=sep)
                print(f"  T9_sample_manifest: loaded from {os.path.basename(p)} "
                      f"({len(df):,} rows)")
                return df
            except Exception as e:
                print(f"  T9 [{p}]: {e}")
    print("  [T9] no sample manifest found; writing summary placeholder.")
    placeholder = pd.DataFrame([
        {'cohort': 'LUAD',     'source': 'TCGA',     'n_samples': 542, 'recount3_project': 'TCGA',  'notes': 'cancer cohort'},
        {'cohort': 'LUNG',     'source': 'GTEx',     'n_samples': 655, 'recount3_project': 'GTEX',  'notes': 'normal cohort for LUAD'},
        {'cohort': 'LUAD-SCN', 'source': 'TCGA',     'n_samples': 59,  'recount3_project': 'TCGA',  'notes': 'solid-tissue normal for LUAD 3C check'},
        {'cohort': 'BLCA',     'source': 'TCGA',     'n_samples': 414, 'recount3_project': 'TCGA',  'notes': 'cancer cohort'},
        {'cohort': 'BLADDER',  'source': 'GTEx',     'n_samples': 21,  'recount3_project': 'GTEX',  'notes': 'normal cohort for BLCA'},
        {'cohort': 'BLCA-SCN', 'source': 'TCGA',     'n_samples': 19,  'recount3_project': 'TCGA',  'notes': 'solid-tissue normal for BLCA 3C check'},
        {'cohort': 'UCEC',     'source': 'TCGA',     'n_samples': 554, 'recount3_project': 'TCGA',  'notes': 'cancer cohort'},
        {'cohort': 'UTERUS',   'source': 'GTEx',     'n_samples': 159, 'recount3_project': 'GTEX',  'notes': 'normal cohort for UCEC'},
        {'cohort': 'UCEC-SCN', 'source': 'TCGA',     'n_samples': 35,  'recount3_project': 'TCGA',  'notes': 'solid-tissue normal for UCEC 3C check'},
    ])
    print(f"  T9_sample_manifest: {len(placeholder):,} rows (cohort-summary placeholder)")
    return placeholder


# T10: software versions
def build_T10():
    rows = [
        ('Python',              '3.12.13',                                 'language',          'Colab'),
        ('NumPy',               'see env',                                 'numerics',          'rint-based integer count reconstruction'),
        ('Pandas',              'see env',                                 'data frames',       'parquet I/O'),
        ('SciPy',               'see env',                                 'statistics',        'chi2 LRT, Mann-Whitney U, Nelder-Mead optimizer'),
        ('statsmodels',         'see env',                                 'multiple testing',  'BH-FDR (multipletests, method=fdr_bh)'),
        ('matplotlib',          'see env',                                 'figures',           'fonttype=42 for vector PDF/Illustrator'),
        ('matplotlib-venn',     'see env',                                 'figures',           'F4 Venn'),
        ('adjustText',          'optional',                                'figures',           'F3 hallmark labels (rotated 90 if missing)'),
        ('openpyxl',            'see env',                                 'spreadsheet',       'this workbook'),
        ('gseapy',              'see env',                                 'enrichment',        'pathway enrichment of conserved core'),
        ('recount3',            '(Bioconductor R)',                        'data',              'TCGA + GTEx Monorail-aligned BAMs and junctions'),
        ('SUPPA2',              '(parsed events)',                         'data',              'cassette exon event catalog (suppa_events_parsed.parquet, 38,951 events)'),
        ('GENCODE',             'v26',                                     'annotation',        'gene/transcript coordinates'),
        ('MyGene.info',         'API',                                     'ID resolution',     'Ensembl gene ID -> HGNC symbol; 828/844 resolved (98.1%)'),
        ('MSigDB',              'hallmark gene sets',                      'enrichment',        'EMT, Myogenesis, neg-control hallmarks'),
        ('Test (primary)',      'beta-binomial LRT',                       'parameter',         'logit link; shared dispersion phi; chi-sq df=1'),
        ('Test (sensitivity)',  'Mann-Whitney rank-sum (two-sided)',       'parameter',         'scipy.stats.mannwhitneyu, nan_policy=omit'),
        ('FDR',                 'Benjamini-Hochberg',                      'parameter',         'q < 0.05'),
        ('Effect size gate',    '|dPSI| >= 0.15 (mean-based)',             'parameter',         'replaces median to avoid snapping'),
        ('Coverage gate',       '>= 10 reads/sample, >= 20 samples/side',  'parameter',         'Kahles 2018 + Buen Abad Najar 2020'),
        ('3C concordance',      'sign(dPSI) == sign(dPSI_SCN) AND |dPSI_SCN| >= 0.05',
         'parameter',
         'SCN well-covered count >= 5; floor 0.05 logged as Deviation 7'),
        ('M2 prime thresholds', 'LUAD >= 500, BLCA >= 300, UCEC >= 300',   'parameter',         'precommit STAGE_C_FIX_PRECOMMIT.md Section 3'),
        ('Background size',     '19,782 protein-coding genes',             'parameter',         'M4 enrichment universe (MyGene-resolved)'),
        ('Random seed',         '42',                                      'parameter',         'F5 jitter; S2 calibration permutation; bootstrap'),
        ('Bootstrap',           '200 reps, 10% sample dropout',            'parameter',         'directional sign-stability on conserved core'),
        ('Calibration',         'label-permutation, 2,000 events/cancer',  'parameter',         'S2 lambda = 1.067 (LUAD) / 0.904 (BLCA) / 0.943 (UCEC)'),
    ]
    df = pd.DataFrame(rows, columns=['tool_or_parameter', 'version_or_value', 'role', 'notes'])
    print(f"  T10_software_versions: {len(df):,} rows")
    return df


# Excel writer
def write_workbook(sheets_in_order, out_path):
    if not HAS_OPENPYXL:
        sys.exit("openpyxl not available. Run:  !pip install openpyxl -q  and rerun.")
    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        for name, df in sheets_in_order:
            if df is None or len(df) == 0:
                print(f"  [skip] {name} is empty")
                continue
            sheet = name[:31]
            df.to_excel(writer, sheet_name=sheet, index=False, freeze_panes=(1, 0))

        wb = writer.book
        header_font = Font(bold=True, color="000000")
        header_fill = PatternFill(start_color="DDDDDD", end_color="DDDDDD", fill_type="solid")
        thin = Side(border_style="thin", color="BBBBBB")
        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            for cell in ws[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = Alignment(horizontal='left', vertical='center')
                cell.border = Border(bottom=thin)
            for col_idx, col_cells in enumerate(ws.columns, start=1):
                lengths = [len(str(c.value)) if c.value is not None else 0
                           for c in col_cells[:60]]
                width = min(max(max(lengths) + 2 if lengths else 12, 10), 42)
                ws.column_dimensions[openpyxl.utils.get_column_letter(col_idx)].width = width


# Main
def main():
    if not os.path.exists("/content/drive/MyDrive"):
        sys.exit("Drive not mounted.")
    if not os.path.exists(STAGE_C_DIR):
        sys.exit(f"Stage2C_corrected not found: {STAGE_C_DIR}")
    if not HAS_OPENPYXL:
        sys.exit("openpyxl missing. Run:  !pip install openpyxl -q  and rerun.")

    print(f"Project root: {PROJECT_ROOT}")
    print(f"Output dir:   {TABLE_DIR}")
    print(f"Date:         {date.today().isoformat()}")
    print()
    print("Loading corrected results parquets ...")
    cancer_dfs = {c: load_corrected(c) for c in CANCERS}

    print()
    print("Building T1/T2 per-cancer catalogs ...")
    t1_t2 = build_T1_T2(cancer_dfs)

    print()
    print("Building T3 conserved core ...")
    t3 = build_T3(cancer_dfs)

    print()
    print("Building T4 top-50 annotated ...")
    t4 = build_T4()

    print()
    print("Building T5 hallmark matrix ...")
    t5 = build_T5(cancer_dfs)

    print()
    print("Building T6 M4 audit ...")
    t6 = build_T6()

    print()
    print("Building T7 pathway enrichment ...")
    t7 = build_T7()

    print()
    print("Building T8 deviations register ...")
    t8 = build_T8()

    print()
    print("Building T9 sample manifest ...")
    t9 = build_T9()

    print()
    print("Building T10 software versions ...")
    t10 = build_T10()

    sheets_in_order = []
    for c in CANCERS:
        sheets_in_order.append((f"T1_{c}_primary", t1_t2.get(f"T1_{c}")))
    for c in CANCERS:
        sheets_in_order.append((f"T2_{c}_secondary", t1_t2.get(f"T2_{c}")))
    sheets_in_order += [
        ("T3_conserved_core",    t3),
        ("T4_top50_annotated",   t4),
        ("T5_hallmark_matrix",   t5),
        ("T6_M4_audit",          t6),
        ("T7_pathway_enrichment", t7),
        ("T8_deviations",        t8),
        ("T9_sample_manifest",   t9),
        ("T10_software",         t10),
    ]

    out_path = os.path.join(TABLE_DIR, "Paper5A_SupplementaryTables.xlsx")
    print()
    print(f"Writing {out_path} ...")
    write_workbook(sheets_in_order, out_path)
    size_kb = os.path.getsize(out_path) / 1024
    print(f"  Wrote {size_kb:.1f} KB")
    print()
    print("Sheet summary:")
    for name, df in sheets_in_order:
        n = 0 if df is None else len(df)
        ncols = 0 if df is None else len(df.columns)
        print(f"  {name:<28} {n:>6,} rows x {ncols:>3} cols")
    print()
    print("Supplementary tables done. Open the workbook in Excel/LibreOffice and skim.")


if __name__ == "__main__":
    main()


---

## End of pipeline

All headline numbers referenced in the manuscript trace to a printed output cell above. For a complete audit trail of methodological deviations from the original pre-commitment, see `STAGE_C_DEVIATIONS.md` in the repository root.

**Corresponding author**: Md. Zulkarnain Sajid, zulkarnainsajid4768@gmail.com
**ORCID**: [0009-0007-9421-3016](https://orcid.org/0009-0007-9421-3016)
**Code**: <https://github.com/nain0017/aberrant-splicing-LUAD-BLCA-UCEC>
**Archive**: [10.5281/zenodo.21306629](https://doi.org/10.5281/zenodo.21306629)